# Supply Chain Twin: full pipeline notebook

Generated by `scripts/build_notebook.py` from the repository (base commit `a52d5f6`, plus any
uncommitted changes at generation time). Every project file is written by its own `%%writefile`
cell, so **all the code is in this notebook** and can be edited here.

**How to use** (Colab or local Jupyter):
1. Run *Setup*, then all *file* cells (Runtime → Run all is fine).
2. Run the tests, then fetch the data (about 115 MB, pinned revision) and run the audit.
3. Run the steps. `FAST = True` uses small sizes so each step finishes in minutes;
   set `FAST = False` for the full runs reported in `docs/results_step*.md`.

**Locally**, open it in a fresh virtual environment: the install cell installs the notebook's
own copy of the package (into `SupplyTwinAI_nb/` next to the notebook).

Rules that every step follows are in `CLAUDE.md` of the repository: rolling-origin
backtesting without leakage, point-in-time features, the official `eval` split is never read.
Sales are normalized units; reconstructed demand is never presented as observed.

## Setup

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path("/content/SupplyTwinAI" if "google.colab" in sys.modules else "SupplyTwinAI_nb")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
for d in ["src/sctwin", "configs/manifests", "scripts", "tests", "data", "runs"]:
    os.makedirs(d, exist_ok=True)
print("working directory:", os.getcwd(), "| CPU cores:", os.cpu_count())

## Files: Packaging
Project metadata, licence and readme (needed by `pip install -e .`).

In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["hatchling>=1.25"]
build-backend = "hatchling.build"

[project]
name = "sctwin"
version = "0.1.0"
description = "Bayesian demand intelligence and adaptive supply-chain decision twin (research code)"
readme = "README.md"
license = "MIT"
license-files = ["LICENSE"]
requires-python = ">=3.11,<3.14"
dependencies = [
    "numpy>=1.26,<3",
    "polars>=1.20,<2",
    "pyarrow>=16",
    "pyyaml>=6",
    "pydantic>=2.7,<3",
    "huggingface_hub>=0.25",
]

[project.optional-dependencies]
dev = ["pytest>=8", "hypothesis>=6.100", "ruff>=0.6", "mypy>=1.11", "types-PyYAML"]
models = ["lightgbm>=4.3", "scipy>=1.13", "scikit-learn>=1.5", "pandas>=2.2"]
bayes = ["jax>=0.4.30", "numpyro>=0.15", "arviz>=0.19"]

[tool.hatch.build.targets.wheel]
packages = ["src/sctwin"]

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-q --strict-markers"
markers = ["data: needs the downloaded dataset (skipped in CI)"]

[tool.ruff]
line-length = 100
target-version = "py311"

[tool.ruff.lint]
select = ["E", "F", "W", "I", "B", "UP", "NPY"]

[tool.mypy]
python_version = "3.11"
ignore_missing_imports = true


In [ ]:
%%writefile LICENSE
MIT License

Copyright (c) 2026 Mehrnoosh Oghbaie

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

---------------------------------------------------------------------------
Third-party data

This license covers the source code in this repository only. The
FreshRetailNet-50K dataset is NOT included or redistributed here; it is
published by Dingdong-Inc under the Creative Commons Attribution 4.0
International License (CC BY 4.0). Any derived artifacts published from it
(for example the compact JSON files in artifacts/demo/) carry the attribution
given in CITATIONS.md.


In [ ]:
%%writefile README.md
# Supply Chain Twin

**Bayesian demand intelligence and an adaptive supply-chain decision twin, built on censored
fresh-retail sales.**

This is research code with two connected pipelines, each evaluated on its own:

- **A. Demand intelligence:** estimates latent demand from sales that stockouts have censored,
  produces calibrated probabilistic forecasts, and updates a latent demand state as historical
  data is replayed in order.
- **B. Decision twin:** feeds those predictive demand distributions into a simulated perishable
  inventory system. It then compares classical replenishment policies, stochastic MPC,
  contextual policy selection and meta-learning on held-out scenarios.

The public demo will be a static site at <https://moghbaie.github.io/supply-chain-twin/>. It
shows precomputed research artifacts. No live model training or inference runs in the browser.

> **What this project does not claim**
> - Reconstructed demand is an *estimate*, never observed ground truth.
> - Replay of historical data is not live retail telemetry.
> - Inventory costs and lead times are **synthetic research scenarios**, not retailer savings.
> - Sales are in the provider's **normalized units**, not physical unit counts.

## Status

| Milestone | Steps | Status |
|---|---|---|
| M1 Reproducible data and baselines | 0 Scaffold · 1 Ingest and audit · 2 Point-in-time features and metrics · 3 Baselines | Steps 0–3 done ([results](docs/results_step3.md)) |
| M2 Bayesian demand estimator | 4 Censoring benchmark · 5 Bayesian model ladder · 6 Particle filter and replay | Step 4 done ([results](docs/results_step4.md)) |
| M3 Stateful simulator | 7 Simulator plus a Python ↔ TypeScript parity slice | Not started |
| M4 Classical decision engine | 8 Order-up-to, newsvendor, SAA-MPC, CVaR | Not started |
| M5 Adaptive and meta-learning | 9 Contextual selection and optimization, Reptile (time-boxed) | Not started |
| M6 Public app | 10 Final evaluation and report · 11 Export, web app, deploy | Not started |

The full plan, with its tests, validation gates and evaluation design, is in [`plan.md`](plan.md).
The project's non-negotiable backtesting and feature rules are in [`CLAUDE.md`](CLAUDE.md).
The original brief is [`plan_Supply_Twin_AI.md`](plan_Supply_Twin_AI.md).

## Key data findings so far

These come from the Step 1 audit. Details and numbers are in [`docs/data_card.md`](docs/data_card.md).

- **The data is clean.** It has 50,000 store-product series × 90 training days, plus the 7-day
  official eval week, which is kept as a locked test set. There are no gaps, duplicates or nulls,
  and daily sales equal the sum of hourly sales exactly.
- **Units:** 99.56% of hourly sales fall on a 0.1 grid, but the normalization is undocumented.
  Models therefore use a continuous likelihood. Treating sales as counts appears only as a
  labelled sensitivity analysis.
- **Stockouts are driven by demand.** Within each series, the in-stock hour just before a
  stockout flag sells **2.2×** that series' normal amount for the hour, while the first flagged
  hour sells **0.18×**. Availability is endogenous, and the pre-flag hour is treated as a
  censored depletion hour.
- **The stockout count field** `stock_hour6_22_cnt` covers hours 6–21. Weather is daily,
  per store, and observed after the fact.

## Repository layout

```
src/sctwin/          library code (the notebooks and scripts only call into it)
  data.py            pinned download, checksums, schema validation, hourly expansion
  audit.py           data audit: integrity gates plus normalization and stockout evidence
  splits.py          temporal and group hold-out splits, tamper-evident manifests
  panel.py           dense series x day x hour arrays; hourly availability classes
  timeunit.py        resampling to the backtest unit ("day" / "hour") with availability rules
  features.py        point-in-time features as of origin T (lags, windows, seasonal, covariates)
  backtest.py        rolling-origin folds; check_split enforces the no-leakage rules
  baseline.py        seasonal naive, seasonal climatology, Tweedie GLM, LightGBM point/quantile
  baseline_runner.py per-fold refit and prediction; evaluation subgroups
  censoring.py       Step 4 benchmark: demand-driven stockout injection, calibration,
                     reconstruction baselines, fully synthetic latent-demand generator
  evaluation.py      metrics per model, per fold, per subgroup; paired bootstrap vs reference
  metrics.py         MAE/WAPE, pinball, WIS, CRPS, coverage, PIT, cluster bootstrap, Holm
  repro.py           seeding, config hashing, run manifests (environment and package versions)
scripts/             command-line entry points (fetch_data.py, audit_data.py)
tests/               unit tests on synthetic fixtures; tests marked `data` use the real dataset
configs/             experiment configs; manifests/ holds the frozen split definitions
docs/data_card.md    dataset provenance, audit findings, modelling implications
docs/data_model.md   schema, entity relationships, constraints, derived representations
notebooks/           Colab drivers for heavy runs (MCMC, meta-learning), from Step 5 on
artifacts/demo/      small, versioned JSON for the web demo (the only data in git)
web/                 Vite + React + TypeScript app (Step 11)
```

Raw data, posterior samples, model weights and large Parquet files never go in git. They live
in `data/` locally (git-ignored), or on Google Drive when running in Colab.

## Quickstart

Requirements: Python 3.11–3.13 and [uv](https://docs.astral.sh/uv/). Node 20+ is needed only
for the web app.

```bash
uv venv --python 3.12 .venv
uv pip install -e ".[dev]"

# Unit tests run on synthetic data, with no download needed
.venv/Scripts/python -m pytest -m "not data"

# Download the dataset at the pinned revision (~115 MB) and record checksums
.venv/Scripts/python scripts/fetch_data.py

# Run the audit; writes data/audit/audit_report.json, the split manifest and a 20-series dev subset
.venv/Scripts/python scripts/audit_data.py

# Tests against the real data (checksums, gates, manifest provenance)
.venv/Scripts/python -m pytest -m data
```

On Linux, macOS or Colab, use `.venv/bin/python`. Optional extras: `.[models]` for LightGBM and
SciPy, `.[bayes]` for JAX, NumPyro and ArviZ.

## Reproducibility

- **Data:** pinned to Hub revision `c374e85975c82816c495acfa904cc55ab14a47d3`. SHA256 checksums
  are verified before every audit run.
- **Splits:** `configs/manifests/splits_v1.json` is written once and protected by a content
  digest. It refuses to be silently overwritten, and edits made after writing are detected.
- **Runs:** every stage writes a `run_manifest.json` recording the config hash, seed, Python and
  package versions, CPU and accelerator.
- **Locked test:** the official eval split is only integrity-checked during development. It will
  be read through a test ledger that allows one logged run per final configuration.
- **CI:** GitHub Actions runs ruff, mypy and the synthetic-data test suite on every push and
  pull request.

## Evaluation tracks

1. **Observed-sales forecasting** on the locked official test week: WIS/CRPS, WAPE, interval
   coverage and sharpness, plus subgroups (low volume, promotions, stockout periods).
2. **Latent-demand recovery** on controlled synthetic censoring, where the truth is known. This
   is reported as a proxy.
3. **Decision quality** in the synthetic inventory environment, using common random numbers and
   a cluster-level paired bootstrap with Holm correction.

## Data and license

- **Code:** MIT, see [`LICENSE`](LICENSE).
- **Data:** FreshRetailNet-50K by Dingdong-Inc, licensed CC BY 4.0. It is not redistributed
  here. Citation and provenance are in [`CITATIONS.md`](CITATIONS.md).

If you use this work, please cite the dataset paper:
Wang et al., *FreshRetailNet-50K: A Stockout-Annotated Censored Demand Dataset for Latent Demand
Recovery and Forecasting in Fresh Retail*, arXiv:2505.16319 (2025).


## Files: Package: data, audit and splits
Pinned download, checksums, Step 1 audit, split manifests.

In [ ]:
%%writefile src/sctwin/__init__.py
"""Supply Chain Twin research package."""

__version__ = "0.1.0"


In [ ]:
%%writefile src/sctwin/data.py
"""Dataset acquisition, schema validation and representation conversion.

Storage representation is the compact one shipped by the provider: one row per
(store_id, product_id, dt) with 24-element hourly arrays. Hourly long format is
materialized only on demand via :func:`to_hourly_long`.
"""

from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl

from sctwin.repro import sha256_file

REPO_ID = "Dingdong-Inc/FreshRetailNet-50K"
SPLIT_FILES = {"train": "data/train.parquet", "eval": "data/eval.parquet"}
KEY = ("store_id", "product_id", "dt")
SERIES_KEY = ("store_id", "product_id")
N_HOURS = 24

# Expected column -> polars dtype family. Integer widths are not enforced (provider may vary).
SCHEMA: dict[str, str] = {
    "city_id": "int",
    "store_id": "int",
    "management_group_id": "int",
    "first_category_id": "int",
    "second_category_id": "int",
    "third_category_id": "int",
    "product_id": "int",
    "dt": "str",
    "sale_amount": "float",
    "hours_sale": "list[float]",
    "stock_hour6_22_cnt": "int",
    "hours_stock_status": "list[int]",
    "discount": "float",
    "holiday_flag": "int",
    "activity_flag": "int",
    "precpt": "float",
    "avg_temperature": "float",
    "avg_humidity": "float",
    "avg_wind_level": "float",
}


# --------------------------------------------------------------------------- fetch


def fetch(
    out_dir: str | Path, revision: str | None = None, repo_id: str = REPO_ID
) -> dict[str, Any]:
    """Download both splits at a pinned Hub commit and write ``provenance.json``.

    If ``revision`` is None the current head commit SHA is resolved and pinned.
    """
    from huggingface_hub import HfApi, hf_hub_download

    api = HfApi()
    info = api.dataset_info(repo_id, revision=revision)
    sha = info.sha
    out = Path(out_dir).resolve()
    out.mkdir(parents=True, exist_ok=True)
    files: dict[str, Any] = {}
    for split, fname in SPLIT_FILES.items():
        local = hf_hub_download(
            repo_id, fname, repo_type="dataset", revision=sha, local_dir=str(out)
        )
        p = Path(local)
        files[split] = {
            "hub_path": fname,
            "local_path": p.relative_to(out).as_posix(),
            "bytes": p.stat().st_size,
            "sha256": sha256_file(p),
        }
    prov = {
        "repo_id": repo_id,
        "revision": sha,
        "license": "CC BY 4.0",
        "card_version": "1.0 (05/08/2025)",
        "files": files,
    }
    (out / "provenance.json").write_text(json.dumps(prov, indent=2), encoding="utf-8")
    return prov


def verify_checksums(data_dir: str | Path) -> dict[str, bool]:
    """Re-hash local files against provenance.json (immutability check)."""
    d = Path(data_dir)
    prov = json.loads((d / "provenance.json").read_text(encoding="utf-8"))
    return {
        split: sha256_file(d / meta["local_path"]) == meta["sha256"]
        for split, meta in prov["files"].items()
    }


def load_split(data_dir: str | Path, split: str) -> pl.DataFrame:
    d = Path(data_dir)
    prov = json.loads((d / "provenance.json").read_text(encoding="utf-8"))
    return pl.read_parquet(d / prov["files"][split]["local_path"])


# --------------------------------------------------------------------------- schema


def _family(dtype: pl.DataType) -> str:
    if dtype.is_integer():
        return "int"
    if dtype.is_float():
        return "float"
    if dtype == pl.String:
        return "str"
    if isinstance(dtype, pl.List | pl.Array):
        return f"list[{_family(dtype.inner)}]"  # type: ignore[arg-type]
    return str(dtype)


@dataclass
class SchemaReport:
    missing: list[str]
    unexpected: list[str]
    wrong_type: dict[str, tuple[str, str]]

    @property
    def ok(self) -> bool:
        return not (self.missing or self.wrong_type)


def validate_schema(df: pl.DataFrame | pl.LazyFrame) -> SchemaReport:
    schema = df.collect_schema() if isinstance(df, pl.LazyFrame) else df.schema
    missing = [c for c in SCHEMA if c not in schema]
    unexpected = [c for c in schema if c not in SCHEMA]
    wrong = {
        c: (SCHEMA[c], _family(schema[c]))
        for c in SCHEMA
        if c in schema and _family(schema[c]) != SCHEMA[c]
    }
    return SchemaReport(missing, unexpected, wrong)


# --------------------------------------------------------------------------- representations


def hourly_matrix(df: pl.DataFrame, col: str, dtype: Any = np.float64) -> np.ndarray:
    """(n_rows, 24) array from a list column. Raises if any row is not length 24."""
    lens = df[col].list.len()
    lo, hi = int(lens.min()), int(lens.max())  # type: ignore[arg-type]
    if lo != N_HOURS or hi != N_HOURS:
        raise ValueError(f"{col}: list lengths in [{lo}, {hi}], expected 24")
    return df[col].list.to_array(N_HOURS).to_numpy().astype(dtype, copy=False)


def to_hourly_long(df: pl.DataFrame | pl.LazyFrame) -> pl.LazyFrame:
    """Expand daily rows with hourly arrays to one row per (series, dt, hour). Lazy."""
    lf = df.lazy() if isinstance(df, pl.DataFrame) else df
    return (
        lf.with_columns(hour=pl.int_ranges(0, pl.col("hours_sale").list.len()))
        .explode(["hours_sale", "hours_stock_status", "hour"], empty_as_null=True)
        .rename({"hours_sale": "sale", "hours_stock_status": "stockout"})
        .with_columns(pl.col("hour").cast(pl.Int8))
    )


In [ ]:
%%writefile src/sctwin/repro.py
"""Reproducibility helpers: seeding, config loading/hashing, run manifests."""

from __future__ import annotations

import hashlib
import json
import os
import platform
import random
import sys
from datetime import UTC, datetime
from importlib import metadata
from pathlib import Path
from typing import Any

import numpy as np
import yaml

TRACKED_PACKAGES = (
    "numpy",
    "polars",
    "pyarrow",
    "pydantic",
    "huggingface_hub",
    "lightgbm",
    "scipy",
    "jax",
    "numpyro",
    "arviz",
)


def seed_everything(seed: int) -> np.random.Generator:
    """Seed Python and NumPy global state; return a fresh Generator for explicit use."""
    random.seed(seed)
    np.random.seed(seed)  # noqa: NPY002 - legacy global state, for third-party code only
    os.environ["PYTHONHASHSEED"] = str(seed)
    return np.random.default_rng(seed)


def load_config(path: str | Path) -> dict[str, Any]:
    with open(path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    if not isinstance(cfg, dict):
        raise ValueError(f"config {path} is not a mapping")
    return cfg


def config_hash(cfg: dict[str, Any]) -> str:
    """Stable short hash of a config (key order independent)."""
    blob = json.dumps(cfg, sort_keys=True, separators=(",", ":"), default=str)
    return hashlib.sha256(blob.encode()).hexdigest()[:12]


def sha256_file(path: str | Path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()


def _pkg_versions() -> dict[str, str | None]:
    out: dict[str, str | None] = {}
    for p in TRACKED_PACKAGES:
        try:
            out[p] = metadata.version(p)
        except metadata.PackageNotFoundError:
            out[p] = None
    return out


def environment_info() -> dict[str, Any]:
    info: dict[str, Any] = {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "cpu_count": os.cpu_count(),
        "packages": _pkg_versions(),
        "colab_runtime": os.environ.get("COLAB_RELEASE_TAG"),
    }
    try:
        import jax  # type: ignore

        info["accelerators"] = [str(d) for d in jax.devices()]
    except Exception:  # noqa: BLE001 - jax optional
        info["accelerators"] = None
    return info


def write_run_manifest(out_dir: str | Path, cfg: dict[str, Any], **extra: Any) -> Path:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    manifest = {
        "created_utc": datetime.now(UTC).isoformat(timespec="seconds"),
        "config_hash": config_hash(cfg),
        "config": cfg,
        "environment": environment_info(),
        **extra,
    }
    path = out / "run_manifest.json"
    path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
    return path


In [ ]:
%%writefile src/sctwin/splits.py
"""Temporal and group hold-out splits with immutable manifests.

Roles:
  fit   - first ``fit_days`` dates of the official train split, non-held-out series
  val   - remaining dates of official train, non-held-out series
  test  - official eval split (locked; accessed only via the test ledger, Step 10)
  group hold-out - series whose store OR product is held out; never used for fitting or tuning.
"""

from __future__ import annotations

import hashlib
import json
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl

from sctwin.data import SERIES_KEY

MANIFEST_VERSION = 1


def _pick(ids: np.ndarray, frac: float, seed: int, salt: str) -> list[int]:
    """Deterministic subset independent of input order: rank ids by a seeded hash."""
    ids = np.unique(ids)
    n = int(round(frac * ids.size))
    keyed = sorted(
        ids.tolist(),
        key=lambda i: hashlib.sha256(f"{seed}:{salt}:{i}".encode()).hexdigest(),
    )
    return sorted(int(i) for i in keyed[:n])


def make_manifest(
    train: pl.DataFrame,
    *,
    fit_days: int,
    val_days: int,
    holdout_store_frac: float,
    holdout_product_frac: float,
    seed: int,
    source: dict[str, Any] | None = None,
) -> dict[str, Any]:
    dates = sorted(train["dt"].str.to_date().unique().to_list())
    if len(dates) != fit_days + val_days:
        raise ValueError(
            f"train has {len(dates)} distinct dates, config expects {fit_days}+{val_days}"
        )
    stores = _pick(train["store_id"].to_numpy(), holdout_store_frac, seed, "store")
    products = _pick(train["product_id"].to_numpy(), holdout_product_frac, seed, "product")
    series = train.select(SERIES_KEY).unique()
    hs, hp = pl.col("store_id").is_in(stores), pl.col("product_id").is_in(products)
    counts = series.select(
        total=pl.len(),
        in_sample=(~hs & ~hp).sum(),
        holdout_store_only=(hs & ~hp).sum(),
        holdout_product_only=(~hs & hp).sum(),
        holdout_both=(hs & hp).sum(),
    ).row(0, named=True)
    return {
        "manifest_version": MANIFEST_VERSION,
        "seed": seed,
        "source": source or {},
        "temporal": {
            "fit": {"start": str(dates[0]), "end": str(dates[fit_days - 1]), "n_days": fit_days},
            "val": {"start": str(dates[fit_days]), "end": str(dates[-1]), "n_days": val_days},
            "test": {"split": "official eval", "locked": True},
        },
        "group_holdout": {
            "rule": "series is held out if store_id in stores OR product_id in products",
            "store_frac": holdout_store_frac,
            "product_frac": holdout_product_frac,
            "stores": stores,
            "products": products,
        },
        "series_counts": {k: int(v) for k, v in counts.items()},
    }


def manifest_digest(m: dict[str, Any]) -> str:
    return hashlib.sha256(json.dumps(m, sort_keys=True).encode()).hexdigest()


def write_manifest(m: dict[str, Any], path: str | Path, force: bool = False) -> Path:
    """Write once. Re-writing identical content is a no-op; different content needs force."""
    p = Path(path)
    if p.exists() and not force:
        old = json.loads(p.read_text(encoding="utf-8"))
        old_digest = old.pop("digest", None)
        if old_digest != manifest_digest(m):
            raise FileExistsError(f"{p} exists with different content; pass force=True to replace")
        return p
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps({**m, "digest": manifest_digest(m)}, indent=2), encoding="utf-8")
    return p


def read_manifest(path: str | Path) -> dict[str, Any]:
    m = json.loads(Path(path).read_text(encoding="utf-8"))
    digest = m.pop("digest")
    if digest != manifest_digest(m):
        raise ValueError(f"manifest {path} was modified after writing (digest mismatch)")
    return m


def assign_roles(df: pl.DataFrame, m: dict[str, Any]) -> pl.DataFrame:
    """Add ``role`` (fit/val) and ``holdout`` (bool) columns to official-train rows."""
    fit_end = pl.lit(m["temporal"]["fit"]["end"]).str.to_date()
    g = m["group_holdout"]
    return df.with_columns(
        role=pl.when(pl.col("dt").str.to_date() <= fit_end)
        .then(pl.lit("fit"))
        .otherwise(pl.lit("val")),
        holdout=pl.col("store_id").is_in(g["stores"]) | pl.col("product_id").is_in(g["products"]),
    )


In [ ]:
%%writefile src/sctwin/audit.py
"""Data audit for FreshRetailNet-style data.

Every check is a pure function of a DataFrame returning a JSON-serializable dict, so the same
code runs on the real data and on synthetic fixtures with planted defects. The audit's purpose
is to establish *what the data measures* (normalization, stockout semantics) before any
modelling choice is made; it reports evidence, it does not decide.
"""

from __future__ import annotations

from typing import Any

import numpy as np
import polars as pl

from sctwin.data import KEY, N_HOURS, SERIES_KEY, hourly_matrix, validate_schema

# Hours covered by stock_hour6_22_cnt are ambiguous in the card ("between 6:00 and 22:00").
CNT_WINDOWS = {"h6_21": (6, 22), "h6_22": (6, 23), "h7_22": (7, 23)}


def _f(x: Any) -> float:
    return float(np.round(float(x), 8))


def _mean(x: np.ndarray) -> float | None:
    """Mean that returns None (valid JSON) instead of NaN on an empty selection."""
    return _f(x.mean()) if x.size else None


def _with_dates(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns(pl.col("dt").str.to_date(strict=False).alias("_date"))


# --------------------------------------------------------------------------- structure


def check_structure(df: pl.DataFrame) -> dict[str, Any]:
    sch = validate_schema(df)
    d = _with_dates(df)
    n_dup = d.height - d.unique(subset=list(KEY)).height
    per_series = d.group_by(list(SERIES_KEY)).agg(
        n_days=pl.len(),
        first=pl.col("_date").min(),
        last=pl.col("_date").max(),
    )
    span = (per_series["last"] - per_series["first"]).dt.total_days() + 1
    gaps = int((span != per_series["n_days"]).sum())
    nulls = {c: int(v) for c, v in d.null_count().row(0, named=True).items() if v}
    return {
        "rows": d.height,
        "schema_ok": sch.ok,
        "schema_missing": sch.missing,
        "schema_unexpected": sch.unexpected,
        "schema_wrong_type": {k: list(v) for k, v in sch.wrong_type.items()},
        "unparseable_dates": int(d["_date"].null_count()),
        "duplicate_keys": n_dup,
        "n_series": per_series.height,
        "n_stores": d["store_id"].n_unique(),
        "n_products": d["product_id"].n_unique(),
        "n_cities": d["city_id"].n_unique(),
        "date_min": str(d["_date"].min()),
        "date_max": str(d["_date"].max()),
        "n_distinct_dates": d["_date"].n_unique(),
        "days_per_series": {
            "min": int(per_series["n_days"].min()),  # type: ignore[arg-type]
            "max": int(per_series["n_days"].max()),  # type: ignore[arg-type]
        },
        "series_with_date_gaps": gaps,
        "series_not_sharing_calendar": int(
            int((per_series["first"] != per_series["first"].min()).sum())
            + int((per_series["last"] != per_series["last"].max()).sum())
        ),
        "null_counts": nulls,
        "hours_sale_len": _len_range(df, "hours_sale"),
        "hours_stock_status_len": _len_range(df, "hours_stock_status"),
    }


def _len_range(df: pl.DataFrame, col: str) -> dict[str, int]:
    lens = df[col].list.len()
    return {
        "min": int(lens.min()),  # type: ignore[arg-type]
        "max": int(lens.max()),  # type: ignore[arg-type]
        "rows_not_24": int((lens != N_HOURS).sum()),
    }


# --------------------------------------------------------------------------- values


def check_values(df: pl.DataFrame) -> dict[str, Any]:
    hs = hourly_matrix(df, "hours_sale")
    daily = df["sale_amount"].to_numpy()
    diff = np.abs(hs.sum(axis=1) - daily)
    out: dict[str, Any] = {
        "hourly_nan": int(np.isnan(hs).sum()),
        "hourly_negative": int((hs < 0).sum()),
        "daily_negative": int((daily < 0).sum()),
        "daily_vs_hourly_sum": {
            "max_abs_diff": _f(np.nanmax(diff)),
            "frac_rows_abs_diff_gt_1e-6": _f((diff > 1e-6).mean()),
            "frac_rows_rel_diff_gt_1pct": _f(
                (diff > 0.01 * np.maximum(np.abs(daily), 1e-12)).mean()
            ),
        },
        "frac_zero_days": _f((daily == 0).mean()),
        "frac_zero_hours": _f((hs == 0).mean()),
        "daily_sale_quantiles": {
            str(q): _f(np.quantile(daily, q)) for q in (0.0, 0.25, 0.5, 0.75, 0.9, 0.99, 1.0)
        },
    }
    return out


def detect_quantum(
    values: np.ndarray, max_divisor: int = 12, tol: float = 1e-6, target: float = 0.999
) -> dict[str, Any]:
    """Look for a global quantum q such that values are integer multiples of q.

    Evidence only: a quantum existing does NOT establish that v/q are transaction counts
    (weighed products, rounding by the provider, etc. could produce the same pattern).
    """
    v = np.asarray(values, dtype=np.float64)
    v = v[np.isfinite(v) & (v > 0)]
    if v.size == 0:
        return {"n_positive": 0, "quantum": None}
    vmin = float(v.min())
    distinct = np.unique(v)
    candidates = []
    chosen = None
    for k in range(1, max_divisor + 1):
        q = vmin / k
        r = v / q
        frac = float((np.abs(r - np.round(r)) < tol * np.maximum(1.0, r)).mean())
        candidates.append({"divisor": k, "q": q, "frac_integer": _f(frac)})
        if chosen is None and frac >= target:
            chosen = {"divisor": k, "q": q, "frac_integer": _f(frac)}
    head = distinct[:10]
    return {
        "n_positive": int(v.size),
        "n_distinct_positive": int(distinct.size),
        "min_positive": vmin,
        "smallest_distinct": [float(x) for x in head],
        "smallest_distinct_over_min": [_f(x / vmin) for x in head],
        "min_gap_between_distinct": float(np.diff(distinct).min()) if distinct.size > 1 else None,
        "candidates": candidates,
        "quantum": chosen,
    }


def check_normalization(df: pl.DataFrame, sample: int = 2_000_000, seed: int = 0) -> dict[str, Any]:
    hs = hourly_matrix(df, "hours_sale").ravel()
    rng = np.random.default_rng(seed)
    pos = hs[hs > 0]
    if pos.size > sample:
        pos = rng.choice(pos, sample, replace=False)
    overall = detect_quantum(pos)
    # Per management group: weighed vs counted goods may differ.
    per_group: dict[str, Any] = {}
    if overall["quantum"] is not None:
        q = overall["quantum"]["q"]
        for (g,), sub in df.group_by(["management_group_id"]):
            v = hourly_matrix(sub, "hours_sale").ravel()
            v = v[v > 0]
            r = v / q
            per_group[str(g)] = {
                "n_positive": int(v.size),
                "frac_integer_multiple": _f(
                    (np.abs(r - np.round(r)) < 1e-6 * np.maximum(1, r)).mean()
                ),
            }
    daily = df["sale_amount"].to_numpy()
    return {"hourly": overall, "daily": detect_quantum(daily), "per_management_group": per_group}


# --------------------------------------------------------------------------- stockouts


def _runs(row: np.ndarray) -> list[tuple[int, int]]:
    """(start, end_exclusive) of runs of ones."""
    padded = np.concatenate(([0], row, [0]))
    d = np.diff(padded)
    return list(zip(np.flatnonzero(d == 1), np.flatnonzero(d == -1), strict=True))


def check_stockouts(df: pl.DataFrame, run_sample: int = 200_000, seed: int = 0) -> dict[str, Any]:
    st = hourly_matrix(df, "hours_stock_status", np.int8)
    hs = hourly_matrix(df, "hours_sale")
    cnt = df["stock_hour6_22_cnt"].to_numpy()
    values, counts = np.unique(st, return_counts=True)
    so = st == 1
    ins = st == 0

    cnt_match = {
        name: _f((st[:, a:b].sum(axis=1) == cnt).mean()) for name, (a, b) in CNT_WINDOWS.items()
    }

    # Is the flag "out of stock for the whole hour" or "went out of stock during the hour"?
    # Compare sales in the first flagged hour of a run with later flagged hours.
    rng = np.random.default_rng(seed)
    idx = np.flatnonzero(so.any(axis=1))
    if idx.size > run_sample:
        idx = rng.choice(idx, run_sample, replace=False)
    first_sales, later_sales, recover, runs_per_day, end_at_23 = [], [], 0, [], 0
    n_runs = 0
    for i in idx:
        runs = _runs(st[i])
        runs_per_day.append(len(runs))
        for a, b in runs:
            n_runs += 1
            first_sales.append(hs[i, a])
            later_sales.extend(hs[i, a + 1 : b].tolist())
            if b < N_HOURS:
                recover += 1
            else:
                end_at_23 += 1
    first_arr = np.asarray(first_sales)
    later_arr = np.asarray(later_sales)

    by_hour = {
        "stockout_rate": [_f(x) for x in so.mean(axis=0)],
        "mean_sale_instock": [_f(x) for x in _masked_mean(hs, ins)],
        "mean_sale_stockout": [_f(x) for x in _masked_mean(hs, so)],
        "frac_positive_sale_stockout": [_f(x) for x in _masked_mean((hs > 0).astype(float), so)],
    }
    return {
        "status_values": {str(int(v)): int(c) for v, c in zip(values, counts, strict=True)},
        "frac_hours_stockout": _f(so.mean()),
        "frac_days_any_stockout": _f(so.any(axis=1).mean()),
        "frac_days_all_24_stockout": _f(so.all(axis=1).mean()),
        "frac_days_all_6_21_stockout": _f(so[:, 6:22].all(axis=1).mean()),
        "cnt_field_match_by_window": cnt_match,
        "frac_stockout_hours_with_positive_sale": _f((hs[so] > 0).mean()) if so.any() else None,
        "mean_sale_stockout_hours": _f(hs[so].mean()) if so.any() else None,
        "mean_sale_instock_hours": _f(hs[ins].mean()) if ins.any() else None,
        "runs": {
            "days_sampled": int(idx.size),
            "n_runs": n_runs,
            "mean_runs_per_stockout_day": _f(np.mean(runs_per_day)) if runs_per_day else None,
            "frac_runs_recovering_same_day": _f(recover / n_runs) if n_runs else None,
            "frac_runs_ending_at_hour_23": _f(end_at_23 / n_runs) if n_runs else None,
            "mean_sale_first_flagged_hour": _f(first_arr.mean()) if first_arr.size else None,
            "frac_positive_first_flagged_hour": _f((first_arr > 0).mean())
            if first_arr.size
            else None,
            "mean_sale_later_flagged_hours": _f(later_arr.mean()) if later_arr.size else None,
            "frac_positive_later_flagged_hours": _f((later_arr > 0).mean())
            if later_arr.size
            else None,
        },
        "by_hour": by_hour,
    }


def _status_values(df: pl.DataFrame) -> dict[str, int]:
    v, c = np.unique(hourly_matrix(df, "hours_stock_status", np.int8), return_counts=True)
    return {str(int(a)): int(b) for a, b in zip(v, c, strict=True)}


def _masked_mean(x: np.ndarray, m: np.ndarray) -> np.ndarray:
    n = m.sum(axis=0)
    s = np.where(m, x, 0).sum(axis=0)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(n > 0, s / np.maximum(n, 1), np.nan)


def check_lattice(df: pl.DataFrame, steps: tuple[float, ...] = (0.01, 0.05, 0.1)) -> dict[str, Any]:
    """Fraction of positive hourly values on each candidate lattice, overall / per series /
    per management group. A lattice is evidence of discreteness, NOT proof of unit counts."""
    hs = hourly_matrix(df, "hours_sale")
    pos = hs > 0
    sid = df.select(pl.struct(*SERIES_KEY).rank("dense")).to_series().to_numpy() - 1
    n_series = int(sid.max()) + 1
    mg = df["management_group_id"].to_numpy()
    out: dict[str, Any] = {}
    for step in steps:
        r = hs / step
        on = np.abs(r - np.round(r)) < 1e-6 * np.maximum(1.0, r)
        off_rows = (pos & ~on).sum(axis=1)
        ser_off = np.bincount(sid, weights=off_rows, minlength=n_series)
        ser_pos = np.bincount(sid, weights=pos.sum(axis=1), minlength=n_series)
        out[str(step)] = {
            "frac_positive_on_lattice": _mean(on[pos]),
            "frac_series_fully_on_lattice": _f((ser_off == 0).mean()),
            "frac_series_gt5pct_off_lattice": _f((ser_off / np.maximum(ser_pos, 1) > 0.05).mean()),
            "per_management_group": {
                str(g): _mean(on[mg == g][pos[mg == g]]) for g in np.unique(mg)
            },
        }
    return out


def check_availability_transitions(df: pl.DataFrame, min_full_days: int = 10) -> dict[str, Any]:
    """Within-series sales around stockout-flag transitions, relative to the same series'
    mean at the same hour on days fully in stock over hours 6-21.

    Ratios near 1 before a run would suggest exogenous stockouts; ratios >> 1 indicate
    demand-driven depletion (availability is endogenous). The first flagged hour's ratio
    measures residual sales (partial availability or flag noise). Hours 6-21 only.
    """
    d = df.sort([*SERIES_KEY, "dt"])
    hs = hourly_matrix(d, "hours_sale")
    st = hourly_matrix(d, "hours_stock_status", np.int8)
    sid = d.select(pl.struct(*SERIES_KEY).rank("dense")).to_series().to_numpy() - 1
    n_series = int(sid.max()) + 1
    full = st[:, 6:22].sum(axis=1) == 0
    base = np.zeros((n_series, N_HOURS))
    n_full = np.bincount(sid[full], minlength=n_series)
    np.add.at(base, sid[full], hs[full])
    base /= np.maximum(n_full, 1)[:, None]
    ok = n_full >= min_full_days

    prev = np.zeros_like(st)
    prev[:, 1:] = st[:, :-1]
    start = (st == 1) & (prev == 0)
    end = (st == 0) & (prev == 1)
    start[:, 0] = end[:, 0] = False

    def ratio(mask: np.ndarray, offset: int) -> dict[str, Any]:
        r, h = np.nonzero(mask)
        hh = h + offset
        keep = (hh >= 6) & (hh <= 21) & ok[sid[r]]
        r, hh = r[keep], hh[keep]
        den = base[sid[r], hh].sum()
        return {"n": int(r.size), "ratio": _f(hs[r, hh].sum() / den) if den > 0 else None}

    return {
        "series_with_baseline": int(ok.sum()),
        "min_full_days": min_full_days,
        "before_run_start": {f"-{k}h": ratio(start, -k) for k in (3, 2, 1)},
        "first_flagged_hour": ratio(start, 0),
        "second_flagged_hour": ratio(start & np.roll(st == 1, -1, axis=1), 1),
        "first_instock_after_restock": ratio(end, 0),
        "second_instock_after_restock": ratio(end, 1),
    }


HIERARCHY_EDGES = (
    ("store_id", "city_id"),
    ("first_category_id", "management_group_id"),
    ("second_category_id", "first_category_id"),
    ("third_category_id", "second_category_id"),
    ("product_id", "third_category_id"),
)


def check_hierarchy(df: pl.DataFrame) -> dict[str, Any]:
    """Each child id must map to exactly one parent id (strict trees)."""
    out: dict[str, Any] = {}
    for child, parent in HIERARCHY_EDGES:
        n_par = df.group_by(child).agg(pl.col(parent).n_unique())[parent]
        out[f"{child}->{parent}"] = {
            "n_children": int(n_par.len()),
            "children_with_multiple_parents": int((n_par > 1).sum()),
        }
    out["is_strict_tree"] = all(
        v["children_with_multiple_parents"] == 0 for k, v in out.items() if "->" in k
    )
    return out


# --------------------------------------------------------------------------- covariates


def check_covariates(df: pl.DataFrame) -> dict[str, Any]:
    disc = df["discount"].to_numpy()

    def nuniq_within(group: list[str], col: str) -> int:
        return int(df.group_by(group).agg(pl.col(col).n_unique())[col].max())  # type: ignore[arg-type]

    weather = ["precpt", "avg_temperature", "avg_humidity", "avg_wind_level"]
    return {
        "discount": {
            "min": _f(disc.min()),
            "max": _f(disc.max()),
            "frac_discounted_lt_1": _f((disc < 1).mean()),
            "frac_gt_1": _f((disc > 1).mean()),
        },
        "activity_flag_rate": _f(df["activity_flag"].mean()),
        "holiday_flag_rate": _f(df["holiday_flag"].mean()),
        # Granularity evidence: 1 => covariate constant at that level.
        "max_distinct_holiday_per_date": nuniq_within(["dt"], "holiday_flag"),
        "max_distinct_weather_per_city_date": {
            w: nuniq_within(["city_id", "dt"], w) for w in weather
        },
        "max_distinct_weather_per_store_date": {
            w: nuniq_within(["store_id", "dt"], w) for w in weather
        },
        "max_distinct_activity_per_store_date": nuniq_within(["store_id", "dt"], "activity_flag"),
        "max_distinct_city_per_store": nuniq_within(["store_id"], "city_id"),
        "max_distinct_category_per_product": nuniq_within(["product_id"], "third_category_id"),
    }


# --------------------------------------------------------------------------- cross split


def check_cross_split(train: pl.DataFrame, test: pl.DataFrame) -> dict[str, Any]:
    tr, te = _with_dates(train), _with_dates(test)
    s_tr = tr.select(SERIES_KEY).unique()
    s_te = te.select(SERIES_KEY).unique()
    last_tr = tr.group_by(list(SERIES_KEY)).agg(last_train=pl.col("_date").max())
    first_te = te.group_by(list(SERIES_KEY)).agg(first_test=pl.col("_date").min())
    j = last_tr.join(first_te, on=list(SERIES_KEY), how="inner")
    gap = (j["first_test"] - j["last_train"]).dt.total_days()
    overlap = tr.select(KEY).join(te.select(KEY), on=list(KEY), how="inner").height
    return {
        "series_train": s_tr.height,
        "series_test": s_te.height,
        "series_in_both": s_tr.join(s_te, on=list(SERIES_KEY), how="inner").height,
        "series_test_only": s_te.join(s_tr, on=list(SERIES_KEY), how="anti").height,
        "overlapping_keys": overlap,
        "series_test_not_strictly_after_train": int((gap <= 0).sum()),
        "gap_days_min": int(gap.min()) if gap.len() else None,  # type: ignore[arg-type]
        "gap_days_max": int(gap.max()) if gap.len() else None,  # type: ignore[arg-type]
    }


# --------------------------------------------------------------------------- gates


def gate(report: dict[str, Any]) -> dict[str, bool]:
    """Pass/fail gates for Step 1. Evidence fields (normalization, stockouts) are not gated."""
    g: dict[str, bool] = {}
    for split in ("train", "eval"):
        s = report[split]["structure"]
        v = report[split]["values"]
        g[f"{split}.schema_ok"] = s["schema_ok"]
        g[f"{split}.no_duplicate_keys"] = s["duplicate_keys"] == 0
        g[f"{split}.dates_parse"] = s["unparseable_dates"] == 0
        g[f"{split}.arrays_len_24"] = (
            s["hours_sale_len"]["rows_not_24"] == 0
            and s["hours_stock_status_len"]["rows_not_24"] == 0
        )
        g[f"{split}.no_negative_or_nan_sales"] = (
            v["hourly_nan"] == 0 and v["hourly_negative"] == 0 and v["daily_negative"] == 0
        )
        g[f"{split}.status_binary"] = set(report[split]["stockouts"]["status_values"]) <= {"0", "1"}
    g["train.hierarchy_strict_tree"] = report["train"]["hierarchy"]["is_strict_tree"]
    c = report["cross_split"]
    g["cross.no_overlapping_keys"] = c["overlapping_keys"] == 0
    g["cross.test_strictly_after_train"] = c["series_test_not_strictly_after_train"] == 0
    return g


def run_audit(train: pl.DataFrame, test: pl.DataFrame) -> dict[str, Any]:
    """Full evidence audit on train; integrity-only audit on the locked test split, so that no
    distributional finding from the test set can inform modelling decisions."""
    report: dict[str, Any] = {
        "train": {
            "structure": check_structure(train),
            "values": check_values(train),
            "normalization": check_normalization(train),
            "stockouts": check_stockouts(train),
            "lattice": check_lattice(train),
            "availability_transitions": check_availability_transitions(train),
            "covariates": check_covariates(train),
            "hierarchy": check_hierarchy(train),
        },
        "eval": {
            "structure": check_structure(test),
            "values": check_values(test),
            "stockouts": {"status_values": _status_values(test)},
        },
    }
    report["cross_split"] = check_cross_split(train, test)
    report["gates"] = gate(report)
    report["all_gates_pass"] = all(report["gates"].values())
    return report


In [ ]:
%%writefile src/sctwin/panel.py
"""Dense panel representation: series x day x hour.

Valid because every series shares one complete calendar (Step 1 audit). ``build_panel`` checks
that the input is a full rectangle and raises otherwise, so the reshape cannot silently mix
series or dates.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from datetime import date

import numpy as np
import polars as pl

from sctwin.data import KEY, N_HOURS, SERIES_KEY, hourly_matrix

HIERARCHY = (
    "store_id",
    "product_id",
    "city_id",
    "management_group_id",
    "first_category_id",
    "second_category_id",
    "third_category_id",
)
WEATHER = ("precpt", "avg_temperature", "avg_humidity", "avg_wind_level")
DAY_HOURS = (6, 22)  # hours 6..21: the window covered by stock_hour6_22_cnt

# Hourly availability classes (docs/data_model.md section 4.3)
CLS_A, CLS_P_PRE, CLS_P_FIRST, CLS_U, CLS_NIGHT = 0, 1, 2, 3, 4


@dataclass
class Panel:
    keys: pl.DataFrame  # (S,) hierarchy columns, row i <-> series i
    dates: list[date]  # (T,)
    sales: np.ndarray  # (S, T, 24) float64, normalized units
    stock: np.ndarray  # (S, T, 24) int8, 1 = out of stock
    discount: np.ndarray  # (S, T) float64
    activity: np.ndarray  # (S, T) int8
    holiday: np.ndarray  # (T,) int8
    weather: np.ndarray  # (S, T, 4) float64, order = WEATHER
    meta: dict[str, object] = field(default_factory=dict)

    @property
    def shape(self) -> tuple[int, int]:
        return self.sales.shape[0], self.sales.shape[1]

    def subset(self, idx: np.ndarray | list[int]) -> Panel:
        i = np.asarray(idx)
        return Panel(
            keys=self.keys[i.tolist()],
            dates=list(self.dates),
            sales=self.sales[i],
            stock=self.stock[i],
            discount=self.discount[i],
            activity=self.activity[i],
            holiday=self.holiday.copy(),
            weather=self.weather[i],
            meta=dict(self.meta),
        )


def build_panel(df: pl.DataFrame) -> Panel:
    if df.select(pl.struct(*KEY).is_duplicated().any()).item():
        raise ValueError("duplicate (store_id, product_id, dt) keys")
    d = df.sort([*SERIES_KEY, "dt"])
    keys = d.select(HIERARCHY).unique(subset=list(SERIES_KEY), maintain_order=True)
    dates = sorted(d["dt"].str.to_date().unique().to_list())
    n_series, n_days = keys.height, len(dates)
    if d.height != n_series * n_days:
        raise ValueError(f"not a full rectangle: {d.height} rows != {n_series} x {n_days}")

    def grid(col: str, dtype: type) -> np.ndarray:
        return d[col].to_numpy().astype(dtype).reshape(n_series, n_days)

    hol = grid("holiday_flag", np.int8)
    if not (hol == hol[:1]).all():
        raise ValueError("holiday_flag differs across series on the same date")
    return Panel(
        keys=keys,
        dates=dates,
        sales=hourly_matrix(d, "hours_sale").reshape(n_series, n_days, N_HOURS),
        stock=hourly_matrix(d, "hours_stock_status", np.int8).reshape(n_series, n_days, N_HOURS),
        discount=grid("discount", np.float64),
        activity=grid("activity_flag", np.int8),
        holiday=hol[0].copy(),
        weather=np.stack([grid(w, np.float64) for w in WEATHER], axis=-1),
    )


def availability_classes(stock: np.ndarray) -> np.ndarray:
    """Per-hour class codes from a (..., 24) 0/1 out-of-stock array (within-day neighbours).

    A: in stock, next hour in stock          P_PRE: in stock, next hour out (depletion hour)
    P_FIRST: out, previous hour in stock     U: out, previous hour out
    NIGHT: hours outside 6..21 regardless of flags.
    """
    st = stock.astype(bool)
    nxt = np.empty_like(st)
    nxt[..., :-1] = st[..., 1:]
    nxt[..., -1] = st[..., -1]
    prv = np.zeros_like(st)
    prv[..., 1:] = st[..., :-1]
    cls = np.where(
        st,
        np.where(prv, CLS_U, CLS_P_FIRST),
        np.where(nxt, CLS_P_PRE, CLS_A),
    ).astype(np.int8)
    hours = np.arange(N_HOURS)
    night = (hours < DAY_HOURS[0]) | (hours >= DAY_HOURS[1])
    cls[..., night] = CLS_NIGHT
    return cls


## Files: Package: point-in-time features and backtesting
Resampling to the backtest unit, as-of-T features, rolling-origin folds (CLAUDE.md Rules 1-2).

In [ ]:
%%writefile src/sctwin/timeunit.py
"""Resampling to a configurable time unit ("day" or "hour") with availability semantics.

Unit index ``u``: the day index for "day", ``day * 24 + hour`` for "hour". Origin ``T`` means
"end of unit T": every value whose availability time is <= end of unit T may be used.

Availability (see CLAUDE.md): hourly sales/flags are available at the end of their hour; every
daily field (weather, discount, activity, daily totals) at the end of its day; the calendar
(weekday, hour of day, holiday_flag) is treated as known in advance.
"""

from __future__ import annotations

from dataclasses import dataclass
from datetime import date

import numpy as np

from sctwin.panel import DAY_HOURS, WEATHER, Panel, availability_classes

UNITS = ("day", "hour")
TIMEZONE = "Asia/Shanghai"  # assumed store-local time (UTC+8, no DST); see CLAUDE.md


@dataclass
class UnitSeries:
    unit: str
    sales: np.ndarray  # (S, N) sales per unit, normalized
    stockout: np.ndarray  # (S, N) hour: 0/1 flag; day: stockout hours in 6..21
    instock_next_known: np.ndarray  # (S, N) bool: unit fully available (needs unit u+1 for hour)
    day_of_unit: np.ndarray  # (N,) day index of each unit
    hour_of_unit: np.ndarray  # (N,) hour of day (0 for "day")
    dates: list[date]  # (D,) calendar days
    holiday: np.ndarray  # (D,) calendar, known in advance
    weather: np.ndarray  # (S, D, 4) available at end of its day
    discount: np.ndarray  # (S, D) available at end of its day
    activity: np.ndarray  # (S, D) available at end of its day
    units_per_day: int
    # hour only: the in-stock indicator of unit u depends on unit u + 1 (lookahead of 1 unit)
    instock_lookahead: int
    # hour only, EVALUATION ONLY (uses neighbouring hours): availability class of each hour
    eval_class: np.ndarray | None = None

    @property
    def n_units(self) -> int:
        return self.sales.shape[1]

    def last_complete_day(self, origin: np.ndarray) -> np.ndarray:
        """Index of the last day fully observed at the end of unit ``origin`` (-1 if none)."""
        o = np.asarray(origin)
        return (o + 1) // self.units_per_day - 1


def resample(p: Panel, unit: str) -> UnitSeries:
    if unit not in UNITS:
        raise ValueError(f"unit must be one of {UNITS}, got {unit!r}")
    n_series, n_days = p.shape
    a, b = DAY_HOURS
    common = dict(
        dates=list(p.dates),
        holiday=p.holiday.astype(np.float64),
        weather=p.weather,
        discount=p.discount,
        activity=p.activity.astype(np.float64),
    )
    if unit == "day":
        so = p.stock[:, :, a:b].sum(axis=-1).astype(np.float64)
        return UnitSeries(
            unit="day",
            sales=p.sales.sum(axis=-1),
            stockout=so,
            instock_next_known=so == 0,  # a complete day with no stockout hour in 6..21
            day_of_unit=np.arange(n_days),
            hour_of_unit=np.zeros(n_days, dtype=np.int64),
            units_per_day=1,
            instock_lookahead=0,
            **common,  # type: ignore[arg-type]
        )
    sales = p.sales.reshape(n_series, n_days * 24)
    flag = p.stock.reshape(n_series, n_days * 24).astype(np.float64)
    nxt = np.ones_like(flag)  # beyond the data: unknown -> treated as not in stock
    nxt[:, :-1] = flag[:, 1:]
    return UnitSeries(
        unit="hour",
        sales=sales,
        stockout=flag,
        instock_next_known=(flag == 0) & (nxt == 0),  # class A: in stock now and next hour
        day_of_unit=np.repeat(np.arange(n_days), 24),
        hour_of_unit=np.tile(np.arange(24), n_days),
        units_per_day=24,
        instock_lookahead=1,
        eval_class=availability_classes(p.stock).reshape(n_series, n_days * 24),
        **common,  # type: ignore[arg-type]
    )


def weather_names() -> tuple[str, ...]:
    return WEATHER


In [ ]:
%%writefile src/sctwin/features.py
"""Point-in-time features at a forecast origin T, for a configurable unit ("day" / "hour").

Follows CLAUDE.md Rule 2:

* every feature is computed as of the origin T (end of unit T), never as of the target time;
* "lag k" = value of unit T - k; rolling windows cover units [T - w + 1, T] (backward only);
* daily fields (weather, discount, activity) come from the last *complete* day at T;
* an in-stock indicator that needs unit u + 1 is only used when u + 1 <= T;
* target-time inputs are restricted to the calendar (steps ahead, weekday, hour, holiday).

Rows are (series, origin T, step h) with target unit u = T + L + h, h = 1..H.
Outcome columns start with ``y`` and are never features.
"""

from __future__ import annotations

import warnings
from dataclasses import dataclass
from typing import Any

import numpy as np
import polars as pl

from sctwin.panel import HIERARCHY
from sctwin.timeunit import UnitSeries, weather_names

# --------------------------------------------------------------------------- primitives


def trailing_mean(x: np.ndarray, k: int, mask: np.ndarray | None = None) -> np.ndarray:
    """Mean over units [j-k, j-1] along axis 1 for each index j (strictly earlier units),
    optionally restricted to ``mask``. NaN where no qualifying unit exists."""
    x = np.asarray(x, dtype=np.float64)
    m = np.ones(x.shape, dtype=np.float64) if mask is None else mask.astype(np.float64)
    zeros = np.zeros((x.shape[0], 1, *x.shape[2:]))
    cs = np.concatenate([zeros, np.cumsum(x * m, axis=1)], axis=1)
    cn = np.concatenate([zeros, np.cumsum(m, axis=1)], axis=1)
    j = np.arange(x.shape[1])
    lo = np.maximum(j - k, 0)
    s = cs[:, j] - cs[:, lo]
    n = cn[:, j] - cn[:, lo]
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(n > 0, s / np.maximum(n, 1), np.nan)


def units_since(event: np.ndarray) -> np.ndarray:
    """For each j: j - (last unit < j with event). NaN if none."""
    n_series, n_units = event.shape
    out = np.full((n_series, n_units), np.nan)
    last = np.full(n_series, np.nan)
    for j in range(n_units):
        out[:, j] = j - last
        last = np.where(event[:, j], j, last)
    return out


def _pad(x: np.ndarray) -> np.ndarray:
    """Append one dummy unit so index N ("information up to the end of unit N-1") exists.
    Strictly-earlier statistics never read it."""
    return np.concatenate([x, np.zeros((x.shape[0], 1, *x.shape[2:]), dtype=x.dtype)], axis=1)


def nanmean_rows(v: np.ndarray) -> np.ndarray:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(v, axis=1)


# --------------------------------------------------------------------------- config


@dataclass(frozen=True)
class FeatureConfig:
    lags: tuple[int, ...]  # sale_lag{k} = sales of unit T - k
    windows: tuple[int, ...]  # rolling means over the last w units
    season: int  # main seasonal period in units
    n_seasons: int  # seasonal history length (in periods) for seasonal statistics
    extra_seasons: tuple[int, ...]  # additional seasonal "last value" periods
    stockout_windows: tuple[int, ...]
    cov_days: int = 7  # rolling window for daily covariates (in days)

    @staticmethod
    def default(unit: str) -> FeatureConfig:
        if unit == "day":
            return FeatureConfig(lags=(0, 1, 6, 13), windows=(7, 14, 28), season=7, n_seasons=8,
                                 extra_seasons=(), stockout_windows=(7, 28))  # fmt: skip
        return FeatureConfig(
            lags=(0, 1, 2, 23, 167),
            windows=(24, 168, 672),
            season=24,
            n_seasons=28,
            extra_seasons=(168,),
            stockout_windows=(24, 168),
        )


# --------------------------------------------------------------------------- builder


class FeatureBuilder:
    """Builds as-of-T feature rows from a :class:`UnitSeries`.

    Internal convention: ``info[name][:, j]`` uses only units < j, so at origin T the value is
    column ``j = T + 1``.
    """

    def __init__(self, us: UnitSeries, keys: pl.DataFrame, cfg: FeatureConfig | None = None):
        self.us = us
        self.keys = keys
        self.cfg = cfg or FeatureConfig.default(us.unit)
        c = self.cfg
        x, so = _pad(us.sales), _pad(us.stockout)
        info: dict[str, np.ndarray] = {}
        for w in c.windows:
            info[f"sale_mean{w}"] = trailing_mean(x, w)
            info[f"frac_zero{w}"] = trailing_mean(x == 0, w)
        for w in c.stockout_windows:
            info[f"stockout_frac{w}"] = trailing_mean(so > 0, w)
            info[f"stockout_mean{w}"] = trailing_mean(so, w)
        info["units_since_stockout"] = units_since(so > 0)
        # In-stock mean: the indicator of unit u needs unit u + lookahead, so at origin T only
        # units u <= T - lookahead qualify: shift the information index by the lookahead.
        la = us.instock_lookahead
        w = max(c.windows)
        im = trailing_mean(x, w, mask=_pad(us.instock_next_known))
        if la:
            im = np.concatenate([np.full((im.shape[0], la), np.nan), im[:, :-la]], axis=1)
        info[f"instock_mean{w}"] = im
        self.info = info
        # daily covariates, indexed by day; at origin T use the last complete day
        bad = (us.discount <= 0) | (us.discount > 1)
        self.discount = np.where(bad, np.nan, us.discount)
        self.discount_bad = bad.astype(np.float64)
        self.cov_mean = {
            "discount_mean7d": trailing_mean(
                _pad(np.nan_to_num(self.discount, nan=1.0)), c.cov_days
            ),
            "activity_mean7d": trailing_mean(_pad(us.activity), c.cov_days),
        }
        self.dow = np.array([d.weekday() for d in us.dates], dtype=np.float64)

    # ------------------------------------------------------------------ names

    @property
    def feature_names(self) -> tuple[str, ...]:
        c = self.cfg
        names = ["steps_ahead", "target_dow", "target_holiday"]
        if self.us.unit == "hour":
            names.append("target_hour")
        names += [f"sale_lag{k}" for k in c.lags]
        names += [f"sale_mean{w}" for w in c.windows] + [f"frac_zero{w}" for w in c.windows]
        names += [f"stockout_frac{w}" for w in c.stockout_windows]
        names += [f"stockout_mean{w}" for w in c.stockout_windows]
        names += ["units_since_stockout", f"instock_mean{max(c.windows)}"]
        names += ["sale_seasonal_last", "sale_seasonal_instock_mean"]
        names += [f"sale_seasonal{p}_last" for p in c.extra_seasons]
        names += [f"{w}_lastday" for w in weather_names()]
        names += ["discount_lastday", "discount_anomaly_lastday", "activity_lastday"]
        names += list(self.cov_mean)
        return tuple(names)

    # ------------------------------------------------------------------ seasonal gathers

    @staticmethod
    def seasonal_index(origin: np.ndarray, target: np.ndarray, period: int, n: int) -> np.ndarray:
        """(rows, n) unit indices of the n most recent units at the target's seasonal position
        that are observed at the origin (index <= origin); -1 where not available."""
        k0 = np.ceil((target - origin) / period).astype(np.int64)  # smallest k: u - kP <= T
        idx = (target - period * k0)[:, None] - period * np.arange(n)[None, :]
        return np.where(idx >= 0, idx, -1)

    def seasonal_values(
        self,
        s: np.ndarray,
        origin: np.ndarray,
        target: np.ndarray,
        period: int,
        n: int,
        instock_only: bool = False,
    ) -> np.ndarray:
        idx = self.seasonal_index(origin, target, period, n)
        ok = idx >= 0
        safe = np.maximum(idx, 0)
        vals = np.where(ok, self.us.sales[s[:, None], safe], np.nan)
        if instock_only:
            known = idx + self.us.instock_lookahead <= origin[:, None]
            good = self.us.instock_next_known[s[:, None], safe]
            vals = np.where(ok & known & good, vals, np.nan)
        return vals

    # ------------------------------------------------------------------ rows

    def rows(
        self,
        origins: np.ndarray | list[int],
        lead: int,
        horizon: int,
        series_idx: np.ndarray | None = None,
        max_target: int | None = None,
    ) -> pl.DataFrame:
        us, c = self.us, self.cfg
        n_series, n_units = us.sales.shape
        s_all = np.arange(n_series) if series_idx is None else np.asarray(series_idx)
        sg, og, hg = np.meshgrid(s_all, np.asarray(origins, dtype=np.int64),
                                 np.arange(1, horizon + 1), indexing="ij")  # fmt: skip
        s, o, h = sg.ravel(), og.ravel(), hg.ravel()
        u = o + lead + h
        limit = n_units - 1 if max_target is None else min(max_target, n_units - 1)
        keep = (o >= 0) & (u <= limit)
        s, o, h, u = s[keep], o[keep], h[keep], u[keep]
        j = o + 1
        tday = us.day_of_unit[u]
        cols: dict[str, Any] = {
            "series_idx": s.astype(np.int32),
            "origin_idx": o.astype(np.int32),
            "target_idx": u.astype(np.int32),
            "step": h.astype(np.int16),
            "target_dt": np.asarray(us.dates, dtype="datetime64[D]")[tday],
        }
        for col in HIERARCHY:
            cols[col] = self.keys[col].to_numpy()[s]
        # target-time calendar (known in advance)
        cols["steps_ahead"] = (lead + h).astype(np.float64)
        cols["target_dow"] = self.dow[tday]
        cols["target_holiday"] = us.holiday[tday]
        if us.unit == "hour":
            cols["target_hour"] = us.hour_of_unit[u].astype(np.float64)
        # history up to and including unit T
        for k in c.lags:
            src = o - k
            cols[f"sale_lag{k}"] = np.where(src >= 0, us.sales[s, np.maximum(src, 0)], np.nan)
        for name, arr in self.info.items():
            cols[name] = arr[s, j]
        cols["sale_seasonal_last"] = self.seasonal_values(s, o, u, c.season, 1)[:, 0]
        cols["sale_seasonal_instock_mean"] = nanmean_rows(
            self.seasonal_values(s, o, u, c.season, c.n_seasons, instock_only=True)
        )
        for p in c.extra_seasons:
            cols[f"sale_seasonal{p}_last"] = self.seasonal_values(s, o, u, p, 1)[:, 0]
        # daily covariates from the last complete day at T
        d = us.last_complete_day(o)
        dd = np.maximum(d, 0)
        has = d >= 0
        for k, w in enumerate(weather_names()):
            cols[f"{w}_lastday"] = np.where(has, us.weather[s, dd, k], np.nan)
        cols["discount_lastday"] = np.where(has, self.discount[s, dd], np.nan)
        cols["discount_anomaly_lastday"] = np.where(has, self.discount_bad[s, dd], np.nan)
        cols["activity_lastday"] = np.where(has, us.activity[s, dd], np.nan)
        for name, arr in self.cov_mean.items():
            cols[name] = arr[s, d + 1]  # days <= d (d = -1 -> index 0 -> NaN)
        # outcomes (never features)
        cols["y"] = us.sales[s, u]
        cols["y_stockout"] = us.stockout[s, u]
        cols["y_day"] = tday.astype(np.int32)
        return pl.DataFrame(cols)


In [ ]:
%%writefile src/sctwin/backtest.py
"""Rolling-origin backtesting (CLAUDE.md Rule 1), in a configurable unit.

At origin T (end of unit T):

* **training examples**: feature time t with target window (t + L, t + L + H]; included only if
  the full window ends at or before T, i.e. ``t <= T - L - H``. Window: expanding (from
  ``min_history``) or sliding (the last ``window_length`` feature times). ``train_stride``
  thins feature times (a subset of past examples; anchored at the latest allowed t).
* **prediction**: features as of T, targets (T + L, T + L + H].

:func:`check_split` asserts both conditions on every fold at run time.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import polars as pl


@dataclass(frozen=True)
class BacktestSpec:
    unit: str
    lead: int  # L
    horizon: int  # H
    origins: tuple[int, ...]  # fold origins T (unit indices)
    window: str = "sliding"  # "sliding" | "expanding"
    window_length: int = 0  # W, number of feature times for a sliding window
    min_history: int = 0  # earliest feature time allowed
    train_stride: int = 1

    def __post_init__(self) -> None:
        if self.window not in ("sliding", "expanding"):
            raise ValueError("window must be 'sliding' or 'expanding'")
        if self.window == "sliding" and self.window_length < 1:
            raise ValueError("sliding window needs window_length >= 1")
        if self.lead < 0 or self.horizon < 1 or self.train_stride < 1:
            raise ValueError("need lead >= 0, horizon >= 1, train_stride >= 1")

    def targets(self, origin: int) -> range:
        return range(origin + self.lead + 1, origin + self.lead + self.horizon + 1)

    def last_train_time(self, origin: int) -> int:
        return origin - self.lead - self.horizon

    def train_times(self, origin: int) -> np.ndarray:
        hi = self.last_train_time(origin)
        lo = self.min_history
        if self.window == "sliding":
            lo = max(lo, hi - self.window_length + 1)
        if hi < lo:
            return np.array([], dtype=np.int64)
        return np.arange(hi, lo - 1, -self.train_stride)[::-1]


def make_origins(
    first_target: int,
    last_target: int,
    lead: int,
    horizon: int,
    step: int,
    n_folds: int | None = None,
) -> tuple[int, ...]:
    """Origins advancing by ``step`` whose full target window lies in [first_target,
    last_target]; optionally only the first ``n_folds``."""
    first = first_target - lead - 1
    last = last_target - lead - horizon
    if last < first:
        raise ValueError("evaluation window shorter than one horizon")
    origins = tuple(range(first, last + 1, step))
    return origins[:n_folds] if n_folds else origins


def check_split(train: pl.DataFrame, test: pl.DataFrame, origin: int, spec: BacktestSpec) -> None:
    """Raise if a training example's target window or features extend past the origin, or if
    test rows are not exactly the origin's forecast window."""
    if train.height:
        if int(train["origin_idx"].max()) > spec.last_train_time(origin):  # type: ignore[arg-type]
            raise AssertionError(f"origin {origin}: training example with t > T - L - H")
        if int(train["target_idx"].max()) > origin:  # type: ignore[arg-type]
            raise AssertionError(f"origin {origin}: training target after the origin")
        n_steps = train.group_by(["series_idx", "origin_idx"]).len()["len"]
        if int(n_steps.min()) != spec.horizon:  # type: ignore[arg-type]
            raise AssertionError(f"origin {origin}: training example with a partial window")
    if test.height:
        if set(test["origin_idx"].unique().to_list()) != {origin}:
            raise AssertionError(f"origin {origin}: test rows from another origin")
        tg = test["target_idx"]
        lo, hi = origin + spec.lead + 1, origin + spec.lead + spec.horizon
        if int(tg.min()) < lo or int(tg.max()) > hi:  # type: ignore[arg-type]
            raise AssertionError(f"origin {origin}: test target outside (T+L, T+L+H]")


## Files: Package: metrics, evaluation and Step 3 baselines

In [ ]:
%%writefile src/sctwin/metrics.py
"""Forecast evaluation metrics and paired comparison statistics.

Conventions: ``y`` has shape (n,). Quantile forecasts have shape (n, K) with matching ``taus``
(K,). Sample forecasts have shape (n, m). All scores are "lower is better" and returned as
per-observation arrays unless the name says otherwise, so they can be aggregated by subgroup
and passed to :func:`paired_bootstrap`.
"""

from __future__ import annotations

import math
from collections.abc import Sequence
from dataclasses import dataclass

import numpy as np

# --------------------------------------------------------------------------- point


def mae(y: np.ndarray, yhat: np.ndarray) -> float:
    return float(np.mean(np.abs(np.asarray(y) - np.asarray(yhat))))


def wape(y: np.ndarray, yhat: np.ndarray, eps: float = 1e-12) -> float | None:
    """sum|y - yhat| / sum|y|. Returns None (not inf/NaN) when the denominator is ~0."""
    y = np.asarray(y, dtype=np.float64)
    den = float(np.abs(y).sum())
    if den <= eps:
        return None
    return float(np.abs(y - np.asarray(yhat)).sum() / den)


# --------------------------------------------------------------------------- quantiles


def sort_quantiles(q: np.ndarray) -> tuple[np.ndarray, float]:
    """Fix quantile crossing by sorting along the last axis. Returns (sorted, frac_rows_fixed)."""
    q = np.asarray(q, dtype=np.float64)
    crossed = np.any(np.diff(q, axis=-1) < 0, axis=-1)
    return np.sort(q, axis=-1), float(crossed.mean()) if crossed.size else 0.0


def pinball(y: np.ndarray, q: np.ndarray, taus: Sequence[float]) -> np.ndarray:
    """Quantile (pinball) loss, shape (n, K)."""
    y = np.asarray(y, dtype=np.float64)[:, None]
    q = np.asarray(q, dtype=np.float64)
    t = np.asarray(taus, dtype=np.float64)[None, :]
    return np.where(y < q, (1 - t) * (q - y), t * (y - q))


def interval_score(y: np.ndarray, lo: np.ndarray, hi: np.ndarray, alpha: float) -> np.ndarray:
    """Gneiting & Raftery (2007) interval score for a central (1 - alpha) interval."""
    y, lo, hi = (np.asarray(a, dtype=np.float64) for a in (y, lo, hi))
    return (hi - lo) + (2 / alpha) * (lo - y) * (y < lo) + (2 / alpha) * (y - hi) * (y > hi)


def _central_pairs(taus: Sequence[float]) -> tuple[int, list[tuple[int, int, float]]]:
    t = np.round(np.asarray(taus, dtype=np.float64), 10)
    med = np.flatnonzero(np.isclose(t, 0.5))
    if med.size != 1:
        raise ValueError("WIS needs the median (tau = 0.5) among taus")
    pairs = []
    for i, ti in enumerate(t):
        if ti < 0.5:
            j = np.flatnonzero(np.isclose(t, 1 - ti))
            if j.size != 1:
                raise ValueError(f"tau {ti} has no symmetric partner {1 - ti}")
            pairs.append((i, int(j[0]), float(2 * ti)))
    if 2 * len(pairs) + 1 != t.size:
        raise ValueError("taus must be the median plus symmetric pairs")
    return int(med[0]), pairs


def wis(y: np.ndarray, q: np.ndarray, taus: Sequence[float]) -> np.ndarray:
    """Weighted interval score (Bracher et al. 2021), per observation.

    WIS = (0.5|y - m| + sum_k (alpha_k / 2) IS_alpha_k) / (K + 0.5).
    """
    q = np.asarray(q, dtype=np.float64)
    m, pairs = _central_pairs(taus)
    total = 0.5 * np.abs(np.asarray(y, dtype=np.float64) - q[:, m])
    for lo, hi, alpha in pairs:
        total = total + (alpha / 2) * interval_score(y, q[:, lo], q[:, hi], alpha)
    return total / (len(pairs) + 0.5)


def coverage(y: np.ndarray, lo: np.ndarray, hi: np.ndarray) -> float:
    y = np.asarray(y)
    return float(np.mean((y >= lo) & (y <= hi)))


def interval_coverage(
    y: np.ndarray, q: np.ndarray, taus: Sequence[float]
) -> dict[str, dict[str, float]]:
    """Empirical coverage and mean width of every central interval in the quantile set."""
    _, pairs = _central_pairs(taus)
    out = {}
    for lo, hi, alpha in pairs:
        level = round(1 - alpha, 6)
        out[str(level)] = {
            "nominal": level,
            "coverage": coverage(y, q[:, lo], q[:, hi]),
            "mean_width": float(np.mean(q[:, hi] - q[:, lo])),
        }
    return out


# --------------------------------------------------------------------------- samples


def crps_samples(y: np.ndarray, samples: np.ndarray) -> np.ndarray:
    """CRPS from an ensemble: E|X - y| - 0.5 E|X - X'| (empirical-distribution form)."""
    x = np.sort(np.asarray(samples, dtype=np.float64), axis=1)
    y = np.asarray(y, dtype=np.float64)
    m = x.shape[1]
    term1 = np.mean(np.abs(x - y[:, None]), axis=1)
    # E|X - X'| for the empirical distribution = (2 / m^2) * sum_i (2i - m - 1) x_(i)
    w = 2 * np.arange(1, m + 1) - m - 1
    term2 = (2.0 / m**2) * (x @ w)
    return term1 - 0.5 * term2


def crps_quantiles(y: np.ndarray, q: np.ndarray, taus: Sequence[float]) -> np.ndarray:
    """CRPS approximated as 2 x mean pinball loss over an (approximately uniform) tau grid.

    Exact in the limit of a dense uniform grid; a coarse grid gives a biased approximation,
    so compare models only on the same grid.
    """
    return 2.0 * pinball(y, q, taus).mean(axis=1)


def crps_normal(y: np.ndarray, mu: np.ndarray, sigma: np.ndarray) -> np.ndarray:
    """Closed-form CRPS of N(mu, sigma^2) (Gneiting et al. 2005). Used as a reference."""
    z = (np.asarray(y) - mu) / sigma
    cdf = 0.5 * (1 + np.vectorize(math.erf)(z / math.sqrt(2)))
    pdf = np.exp(-0.5 * z**2) / math.sqrt(2 * math.pi)
    return sigma * (z * (2 * cdf - 1) + 2 * pdf - 1 / math.sqrt(math.pi))


def randomized_pit(
    y: np.ndarray, samples: np.ndarray, rng: np.random.Generator | None = None
) -> np.ndarray:
    """Randomized PIT from samples; handles point masses (e.g. zero sales) correctly."""
    rng = rng or np.random.default_rng(0)
    s = np.asarray(samples)
    y = np.asarray(y)[:, None]
    below = (s < y).mean(axis=1)
    equal = (s == y).mean(axis=1)
    return below + rng.uniform(size=below.shape) * equal


# --------------------------------------------------------------------------- comparison


@dataclass
class BootstrapResult:
    mean_diff: float  # mean(loss_a - loss_b); negative => A better
    ci_low: float
    ci_high: float
    p_value: float  # two-sided, from the bootstrap distribution
    n_clusters: int
    n_obs: int


def paired_bootstrap(
    loss_a: np.ndarray,
    loss_b: np.ndarray,
    clusters: np.ndarray,
    n_boot: int = 2000,
    level: float = 0.95,
    seed: int = 0,
) -> BootstrapResult:
    """Cluster (block) bootstrap of the mean paired loss difference.

    Whole clusters (e.g. series or stores) are resampled with replacement, preserving temporal
    and within-cluster dependence. The statistic is the observation-weighted mean difference.
    """
    d = np.asarray(loss_a, dtype=np.float64) - np.asarray(loss_b, dtype=np.float64)
    _, cid = np.unique(np.asarray(clusters), return_inverse=True)
    n_cl = int(np.max(cid)) + 1
    sums = np.bincount(cid, weights=d, minlength=n_cl)
    cnts = np.bincount(cid, minlength=n_cl).astype(np.float64)
    rng = np.random.default_rng(seed)
    draws = rng.integers(0, n_cl, size=(n_boot, n_cl))
    boot = sums[draws].sum(axis=1) / cnts[draws].sum(axis=1)
    est = float(sums.sum() / cnts.sum())
    a = (1 - level) / 2
    lo, hi = np.quantile(boot, [a, 1 - a])
    # Two-sided p-value: how often the centred bootstrap distribution is as extreme as est.
    p = float(np.mean(np.abs(boot - est) >= abs(est)))
    return BootstrapResult(est, float(lo), float(hi), max(p, 1.0 / n_boot), n_cl, d.size)


def holm(p_values: Sequence[float]) -> list[float]:
    """Holm-Bonferroni adjusted p-values (step-down, monotone, capped at 1)."""
    p = np.asarray(p_values, dtype=np.float64)
    order = np.argsort(p)
    n = p.size
    adj = np.empty(n)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (n - rank) * p[i])
        adj[i] = min(running, 1.0)
    return adj.tolist()


In [ ]:
%%writefile src/sctwin/evaluation.py
"""Evaluation harness: metrics per model, per subgroup, and paired comparisons.

A predictions frame has one row per forecast target with:
  * ``y`` (observed outcome), ``cluster`` (bootstrap cluster id, e.g. series_idx)
  * per model ``m``: ``{m}__point`` and optionally ``{m}__q{tau}`` quantile columns
  * boolean subgroup columns named ``g_<name>``
"""

from __future__ import annotations

from typing import Any

import numpy as np
import polars as pl

from sctwin import metrics as M


def qcol(model: str, tau: float) -> str:
    return f"{model}__q{tau:g}"


def model_names(df: pl.DataFrame) -> list[str]:
    return sorted({c.split("__")[0] for c in df.columns if "__" in c})


def _quantiles(df: pl.DataFrame, model: str, taus: tuple[float, ...]) -> np.ndarray | None:
    cols = [qcol(model, t) for t in taus]
    if not all(c in df.columns for c in cols):
        return None
    return df.select(cols).to_numpy().astype(np.float64)


def score_frame(df: pl.DataFrame, taus: tuple[float, ...]) -> pl.DataFrame:
    """Add per-row loss columns ``{m}__ae`` and ``{m}__wis`` / ``{m}__crpsq``."""
    y = df["y"].to_numpy().astype(np.float64)
    new = {}
    for m in model_names(df):
        if f"{m}__point" in df.columns:
            new[f"{m}__ae"] = np.abs(y - df[f"{m}__point"].to_numpy())
        q = _quantiles(df, m, taus)
        if q is not None:
            new[f"{m}__wis"] = M.wis(y, q, taus)
            new[f"{m}__crpsq"] = M.crps_quantiles(y, q, taus)
    return df.with_columns(**{k: pl.Series(v) for k, v in new.items()})


def summarize(df: pl.DataFrame, taus: tuple[float, ...]) -> dict[str, Any]:
    """Metrics for every model on the rows of ``df`` (already scored)."""
    y = df["y"].to_numpy().astype(np.float64)
    out: dict[str, Any] = {"n": int(df.height), "n_clusters": int(df["cluster"].n_unique()),
                           "y_mean": float(y.mean()) if y.size else None, "models": {}}  # fmt: skip
    for m in model_names(df):
        r: dict[str, Any] = {}
        if f"{m}__point" in df.columns:
            f = df[f"{m}__point"].to_numpy()
            r["mae"] = M.mae(y, f)
            r["wape"] = M.wape(y, f)
            r["bias"] = float(np.mean(f - y))
        q = _quantiles(df, m, taus)
        if q is not None:
            r["wis"] = float(df[f"{m}__wis"].mean())  # type: ignore[arg-type]
            r["crps_q"] = float(df[f"{m}__crpsq"].mean())  # type: ignore[arg-type]
            r["intervals"] = M.interval_coverage(y, q, taus)
            if "point" not in r and 0.5 in taus:
                med = q[:, taus.index(0.5)]
                r["mae_median"] = M.mae(y, med)
                r["wape_median"] = M.wape(y, med)
        out["models"][m] = r
    return out


def evaluate(df: pl.DataFrame, taus: tuple[float, ...]) -> dict[str, Any]:
    scored = score_frame(df, taus)
    res: dict[str, Any] = {"overall": summarize(scored, taus), "folds": {}, "subgroups": {}}
    if "origin_idx" in scored.columns:  # Rule 1: per-fold and aggregated
        for (o,), sub in scored.group_by(["origin_idx"], maintain_order=True):
            res["folds"][str(o)] = summarize(sub, taus)
    for g in [c for c in scored.columns if c.startswith("g_")]:
        sub = scored.filter(pl.col(g))
        if sub.height:
            res["subgroups"][g[2:]] = summarize(sub, taus)
    return res


def compare_to_reference(
    df: pl.DataFrame,
    reference: str,
    loss: str,
    taus: tuple[float, ...],
    n_boot: int = 2000,
    seed: int = 0,
) -> dict[str, Any]:
    """Paired cluster bootstrap of ``loss`` for every model vs ``reference``, Holm-adjusted.

    Negative mean_diff means the model has lower loss than the reference.
    """
    scored = score_frame(df, taus)
    ref = f"{reference}__{loss}"
    others = [m for m in model_names(df) if m != reference and f"{m}__{loss}" in scored.columns]
    cl = scored["cluster"].to_numpy()
    res: dict[str, dict[str, Any]] = {}
    ref_mean = float(np.mean(scored[ref].to_numpy()))
    for m in others:
        r = M.paired_bootstrap(scored[f"{m}__{loss}"].to_numpy(), scored[ref].to_numpy(), cl,
                               n_boot=n_boot, seed=seed)  # fmt: skip
        res[m] = {
            "mean_diff": r.mean_diff,
            "ci": [r.ci_low, r.ci_high],
            "p": r.p_value,
            "relative": r.mean_diff / ref_mean,
        }
    adj = M.holm([float(v["p"]) for v in res.values()])
    for (_, v), a in zip(res.items(), adj, strict=True):
        v["p_holm"] = a
    return {"reference": reference, "loss": loss, "comparisons": res}


In [ ]:
%%writefile src/sctwin/baseline.py
"""Step 3 baseline forecasters for observed sales.

All models predict day t from the as-of features of :mod:`sctwin.features` (origin = end of
day t-1). They forecast *observed* sales; nothing here estimates latent demand.

* ``snaive``   - seasonal naive point forecast (same weekday last week / same hour last week)
* ``clim``     - empirical quantiles of the last n seasonal-position values observed by T
* ``glm``      - pooled Tweedie GLM (log link); predictive quantiles by simulating the
                 compound Poisson-Gamma distribution with a Pearson dispersion estimate
* ``lgbm``     - LightGBM point forecast (Tweedie objective)
* ``lgbm_q``   - LightGBM quantile regression, one model per tau, crossings fixed by sorting
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any

import numpy as np
import polars as pl

TAUS = (0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95)


# --------------------------------------------------------------------------- GLM


@dataclass
class GLMDesign:
    """Feature transformation for the GLM, fitted on training rows only."""

    numeric: list[str]
    categorical: list[str]
    log_cols: list[str]
    no_missing_flag: list[str] = field(default_factory=list)
    means: dict[str, float] = field(default_factory=dict)
    stds: dict[str, float] = field(default_factory=dict)
    levels: dict[str, list[Any]] = field(default_factory=dict)

    def fit(self, df: pl.DataFrame) -> GLMDesign:
        for c in self.numeric:
            v = self._num(df, c)
            v = v[~np.isnan(v)]
            # an all-missing column becomes a constant 0 plus its missing indicator
            self.means[c] = float(v.mean()) if v.size else 0.0
            self.stds[c] = (float(v.std()) if v.size else 0.0) or 1.0
        for c in self.categorical:
            self.levels[c] = sorted(df[c].unique().to_list())[1:]  # drop first level
        return self

    def _num(self, df: pl.DataFrame, c: str) -> np.ndarray:
        v = df[c].cast(pl.Float64).fill_nan(None).to_numpy().astype(np.float64)
        return np.log1p(v) if c in self.log_cols else v

    def transform(self, df: pl.DataFrame) -> np.ndarray:
        cols = []
        for c in self.numeric:
            v = self._num(df, c)
            miss = np.isnan(v)
            cols.append(np.where(miss, 0.0, (v - self.means[c]) / self.stds[c]))
            if c not in self.no_missing_flag:
                cols.append(miss.astype(np.float64))
        for c in self.categorical:
            v = df[c].to_numpy()
            cols.extend((v == lev).astype(np.float64) for lev in self.levels[c])
        return np.column_stack(cols)


def fit_tweedie_glm(
    x: np.ndarray, y: np.ndarray, power: float = 1.5, alpha: float = 1e-4, max_iter: int = 300
) -> tuple[Any, float]:
    """Fit a log-link Tweedie GLM; return (model, Pearson dispersion phi)."""
    from sklearn.linear_model import TweedieRegressor

    m = TweedieRegressor(power=power, link="log", alpha=alpha, max_iter=max_iter)
    m.fit(x, y)
    mu = m.predict(x)
    dof = max(len(y) - x.shape[1] - 1, 1)
    phi = float(np.sum((y - mu) ** 2 / np.maximum(mu, 1e-9) ** power) / dof)
    return m, phi


def tweedie_sample(
    mu: np.ndarray, phi: float, power: float, n: int, rng: np.random.Generator
) -> np.ndarray:
    """Samples (len(mu), n) from compound Poisson-Gamma Tweedie(mu, phi, 1 < power < 2)."""
    mu = np.asarray(mu, dtype=np.float64)[:, None]
    lam = mu ** (2 - power) / (phi * (2 - power))
    shape = (2 - power) / (power - 1)
    scale = phi * (power - 1) * mu ** (power - 1)
    counts = rng.poisson(np.broadcast_to(lam, (mu.shape[0], n)))
    out = np.zeros(counts.shape)
    pos = counts > 0
    out[pos] = rng.gamma(counts[pos] * shape, np.broadcast_to(scale, counts.shape)[pos])
    return out


def tweedie_quantiles(
    mu: np.ndarray,
    phi: float,
    power: float,
    taus: tuple[float, ...] = TAUS,
    n: int = 400,
    seed: int = 0,
    batch: int = 50_000,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    out = np.empty((len(mu), len(taus)))
    for i in range(0, len(mu), batch):
        s = tweedie_sample(mu[i : i + batch], phi, power, n, rng)
        out[i : i + batch] = np.quantile(s, taus, axis=1).T
    return out


# --------------------------------------------------------------------------- LightGBM

LGBM_DEFAULTS: dict[str, Any] = {
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 200,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 1.0,
    "verbose": -1,
    "deterministic": True,
    "force_row_wise": True,
}


def fit_lgbm(
    x: pl.DataFrame,
    y: np.ndarray,
    categorical: list[str],
    objective: str,
    num_rounds: int,
    seed: int,
    num_threads: int,
    alpha: float | None = None,
    params: dict[str, Any] | None = None,
) -> Any:
    import lightgbm as lgb

    prm = {**LGBM_DEFAULTS, **(params or {}), "objective": objective, "seed": seed,
           "num_threads": num_threads}  # fmt: skip
    if alpha is not None:
        prm["alpha"] = alpha
    ds = lgb.Dataset(x.to_pandas(), label=y, categorical_feature=categorical, free_raw_data=True)
    return lgb.train(prm, ds, num_boost_round=num_rounds)


def predict_lgbm(model: Any, x: pl.DataFrame) -> np.ndarray:
    return np.asarray(model.predict(x.to_pandas(), num_threads=model.params.get("num_threads", 0)))


In [ ]:
%%writefile src/sctwin/baseline_runner.py
"""Rolling-origin backtest of the Step 3 baselines, in a configurable unit.

For every fold origin T: training examples with t <= T - L - H (full target windows observed by
T), verified by ``check_split``; every fitted statistic (GLM scaling/imputation, model
parameters) is fitted on that fold's training rows only; predictions for (T + L, T + L + H].
"""

from __future__ import annotations

import time
import warnings
from typing import Any

import numpy as np
import polars as pl

from sctwin.backtest import BacktestSpec, check_split
from sctwin.baseline import GLMDesign, fit_lgbm, fit_tweedie_glm, predict_lgbm, tweedie_quantiles
from sctwin.evaluation import qcol
from sctwin.features import FeatureBuilder
from sctwin.metrics import sort_quantiles
from sctwin.panel import CLS_A, CLS_NIGHT, CLS_P_FIRST, CLS_P_PRE, CLS_U

LGBM_CATEGORICAL = ["management_group_id", "first_category_id", "city_id"]


def _glm_design(fb: FeatureBuilder) -> GLMDesign:
    cat = ["target_dow", "management_group_id"]
    if fb.us.unit == "hour":
        cat.append("target_hour")
    num = [f for f in fb.feature_names if f not in cat]
    log_cols = [c for c in num if c.startswith(("sale_", "instock_mean"))]
    return GLMDesign(numeric=num, categorical=cat, log_cols=log_cols,
                     no_missing_flag=["steps_ahead", "target_holiday"])  # fmt: skip


def clim_quantiles(fb: FeatureBuilder, test: pl.DataFrame, taus: tuple[float, ...]) -> np.ndarray:
    """Empirical quantiles of the last n_seasons values at the target's seasonal position that
    are observed at the origin."""
    vals = fb.seasonal_values(
        test["series_idx"].to_numpy(), test["origin_idx"].to_numpy().astype(np.int64),
        test["target_idx"].to_numpy().astype(np.int64), fb.cfg.season, fb.cfg.n_seasons,
    )  # fmt: skip
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanquantile(vals, taus, axis=1).T


def run_backtest(
    fb: FeatureBuilder,
    spec: BacktestSpec,
    train_series: np.ndarray,
    eval_series: np.ndarray,
    cfg: dict[str, Any],
    log: Any = print,
) -> tuple[pl.DataFrame, list[dict[str, Any]]]:
    """Returns (predictions, per-fold log)."""
    if spec.unit != fb.us.unit:
        raise ValueError(f"spec unit {spec.unit} != feature unit {fb.us.unit}")
    taus = tuple(cfg["taus"])
    features = list(fb.feature_names)
    lgbm_x = features + LGBM_CATEGORICAL
    seed = int(cfg["seed"])
    lg, gl = cfg["lgbm"], cfg["glm"]
    out, fitlog = [], []

    for o in spec.origins:
        t0 = time.perf_counter()
        train = fb.rows(spec.train_times(o), spec.lead, spec.horizon, series_idx=train_series)
        test = fb.rows([o], spec.lead, spec.horizon, series_idx=eval_series)
        check_split(train, test, o, spec)
        y_tr = train["y"].to_numpy().astype(np.float64)
        pred: dict[str, np.ndarray] = {}

        pred["snaive__point"] = test["sale_seasonal_last"].to_numpy()
        cq = clim_quantiles(fb, test, taus)
        for k, tau in enumerate(taus):
            pred[qcol("clim", tau)] = cq[:, k]

        design = _glm_design(fb).fit(train)  # scaling/imputation fitted on this fold's train
        glm, phi = fit_tweedie_glm(design.transform(train), y_tr, power=gl["power"],
                                   alpha=gl["alpha"])  # fmt: skip
        mu = glm.predict(design.transform(test))
        pred["glm__point"] = mu
        gq = tweedie_quantiles(mu, phi, gl["power"], taus, n=gl["n_samples"], seed=seed + o)
        for k, tau in enumerate(taus):
            pred[qcol("glm", tau)] = gq[:, k]

        xtr, xte = train.select(lgbm_x), test.select(lgbm_x)
        common = dict(categorical=LGBM_CATEGORICAL, num_rounds=lg["num_rounds"], seed=seed,
                      num_threads=lg["num_threads"], params=lg.get("params"))  # fmt: skip
        pred["lgbm__point"] = predict_lgbm(fit_lgbm(xtr, y_tr, objective="tweedie", **common), xte)
        qs = np.column_stack(
            [
                predict_lgbm(fit_lgbm(xtr, y_tr, objective="quantile", alpha=t, **common), xte)
                for t in taus
            ]  # fmt: skip
        )
        qs, crossed = sort_quantiles(np.maximum(qs, 0.0))
        for k, tau in enumerate(taus):
            pred[qcol("lgbm_q", tau)] = qs[:, k]

        keep = ["series_idx", "origin_idx", "target_idx", "step", "steps_ahead", "target_dt",
                "y_day", "y_stockout", "management_group_id", "store_id", "product_id"]  # fmt: skip
        block = test.select(keep).with_columns(
            y=pl.Series(test["y"].to_numpy().astype(np.float64)),
            cluster=pl.col("series_idx"),
            **{k: pl.Series(v.astype(np.float64)) for k, v in pred.items()},
        )
        out.append(block)
        t_range = (
            [int(train["origin_idx"].min()), int(train["origin_idx"].max())]  # type: ignore[arg-type]
            if train.height
            else None
        )
        rec = {"origin": int(o), "train_rows": train.height, "test_rows": test.height,
               "train_t_range": t_range, "glm_phi": phi, "lgbm_q_crossing_frac": crossed,
               "seconds": round(time.perf_counter() - t0, 1)}  # fmt: skip
        fitlog.append(rec)
        log(f"[{spec.unit}] origin {o}: {rec}")
    return pl.concat(out), fitlog


def add_groups(pred: pl.DataFrame, fb: FeatureBuilder, low_volume_series: set[int]) -> pl.DataFrame:
    """Boolean subgroup columns (``g_*``) for evaluation. Uses target-time information, which
    is allowed for *grouping outcomes* only, never as features."""
    us = fb.us
    s = pred["series_idx"].to_numpy()
    act = us.activity[s, pred["y_day"].to_numpy()]
    g: dict[str, Any] = {
        "g_promo": pl.Series(act == 1),
        "g_no_promo": pl.Series(act == 0),
        "g_low_volume": pl.col("series_idx").is_in(sorted(low_volume_series)),
    }
    if us.unit == "day":
        g["g_available"] = pl.col("y_stockout") == 0
        g["g_censored"] = pl.col("y_stockout") > 0
    else:
        assert us.eval_class is not None
        cls = us.eval_class[s, pred["target_idx"].to_numpy()]
        for name, code in (("A", CLS_A), ("P_pre", CLS_P_PRE), ("P_first", CLS_P_FIRST),
                           ("U", CLS_U), ("night", CLS_NIGHT)):  # fmt: skip
            g[f"g_class_{name}"] = pl.Series(cls == code)
    for k in sorted(pred["steps_ahead"].unique().to_list()):
        g[f"g_steps_ahead_{int(k)}"] = pl.col("steps_ahead") == k
    return pred.with_columns(**g)


## Files: Package: Step 4 censoring benchmark and Step 5 Bayesian model

In [ ]:
%%writefile src/sctwin/censoring.py
"""Step 4: censoring benchmark (evaluation track 2, latent-demand recovery).

Real FreshRetailNet sales are censored by stockouts and the lost demand is never observed. To
test methods that reconstruct demand we need cases with a known answer:

* **Semi-synthetic injection.** Take real days that were fully in stock in hours 6-21, treat
  their sales as the demand (a *proxy*: in-stock sales are the closest observable stand-in for
  demand, not demand itself) and inject a stockout whose timing is driven by that demand.
  Each day gets an opening stock K; the stockout starts in the hour where cumulative sales
  reach K, as in the real data, where stockouts are demand-driven (docs/data_card.md,
  Finding 3).
* **Fully synthetic panel.** Latent demand from a known generative model, censored with the
  same mechanism. Used for parameter recovery in Step 5.

Injection mechanism, for one day, over hours 6-21 (the window of ``DAY_HOURS``):

1. Opening stock ``K = coverage * E * exp(sd * eps - sd^2 / 2)``, where ``E`` is the expected
   window demand (for real days: the trailing in-stock mean over the previous ``window`` days,
   i.e. data available before the day; the methods being evaluated never see K).
2. The depletion hour ``h*`` is the first hour where cumulative demand reaches ``K``. It stays
   flagged in stock and its observed sales are ``K - (demand before h*)``: a lower bound.
3. Hours after ``h*`` are flagged out of stock. A fraction ``leak_first`` (first flagged hour)
   and ``leak_later`` (later hours) of demand is still observed, matching the residual sales in
   flagged hours seen in the real data (partial availability or flag noise).
4. With probability ``p_restock`` stock returns after a geometric number of hours (mean
   ``restock_mean_hours``); from then on observed sales equal demand.

Hours outside 6-21 are never censored. Observed sales never exceed demand.

The reconstruction task is defined at the origin T = end of the censored day: methods see that
day's observed sales and flags plus summaries of days *before* it (``trailing_hour_means``),
never the demand, K, the depletion hour or any later day.
"""

from __future__ import annotations

import dataclasses
from collections.abc import Callable
from dataclasses import dataclass, field

import numpy as np
import polars as pl

from sctwin.panel import CLS_A, CLS_P_FIRST, CLS_P_PRE, CLS_U, DAY_HOURS, availability_classes

LO, HI = DAY_HOURS
NW = HI - LO  # hours in the censoring window (16)


@dataclass(frozen=True)
class CensorConfig:
    coverage: float = 1.0  # opening stock as a multiple of expected window demand
    coverage_sd: float = 0.5  # log-normal spread of the stock level
    leak_first: float = 0.18  # share of demand still sold in the first flagged hour
    leak_later: float = 0.08  # ... and in later flagged hours
    p_restock: float = 0.10  # probability that stock returns within the window
    restock_mean_hours: float = 3.0  # mean flagged hours before a restock

    def __post_init__(self) -> None:
        if not (0 <= self.leak_first <= 1 and 0 <= self.leak_later <= 1):
            raise ValueError("leak fractions must lie in [0, 1]")
        if not 0 <= self.p_restock <= 1:
            raise ValueError("p_restock must lie in [0, 1]")
        if self.coverage <= 0 or self.coverage_sd < 0 or self.restock_mean_hours < 1:
            raise ValueError("coverage > 0, coverage_sd >= 0, restock_mean_hours >= 1")


@dataclass
class Injection:
    observed: np.ndarray  # (N, 24) observed sales after censoring
    flag: np.ndarray  # (N, 24) int8, 1 = flagged out of stock
    depletion: np.ndarray  # (N,) hour of day of the depletion hour, -1 = no stockout
    restock: np.ndarray  # (N,) first in-stock hour after the stockout, -1 = none in window
    censored: np.ndarray  # (N, 24) bool: depletion hour and flagged hours (observed <= demand)

    @property
    def stocked_out(self) -> np.ndarray:
        return self.depletion >= 0


def stock_levels(expected: np.ndarray, rng: np.random.Generator, cfg: CensorConfig) -> np.ndarray:
    eps = rng.standard_normal(np.shape(expected))
    s = cfg.coverage_sd
    return cfg.coverage * np.asarray(expected, dtype=np.float64) * np.exp(s * eps - s * s / 2)


def inject_stockouts(
    demand: np.ndarray, stock: np.ndarray, rng: np.random.Generator, cfg: CensorConfig
) -> Injection:
    """Censor (N, 24) demand with opening stocks (N,). Random draws are made for every row, so
    the same seed gives common random numbers across configurations."""
    demand = np.asarray(demand, dtype=np.float64)
    stock = np.asarray(stock, dtype=np.float64)
    if demand.ndim != 2 or demand.shape[1] != 24 or stock.shape != demand.shape[:1]:
        raise ValueError("demand must be (N, 24) and stock (N,)")
    if (demand < 0).any() or not (stock > 0).all():
        raise ValueError("demand must be >= 0 and stock > 0")
    n = demand.shape[0]
    restock_draw = rng.random(n) < cfg.p_restock
    duration = rng.geometric(1.0 / cfg.restock_mean_hours, n)  # flagged hours, >= 1

    w = demand[:, LO:HI]
    cum = np.cumsum(w, axis=1)
    hit = cum >= stock[:, None]
    has = hit.any(axis=1)
    j = np.where(has, hit.argmax(axis=1), -1)  # depletion index within the window
    r = np.where(has & restock_draw, j + 1 + duration, NW)  # window index where stock returns
    r_end = np.minimum(r, NW)
    k = np.arange(NW)[None, :]
    flag_w = has[:, None] & (k > j[:, None]) & (k < r_end[:, None])
    first = flag_w & (k == j[:, None] + 1)

    obs_w = w.copy()
    rows = np.flatnonzero(has)
    jr = j[rows]
    obs_w[rows, jr] = stock[rows] - (cum[rows, jr] - w[rows, jr])
    obs_w[first] *= cfg.leak_first
    obs_w[flag_w & ~first] *= cfg.leak_later
    depl = np.zeros_like(flag_w)
    depl[rows, jr] = True

    observed = demand.copy()
    observed[:, LO:HI] = obs_w
    flag = np.zeros(demand.shape, dtype=np.int8)
    flag[:, LO:HI] = flag_w
    censored = np.zeros(demand.shape, dtype=bool)
    censored[:, LO:HI] = depl | flag_w
    return Injection(
        observed=observed,
        flag=flag,
        depletion=np.where(has, j + LO, -1),
        restock=np.where(has & (r < NW), r + LO, -1),
        censored=censored,
    )


# --------------------------------------------------------------------------- point-in-time history


def _trailing(x: np.ndarray, days: np.ndarray, window: int) -> np.ndarray:
    """Sum of x[s, d - window : d] (days strictly before d) for rows (s, d); x is (S, D, ...)."""
    c = np.concatenate([np.zeros_like(x[:, :1]), np.cumsum(x, axis=1)], axis=1)
    return c[:, days] - c[:, np.maximum(days - window, 0)]


def trailing_hour_means(
    sales: np.ndarray, stock: np.ndarray, series: np.ndarray, days: np.ndarray, window: int
) -> np.ndarray:
    """(N, 24) mean in-stock sales per hour of day over days [d - window, d - 1] for each row
    (series[i], days[i]). NaN where the hour was never in stock. Uses no data from day d or later.
    """
    if (np.asarray(days) < 1).any():
        raise ValueError("every row needs at least one day of history")
    inst = (stock == 0).astype(np.float64)
    out = np.full((len(series), 24), np.nan)
    for s in np.unique(series):  # per-series cumulative sums keep memory O(D x 24)
        i = np.flatnonzero(series == s)
        num = _trailing((sales[s] * inst[s])[None], days[i], window)[0]
        den = _trailing(inst[s][None], days[i], window)[0]
        with np.errstate(invalid="ignore", divide="ignore"):
            out[i] = np.where(den > 0, num / den, np.nan)
    return out


def expected_window_demand(hour_means: np.ndarray) -> np.ndarray:
    """Expected hours 6-21 demand from trailing in-stock hour means (NaN if any hour unknown)."""
    return hour_means[:, LO:HI].sum(axis=1)


# --------------------------------------------------------------------------- benchmark


@dataclass
class Benchmark:
    series: np.ndarray  # (N,) panel series index
    day: np.ndarray  # (N,) day index
    demand: np.ndarray  # (N, 24) real in-stock sales of that day (demand proxy)
    stock: np.ndarray  # (N,) opening stock K
    hour_means: np.ndarray  # (N, 24) trailing in-stock hour means (inputs for the methods)
    injection: Injection
    cfg: CensorConfig = field(default_factory=CensorConfig)


def candidate_days(
    sales: np.ndarray,
    stock: np.ndarray,
    series: np.ndarray,
    days: np.ndarray,
    window: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Rows (series, day) whose hours 6-21 were fully in stock, with a finite positive expected
    window demand from the previous ``window`` days. Returns (series, day, hour_means)."""
    ss, dd = np.meshgrid(np.asarray(series), np.asarray(days), indexing="ij")
    ss, dd = ss.ravel(), dd.ravel()
    full = (stock[ss, dd, LO:HI] == 0).all(axis=1) & (dd >= window)
    ss, dd = ss[full], dd[full]
    hm = trailing_hour_means(sales, stock, ss, dd, window)
    e = expected_window_demand(hm)
    ok = np.isfinite(e) & (e > 0)
    return ss[ok], dd[ok], hm[ok]


def build_benchmark(
    sales: np.ndarray,
    rows: tuple[np.ndarray, np.ndarray, np.ndarray],
    cfg: CensorConfig,
    seed: int,
) -> Benchmark:
    ss, dd, hm = rows
    rng = np.random.default_rng(seed)
    demand = sales[ss, dd].astype(np.float64)
    k = stock_levels(expected_window_demand(hm), rng, cfg)
    return Benchmark(ss, dd, demand, k, hm, inject_stockouts(demand, k, rng, cfg), cfg)


# --------------------------------------------------------------------------- realism diagnostics

QUANTILES = (25, 50, 75)


def stockout_stats(flag: np.ndarray) -> dict[str, float]:
    """Stockout prevalence and timing on (N, 24) flags, over days in stock at hour 6."""
    f = flag[:, LO:HI].astype(bool)
    f = f[~f[:, 0]]
    so = f.any(axis=1)
    first = f[so].argmax(axis=1)
    k = np.arange(NW)[None, :]
    after = k >= first[:, None]
    back = (after & ~f[so]).any(axis=1)
    dur = np.argmax(after & ~f[so], axis=1) - first
    out = {"n_days": float(f.shape[0]), "stockout_rate": float(so.mean()) if f.size else np.nan,
           "restock_rate": float(back.mean()) if so.any() else np.nan,
           "restock_mean_hours": float(dur[back].mean()) if back.any() else np.nan}  # fmt: skip
    for q in QUANTILES:
        out[f"first_hour_q{q}"] = float(np.percentile(first + LO, q)) if so.any() else np.nan
    return out


def position_ratios(
    observed: np.ndarray, flag: np.ndarray, baseline: np.ndarray, offsets: tuple[int, ...]
) -> dict[str, float]:
    """Sales at hours relative to the first flagged hour (offset 0), as a ratio of summed
    observed sales to summed baseline (the series' usual in-stock sales at that hour)."""
    f = flag[:, LO:HI].astype(bool)
    keep = f.any(axis=1) & ~f[:, 0]
    f = f[keep]
    j = f.argmax(axis=1)  # first flagged index in the window
    k = np.arange(NW)[None, :]
    back = (k >= j[:, None]) & ~f
    run = np.where(back.any(axis=1), back.argmax(axis=1), NW) - j  # length of the first run
    first = j + LO
    obs, base = observed[keep], baseline[keep]
    out = {}
    for o in offsets:
        h = first + o
        ok = (h >= LO) & (h < HI) & (o < run)  # offsets >= 0 must still be in the first run
        r = np.flatnonzero(ok)
        num, den = obs[r, h[r]].sum(), np.nansum(base[r, h[r]])
        out[f"{o:+d}"] = float(num / den) if den > 0 else np.nan
    return out


def calibrate(
    demand: np.ndarray,
    expected: np.ndarray,
    target: dict[str, float],
    base: CensorConfig,
    coverages: np.ndarray,
    sds: np.ndarray,
    seed: int,
) -> tuple[CensorConfig, list[dict[str, float]]]:
    """Grid search (coverage, coverage_sd) so that injected stockouts on fully in-stock days
    match the real stockout rate and first-stockout-hour quartiles. Common random numbers."""
    grid = []
    for c in coverages:
        for sd in sds:
            cfg = CensorConfig(**{**base.__dict__, "coverage": float(c), "coverage_sd": float(sd)})
            rng = np.random.default_rng(seed)
            inj = inject_stockouts(demand, stock_levels(expected, rng, cfg), rng, cfg)
            st = stockout_stats(inj.flag)
            loss = ((st["stockout_rate"] - target["stockout_rate"]) / 0.02) ** 2 + sum(
                ((st[f"first_hour_q{q}"] - target[f"first_hour_q{q}"]) / 1.0) ** 2
                for q in QUANTILES
            )
            grid.append({"coverage": float(c), "coverage_sd": float(sd), "loss": float(loss), **st})
    best = min(grid, key=lambda g: g["loss"] if np.isfinite(g["loss"]) else np.inf)
    cfg = CensorConfig(**{**base.__dict__, "coverage": best["coverage"],
                          "coverage_sd": best["coverage_sd"]})  # fmt: skip
    return cfg, grid


def calibrate_restock(
    demand: np.ndarray,
    expected: np.ndarray,
    target: dict[str, float],
    cfg: CensorConfig,
    p_grid: np.ndarray,
    mean_grid: np.ndarray,
    seed: int,
) -> CensorConfig:
    """Grid search (p_restock, restock_mean_hours) so that the *measured* restock rate and mean
    restock duration match the real data. Both are truncated by the end of the window in the
    same way, which is why the raw real values cannot be plugged in directly."""
    best, best_loss = cfg, np.inf
    for p in p_grid:
        for m in mean_grid:
            c = dataclasses.replace(cfg, p_restock=float(p), restock_mean_hours=float(m))
            rng = np.random.default_rng(seed)
            st = stockout_stats(
                inject_stockouts(demand, stock_levels(expected, rng, c), rng, c).flag
            )
            loss = ((st["restock_rate"] - target["restock_rate"]) / 0.01) ** 2 + (
                (st["restock_mean_hours"] - target["restock_mean_hours"]) / 0.25
            ) ** 2
            if np.isfinite(loss) and loss < best_loss:
                best, best_loss = c, loss
    return best


def calibrate_leak(
    demand: np.ndarray,
    expected: np.ndarray,
    baseline: np.ndarray,
    target: dict[str, float],
    cfg: CensorConfig,
    seed: int,
) -> CensorConfig:
    """Set leak_first / leak_later so that sales in the first and second flagged hour, relative
    to the usual in-stock sales at that hour (``position_ratios`` offsets 0 and +1), match the
    real data. Flags do not depend on the leak, so the ratio is linear in it: one exact step."""
    rng = np.random.default_rng(seed)
    inj = inject_stockouts(demand, stock_levels(expected, rng, cfg), rng, cfg)
    got = position_ratios(inj.observed, inj.flag, baseline, (0, 1))
    new = {}
    for key, name in (("+0", "leak_first"), ("+1", "leak_later")):
        cur = getattr(cfg, name)
        ok = cur > 0 and np.isfinite(got[key]) and got[key] > 0 and np.isfinite(target[key])
        new[name] = float(min(1.0, cur * target[key] / got[key])) if ok else cur
    return dataclasses.replace(cfg, **new)


# ------------------------------------------------------------------- reconstruction baselines

Imputer = Callable[[np.ndarray, np.ndarray, np.ndarray], np.ndarray]


def impute_observed(observed: np.ndarray, flag: np.ndarray, hour_means: np.ndarray) -> np.ndarray:
    """Ignore censoring: demand = observed sales."""
    return observed.copy()


def impute_hour_mean(observed: np.ndarray, flag: np.ndarray, hour_means: np.ndarray) -> np.ndarray:
    """Flagged hours get the series' trailing in-stock mean for that hour (never below observed).
    Ignores that stockouts happen on high-demand days."""
    est = observed.copy()
    f = flag.astype(bool)
    f[:, :LO] = f[:, HI:] = False
    fill = np.maximum(observed, np.nan_to_num(hour_means))
    est[f] = fill[f]
    return est


def _impute_profile(
    observed: np.ndarray, flag: np.ndarray, hour_means: np.ndarray, pre: bool
) -> np.ndarray:
    m = np.nan_to_num(hour_means[:, LO:HI])
    tot = m.sum(axis=1, keepdims=True)
    share = np.divide(m, tot, out=np.zeros_like(m), where=tot > 0)
    f = flag[:, LO:HI].astype(bool)
    cens = f.copy()
    if pre:  # the in-stock hour before a flagged hour is the likely depletion hour
        nxt = np.zeros_like(f)
        nxt[:, :-1] = f[:, 1:]
        cens |= ~f & nxt
    obs = observed[:, LO:HI]
    den = (share * ~cens).sum(axis=1)
    num = (obs * ~cens).sum(axis=1)
    level = np.where(den > 0, num / np.where(den > 0, den, 1.0), tot[:, 0])
    fill = np.maximum(obs, level[:, None] * share)
    est = observed.copy()
    w = est[:, LO:HI]
    w[cens] = fill[cens]
    return est


def impute_profile(observed: np.ndarray, flag: np.ndarray, hour_means: np.ndarray) -> np.ndarray:
    """Scale the series' hourly profile to the day's in-stock hours; fill flagged hours."""
    return _impute_profile(observed, flag, hour_means, pre=False)


def impute_profile_pre(
    observed: np.ndarray, flag: np.ndarray, hour_means: np.ndarray
) -> np.ndarray:
    """As ``profile``, but the hour before each stockout is also treated as censored."""
    return _impute_profile(observed, flag, hour_means, pre=True)


IMPUTERS: dict[str, Imputer] = {
    "observed": impute_observed,
    "hour_mean": impute_hour_mean,
    "profile": impute_profile,
    "profile_pre": impute_profile_pre,
}


# --------------------------------------------------------------------------- scoring frames


def recovery_frames(
    bm: Benchmark, estimates: dict[str, np.ndarray]
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Evaluation frames (sctwin.evaluation format) for reconstructed demand.

    hourly: one row per censored hour (depletion hour and flagged hours), y = demand proxy.
    daily: one row per stocked-out day, y = hours 6-21 demand total.
    Subgroups: availability class of the hour, depletion time (early < 12 <= mid < 17 <= late),
    restock within the day, and censoring a method cannot see (depletion in the last hour).
    """
    inj = bm.injection
    cls = availability_classes(inj.flag)
    r, h = np.nonzero(inj.censored)
    dep = inj.depletion
    groups = {
        "g_early": dep < 12,
        "g_mid": (dep >= 12) & (dep < 17),
        "g_late": dep >= 17,
        "g_restock": inj.restock >= 0,
    }
    hourly = pl.DataFrame(
        {
            "row": r,
            "series_idx": bm.series[r],
            "day_idx": bm.day[r],
            "hour": h,
            "cluster": bm.series[r],
            "y": bm.demand[r, h],
            "observed": inj.observed[r, h],
            "g_P_pre": cls[r, h] == CLS_P_PRE,
            "g_P_first": cls[r, h] == CLS_P_FIRST,
            "g_U": cls[r, h] == CLS_U,
            "g_undetected": cls[r, h] == CLS_A,
            **{g: v[r] for g, v in groups.items()},
            **{f"{m}__point": e[r, h] for m, e in estimates.items()},
        }
    )
    d = np.flatnonzero(inj.stocked_out)
    daily = pl.DataFrame(
        {
            "row": d,
            "series_idx": bm.series[d],
            "day_idx": bm.day[d],
            "cluster": bm.series[d],
            "y": bm.demand[d, LO:HI].sum(axis=1),
            "observed": inj.observed[d, LO:HI].sum(axis=1),
            **{g: v[d] for g, v in groups.items()},
            **{f"{m}__point": e[d, LO:HI].sum(axis=1) for m, e in estimates.items()},
        }
    )
    return hourly, daily


# --------------------------------------------------------------------------- fully synthetic panel


@dataclass(frozen=True)
class SyntheticParams:
    level_mean: float = -1.0  # mean log hourly demand at a profile value of 1
    level_sd: float = 0.7  # between-series spread of the log level
    dow: tuple[float, ...] = (0.0, -0.05, -0.05, 0.0, 0.1, 0.2, 0.15)  # log weekday effects
    rho: float = 0.6  # AR(1) coefficient of the daily log state
    sigma_z: float = 0.25  # innovation sd of the daily log state
    phi: float = 1.0  # Tweedie dispersion
    power: float = 1.5  # Tweedie power


def hour_profile() -> np.ndarray:
    """Default hourly demand profile: small at night, two daytime peaks. Mean 1 over 6-21."""
    h = np.arange(24)
    day = (
        0.6
        + 0.5 * np.exp(-0.5 * ((h - 11) / 2.0) ** 2)
        + 0.8 * np.exp(-0.5 * ((h - 18) / 2.0) ** 2)
    )
    p = np.where((h >= LO) & (h < HI), day, 0.05)
    return p / p[LO:HI].mean()


@dataclass
class SyntheticPanel:
    params: SyntheticParams
    profile: np.ndarray  # (24,)
    log_level: np.ndarray  # (S,)
    z: np.ndarray  # (S, D) daily latent state
    mu: np.ndarray  # (S, D, 24) mean demand
    demand: np.ndarray  # (S, D, 24) latent demand (known truth)


def simulate_latent(
    n_series: int, n_days: int, params: SyntheticParams, seed: int
) -> SyntheticPanel:
    """Latent demand D ~ Tweedie(mu, phi, power) with
    log mu[s, d, h] = log_level[s] + dow[d % 7] + z[s, d] + log profile[h],
    z[s, d] = rho z[s, d-1] + N(0, sigma_z^2), z stationary at d = 0."""
    from sctwin.baseline import tweedie_sample

    rng = np.random.default_rng(seed)
    a = rng.normal(params.level_mean, params.level_sd, n_series)
    z = np.empty((n_series, n_days))
    z[:, 0] = rng.normal(0, params.sigma_z / np.sqrt(1 - params.rho**2), n_series)
    for d in range(1, n_days):
        z[:, d] = params.rho * z[:, d - 1] + rng.normal(0, params.sigma_z, n_series)
    prof = hour_profile()
    dow = np.asarray(params.dow)[np.arange(n_days) % 7]
    mu = np.exp(a[:, None, None] + dow[None, :, None] + z[:, :, None]) * prof[None, None, :]
    dem = tweedie_sample(mu.ravel(), params.phi, params.power, 1, rng)[:, 0].reshape(mu.shape)
    return SyntheticPanel(params, prof, a, z, mu, dem)


def censor_synthetic(sp: SyntheticPanel, cfg: CensorConfig, seed: int) -> Injection:
    """Censor every synthetic day; the opening stock uses the true expected demand."""
    n_s, n_d, _ = sp.demand.shape
    rng = np.random.default_rng(seed)
    expected = sp.mu[:, :, LO:HI].sum(axis=-1).ravel()
    inj = inject_stockouts(sp.demand.reshape(-1, 24), stock_levels(expected, rng, cfg), rng, cfg)
    return Injection(
        observed=inj.observed.reshape(n_s, n_d, 24),
        flag=inj.flag.reshape(n_s, n_d, 24),
        depletion=inj.depletion.reshape(n_s, n_d),
        restock=inj.restock.reshape(n_s, n_d),
        censored=inj.censored.reshape(n_s, n_d, 24),
    )


In [ ]:
%%writefile src/sctwin/demand_model.py
"""Step 5: Bayesian latent-demand model ladder (NumPyro).

Hourly demand in the day window (hours 6-21) of series s, day d, hour h:

    eta[s,d,h] = alpha + a[s] + gamma[h] + g[s,h] + dow[d] + b_disc x_disc[s,d]
                 + b_act x_act[s,d] + u[s,d]
    P(D > 0) = sigmoid(k0 + k1 (eta - alpha))
    log D | D > 0 ~ Normal(m, sigma_y^2),  m chosen so that E[D] = exp(eta)

Sales are normalized, continuous values (docs/data_card.md), so the likelihood is a continuous
hurdle-lognormal rather than a count model. (A hurdle-Gamma was tried first; the gradient of
its survival function with respect to the shape made NUTS about 10x slower.) All random
effects are non-centred.

Observation model per hourly availability class (``sctwin.panel.availability_classes``):

    ==========  ==================  =====================================================
    treatment   meaning             log-likelihood contribution
    ==========  ==================  =====================================================
    exact       D = y               log f(y)
    censored    D >= y (lower bnd)  log P(D >= y)   (0 when y = 0)
    drop        no information      0
    ==========  ==================  =====================================================

Ladder:
    M0  static hierarchical, i.i.d. day effects; stockout hours treated as missing
        (A, P_pre exact; P_first, U dropped)
    M1  M0 + censoring: P_pre and P_first right-censored at the observed sales, U dropped
    M2  M1 + AR(1) day effects u[s,d] = rho u[s,d-1] + sigma_u e[s,d]
    M3  M2 on more series (same code, larger data)

Reconstruction of a censored day (Step 4 benchmark) is made as of the end of that day: global
parameters come from a posterior fitted on days strictly before it; the day's own effect
u[s,d] is inferred from that day's censored observations alone (grid posterior per draw); the
demand of censored hours is drawn from the predictive distribution truncated at the observation.
"""

from __future__ import annotations

import time
from dataclasses import dataclass, field
from typing import Any

import jax
import jax.numpy as jnp
import numpy as np
import numpyro
import numpyro.distributions as dist
from jax.scipy.special import log_ndtr
from scipy import special as sps

from sctwin.panel import CLS_A, CLS_P_FIRST, CLS_P_PRE, CLS_U, DAY_HOURS, availability_classes

jax.config.update("jax_enable_x64", True)

LO, HI = DAY_HOURS
NH = HI - LO
EXACT, CENSORED, DROP = 0, 1, 2
TREATMENT_CODES = {"exact": EXACT, "censored": CENSORED, "drop": DROP}
CLASS_NAMES = {CLS_A: "A", CLS_P_PRE: "P_pre", CLS_P_FIRST: "P_first", CLS_U: "U"}
DERIVED = ("a", "gamma", "g", "dow_eff", "u")
SCALARS = ("alpha", "sigma_a", "tau_g", "sigma_u", "b_disc", "b_act", "sigma_y", "k0", "k1")


@dataclass(frozen=True)
class ModelSpec:
    name: str
    treatment: dict[str, str]  # class name -> exact | censored | drop
    dynamic: bool = False

    def codes(self) -> np.ndarray:
        """Treatment code per class code (indexed by CLS_*)."""
        out = np.full(5, DROP, dtype=np.int8)
        for c, n in CLASS_NAMES.items():
            out[c] = TREATMENT_CODES[self.treatment[n]]
        return out


LADDER = {
    "M0": ModelSpec("M0", {"A": "exact", "P_pre": "exact", "P_first": "drop", "U": "drop"}),
    "M1": ModelSpec("M1", {"A": "exact", "P_pre": "censored", "P_first": "censored", "U": "drop"}),
    "M2": ModelSpec(
        "M2", {"A": "exact", "P_pre": "censored", "P_first": "censored", "U": "drop"}, True
    ),
    # Variants added after the Step 5 recovery run (not pre-declared): the first flagged hour
    # carries residual sales proportional to demand, for which "D >= y" is not the likelihood
    # and biases the level upward. Here only the depletion hour is censored.
    "M1d": ModelSpec("M1d", {"A": "exact", "P_pre": "censored", "P_first": "drop", "U": "drop"}),
    "M2d": ModelSpec(
        "M2d", {"A": "exact", "P_pre": "censored", "P_first": "drop", "U": "drop"}, True
    ),
}


# --------------------------------------------------------------------------- data


@dataclass
class ModelData:
    y: np.ndarray  # (S, D, NH) observed sales in hours 6-21
    cls: np.ndarray  # (S, D, NH) availability class codes
    dow: np.ndarray  # (D,) weekday 0-6
    x_disc: np.ndarray  # (S, D) 1 - discount (0 where the discount is invalid)
    x_act: np.ndarray  # (S, D) activity flag
    series: np.ndarray  # (S,) panel series index
    days: np.ndarray  # (D,) panel day index (consecutive)

    @property
    def shape(self) -> tuple[int, int]:
        return self.y.shape[0], self.y.shape[1]


def discount_feature(discount: np.ndarray) -> np.ndarray:
    ok = np.isfinite(discount) & (discount > 0) & (discount <= 1.5)
    return np.where(ok, 1.0 - discount, 0.0)


def make_model_data(
    sales: np.ndarray,
    stock: np.ndarray,
    discount: np.ndarray,
    activity: np.ndarray,
    weekday: np.ndarray,
    series: np.ndarray,
    days: np.ndarray,
) -> ModelData:
    """Model inputs for consecutive ``days`` of ``series`` (panel arrays are (S, D, 24))."""
    days = np.asarray(days)
    if days.size == 0 or not (np.diff(days) == 1).all():
        raise ValueError("days must be consecutive")
    ix = np.ix_(series, days)
    return ModelData(
        y=sales[ix][..., LO:HI].astype(np.float64),
        cls=availability_classes(stock[ix])[..., LO:HI],
        dow=np.asarray(weekday)[days].astype(np.int32),
        x_disc=discount_feature(discount[ix]),
        x_act=activity[ix].astype(np.float64),
        series=np.asarray(series),
        days=days,
    )


# --------------------------------------------------------------------------- likelihood


def hurdle_terms(eta: Any, alpha: Any, sigma_y: Any, k0: Any, k1: Any, xp: Any = jnp) -> tuple:
    """(log P(D>0), log P(D=0), m) where log D | D > 0 ~ N(m, sigma_y^2) and E[D] = exp(eta)."""
    z = k0 + k1 * (eta - alpha)
    log_p = -xp.logaddexp(0.0, -z)
    log_q = -xp.logaddexp(0.0, z)
    m = eta - log_p - 0.5 * sigma_y**2
    return log_p, log_q, m


def loglik(y: Any, code: Any, eta: Any, alpha: Any, sigma_y: Any, k0: Any, k1: Any) -> Any:
    """Pointwise log-likelihood (JAX) under the per-observation treatment codes."""
    log_p, log_q, m = hurdle_terms(eta, alpha, sigma_y, k0, k1)
    pos = y > 0
    ly = jnp.log(jnp.where(pos, y, 1.0))
    r = (ly - m) / sigma_y
    log_dens = -0.5 * r**2 - jnp.log(sigma_y) - 0.5 * jnp.log(2 * jnp.pi) - ly
    exact = jnp.where(pos, log_p + log_dens, log_q)
    cens = jnp.where(pos, log_p + log_ndtr(-r), 0.0)
    return jnp.where(code == EXACT, exact, jnp.where(code == CENSORED, cens, 0.0))


_loglik_jit = jax.jit(loglik)


# --------------------------------------------------------------------------- model


def _centred(x: Any, axis: int = -1) -> Any:
    return x - x.mean(axis=axis, keepdims=True)


def ar1(raw: Any, rho: Any, sigma: Any) -> Any:
    """AR(1) paths over the last axis from standard normal innovations (stationary start)."""
    first = raw[:, 0] * sigma / jnp.sqrt(1 - rho**2)

    def step(prev: Any, e: Any) -> tuple[Any, Any]:
        cur = rho * prev + sigma * e
        return cur, cur

    _, rest = jax.lax.scan(step, first, raw[:, 1:].T)
    return jnp.concatenate([first[:, None], rest.T], axis=1)


def linear_predictor(p: dict[str, Any], dow: Any, x_disc: Any, x_act: Any) -> Any:
    """eta without the day effect: (S, D, NH)."""
    return (
        p["alpha"]
        + p["a"][:, None, None]
        + p["gamma"][None, None, :]
        + p["g"][:, None, :]
        + p["dow_eff"][dow][None, :, None]
        + (p["b_disc"] * x_disc + p["b_act"] * x_act)[:, :, None]
    )


def demand_model(
    y: Any, code: Any, dow: Any, x_disc: Any, x_act: Any, dynamic: bool = False
) -> None:
    n_s, n_d = x_disc.shape
    n_h = NH
    alpha = numpyro.sample("alpha", dist.Normal(-2.0, 1.5))
    sigma_a = numpyro.sample("sigma_a", dist.HalfNormal(1.0))
    a = numpyro.deterministic(
        "a", sigma_a * numpyro.sample("a_raw", dist.Normal(0, 1).expand([n_s]))
    )
    gamma = numpyro.deterministic(
        "gamma", _centred(numpyro.sample("gamma_raw", dist.Normal(0, 1).expand([n_h])))
    )
    tau_g = numpyro.sample("tau_g", dist.HalfNormal(0.5))
    g_raw = numpyro.sample("g_raw", dist.Normal(0, 1).expand([n_s, n_h]))
    g = numpyro.deterministic("g", tau_g * _centred(_centred(g_raw, -1), 0))
    dow_eff = numpyro.deterministic(
        "dow_eff", _centred(numpyro.sample("dow_raw", dist.Normal(0, 0.5).expand([7])))
    )
    b_disc = numpyro.sample("b_disc", dist.Normal(0, 1.0))
    b_act = numpyro.sample("b_act", dist.Normal(0, 0.5))
    sigma_u = numpyro.sample("sigma_u", dist.HalfNormal(0.5))
    u_raw = numpyro.sample("u_raw", dist.Normal(0, 1).expand([n_s, n_d]))
    if dynamic:
        rho = numpyro.sample("rho", dist.Beta(2.0, 2.0))
        u = numpyro.deterministic("u", ar1(u_raw, rho, sigma_u))
    else:
        u = numpyro.deterministic("u", sigma_u * u_raw)
    sigma_y = numpyro.sample("sigma_y", dist.HalfNormal(1.5))
    k0 = numpyro.sample("k0", dist.Normal(0.0, 2.0))
    k1 = numpyro.sample("k1", dist.Normal(1.0, 1.0))
    p = {"alpha": alpha, "a": a, "gamma": gamma, "g": g, "dow_eff": dow_eff,
         "b_disc": b_disc, "b_act": b_act}  # fmt: skip
    eta = linear_predictor(p, dow, x_disc, x_act) + u[:, :, None]
    if y is not None:
        numpyro.factor("obs", loglik(y, code, eta, alpha, sigma_y, k0, k1).sum())


def model_args(data: ModelData, spec: ModelSpec) -> dict[str, Any]:
    return {
        "y": jnp.asarray(data.y),
        "code": jnp.asarray(spec.codes()[data.cls]),
        "dow": jnp.asarray(data.dow),
        "x_disc": jnp.asarray(data.x_disc),
        "x_act": jnp.asarray(data.x_act),
        "dynamic": spec.dynamic,
    }


# --------------------------------------------------------------------------- fitting


@dataclass
class FitResult:
    spec: ModelSpec
    samples: dict[str, np.ndarray]  # flattened over chains: (draws, ...)
    diagnostics: dict[str, Any]
    by_chain: dict[str, np.ndarray] = field(repr=False, default_factory=dict)


def fit(
    data: ModelData,
    spec: ModelSpec,
    num_warmup: int = 1000,
    num_samples: int = 1000,
    num_chains: int = 4,
    seed: int = 0,
    target_accept: float = 0.9,
    max_tree_depth: int = 10,
    chain_method: str = "parallel",
    progress: bool = False,
) -> FitResult:
    from numpyro.diagnostics import summary
    from numpyro.infer import MCMC, NUTS

    kernel = NUTS(demand_model, target_accept_prob=target_accept, max_tree_depth=max_tree_depth)
    if chain_method == "parallel" and jax.local_device_count() < num_chains:
        chain_method = "sequential"
    mcmc = MCMC(kernel, num_warmup=num_warmup, num_samples=num_samples, num_chains=num_chains,
                chain_method=chain_method, progress_bar=progress)  # fmt: skip
    t0 = time.perf_counter()
    mcmc.run(jax.random.PRNGKey(seed), **model_args(data, spec), extra_fields=("diverging",))
    jax.block_until_ready(mcmc.get_samples())  # JAX dispatch is asynchronous: wait for it
    runtime = time.perf_counter() - t0
    grouped = {k: np.asarray(v) for k, v in mcmc.get_samples(group_by_chain=True).items()}
    # convergence is judged on the sampled (non-centred) coordinates, not on derived quantities
    summ = summary({k: v for k, v in grouped.items() if k not in DERIVED}, group_by_chain=True)
    rhat = {k: float(np.nanmax(v["r_hat"])) for k, v in summ.items()}
    ess = {k: float(np.nanmin(v["n_eff"])) for k, v in summ.items()}
    div = int(np.asarray(mcmc.get_extra_fields()["diverging"]).sum())
    diag = {
        "num_warmup": num_warmup, "num_samples": num_samples, "num_chains": num_chains,
        "runtime_s": round(runtime, 1), "divergences": div,
        "max_rhat": max(rhat.values()), "min_ess": min(ess.values()),
        "rhat": rhat, "ess": ess,
        "scalars": {k: {"mean": float(summ[k]["mean"]), "sd": float(summ[k]["std"]),
                        "rhat": rhat[k], "ess": ess[k]}
                    for k in (*SCALARS, "rho") if k in summ},
    }  # fmt: skip
    flat = {k: v.reshape(-1, *v.shape[2:]) for k, v in grouped.items()}
    return FitResult(spec, flat, diag, grouped)


# --------------------------------------------------------------------------- checks


def pointwise_loglik(
    res: FitResult, data: ModelData, classes: tuple[int, ...] = (CLS_A,), max_draws: int = 1000
) -> np.ndarray:
    """(draws, n_obs) log-likelihood of the exact-treated observations in ``classes``."""
    s = res.samples
    idx = np.linspace(0, len(s["alpha"]) - 1, min(max_draws, len(s["alpha"]))).astype(int)
    mask = np.isin(data.cls, classes)
    code = jnp.zeros(int(mask.sum()), dtype=jnp.int8)
    y = jnp.asarray(data.y[mask])

    def one(i: int) -> Any:
        p = {k: s[k][i] for k in ("alpha", "a", "gamma", "g", "dow_eff", "b_disc", "b_act")}
        eta = linear_predictor(p, data.dow, data.x_disc, data.x_act) + s["u"][i][:, :, None]
        return loglik(y, code, eta[mask], s["alpha"][i], s["sigma_y"][i], s["k0"][i], s["k1"][i])

    return np.stack([np.asarray(one(int(i))) for i in idx])


def sample_hurdle(
    eta: np.ndarray, alpha: Any, sigma_y: Any, k0: Any, k1: Any, rng: np.random.Generator
) -> np.ndarray:
    log_p, _, m = hurdle_terms(eta, alpha, sigma_y, k0, k1, xp=np)
    pos = rng.random(eta.shape) < np.exp(log_p)
    return np.where(pos, np.exp(m + sigma_y * rng.standard_normal(eta.shape)), 0.0)


def sample_truncated(
    eta: np.ndarray, lower: np.ndarray, alpha: Any, sigma_y: Any, k0: Any, k1: Any,
    rng: np.random.Generator,
) -> np.ndarray:  # fmt: skip
    """Draws of D | D >= lower (lower > 0): the positive lognormal part truncated below,
    by inverting the survival function in log space (stable far in the tail)."""
    _, _, m = hurdle_terms(eta, alpha, sigma_y, k0, k1, xp=np)
    r0 = (np.log(lower) - m) / sigma_y
    log_q = sps.log_ndtr(-r0)  # log P(Z >= r0)
    log_w = log_q + np.log(rng.random(eta.shape))  # uniform on (0, q) in log space
    r = -sps.ndtri_exp(log_w)
    return np.maximum(np.exp(m + sigma_y * r), lower)


def predictive_checks(
    res: FitResult, data: ModelData, n_draws: int = 200, seed: int = 0
) -> dict[str, Any]:
    """Posterior predictive check on fully available (A) hours: statistics of the data vs the
    replicated data, with the share of replicates above the observed value (PPC p-value)."""
    rng = np.random.default_rng(seed)
    s = res.samples
    idx = rng.choice(len(s["alpha"]), size=min(n_draws, len(s["alpha"])), replace=False)
    mask = data.cls == CLS_A
    y = data.y[mask]

    def stats(v: np.ndarray) -> dict[str, float]:
        return {"zero_frac": float((v == 0).mean()), "mean": float(v.mean()),
                "q50": float(np.quantile(v, 0.5)), "q90": float(np.quantile(v, 0.9)),
                "q99": float(np.quantile(v, 0.99)), "max": float(v.max())}  # fmt: skip

    reps = []
    for i in idx:
        p = {k: s[k][i] for k in ("alpha", "a", "gamma", "g", "dow_eff", "b_disc", "b_act")}
        eta = np.asarray(linear_predictor(p, data.dow, data.x_disc, data.x_act))
        eta = eta + s["u"][i][:, :, None]
        reps.append(stats(sample_hurdle(eta[mask], s["alpha"][i], s["sigma_y"][i], s["k0"][i],
                                        s["k1"][i], rng)))  # fmt: skip
    obs = stats(y)
    out = {}
    for k, v in obs.items():
        r = np.array([x[k] for x in reps])
        lo, hi = np.quantile(r, [0.05, 0.95])
        out[k] = {"observed": v, "rep_q05": float(lo), "rep_q95": float(hi),
                  "ppc_p": float((r >= v).mean())}  # fmt: skip
    return out


# --------------------------------------------------------------------------- reconstruction


@dataclass
class DayRows:
    """Censored days to reconstruct: one row per (series, day), hours 6-21."""

    s_pos: np.ndarray  # (R,) position of the series in the fitted ModelData
    day: np.ndarray  # (R,) panel day index (must be after every fitted day)
    y: np.ndarray  # (R, NH) observed sales
    cls: np.ndarray  # (R, NH) availability classes from the observed flags
    dow: np.ndarray  # (R,)
    x_disc: np.ndarray  # (R,)
    x_act: np.ndarray  # (R,)


def reconstruct(
    res: FitResult,
    data: ModelData,
    rows: DayRows,
    n_draws: int = 400,
    grid: int = 121,
    seed: int = 0,
    chunk: int = 64,
) -> np.ndarray:
    """Posterior draws (R, n_draws, NH) of demand on each row's hours 6-21, as of the end of
    that day. Raises if a row's day is not strictly after the fitted window (point-in-time)."""
    if (rows.day <= data.days.max()).any():
        raise ValueError("reconstructed days must be after every day used for fitting")
    rng = np.random.default_rng(seed)
    s = res.samples
    idx = np.linspace(0, len(s["alpha"]) - 1, n_draws).astype(int)
    codes = res.spec.codes()[rows.cls]  # (R, NH)
    z = np.linspace(-6.0, 6.0, grid)
    out = np.empty((rows.y.shape[0], n_draws, NH))
    for j, i in enumerate(idx):
        p = {k: s[k][i] for k in ("alpha", "a", "gamma", "g", "dow_eff", "b_disc", "b_act")}
        al, sh, k0, k1 = (float(s[k][i]) for k in ("alpha", "sigma_y", "k0", "k1"))
        base = (p["alpha"] + p["a"][rows.s_pos][:, None] + p["gamma"][None, :]
                + p["g"][rows.s_pos] + p["dow_eff"][rows.dow][:, None]
                + (p["b_disc"] * rows.x_disc + p["b_act"] * rows.x_act)[:, None])  # fmt: skip
        su = float(s["sigma_u"][i])
        if res.spec.dynamic:
            rho = float(s["rho"][i])
            k = rows.day - data.days.max()
            m = rho**k * s["u"][i][rows.s_pos, -1]
            sd = su * np.sqrt((1 - rho ** (2 * k)) / (1 - rho**2))
        else:
            m, sd = np.zeros(len(rows.day)), np.full(len(rows.day), su)
        for c0 in range(0, len(rows.day), chunk):
            sl = slice(c0, c0 + chunk)
            u = m[sl, None] + sd[sl, None] * z[None, :]  # (r, G)
            eta = base[sl, None, :] + u[:, :, None]  # (r, G, NH)
            yy, cc = jnp.asarray(rows.y[sl, None, :]), jnp.asarray(codes[sl, None, :])
            ll = np.asarray(_loglik_jit(yy, cc, jnp.asarray(eta), al, sh, k0, k1)).sum(-1)
            lw = ll - 0.5 * z[None, :] ** 2
            w = np.exp(lw - lw.max(axis=1, keepdims=True))
            cdf = np.cumsum(w, axis=1)
            cdf /= cdf[:, -1:]
            pick = (cdf < rng.random((cdf.shape[0], 1))).sum(axis=1)
            zz = z[np.minimum(pick, grid - 1)] + (rng.random(pick.shape) - 0.5) * (z[1] - z[0])
            e = base[sl] + (m[sl] + sd[sl] * zz)[:, None]  # (r, NH)
            yv, cv = rows.y[sl], codes[sl]
            draw = sample_hurdle(e, al, sh, k0, k1, rng)
            trunc = (cv == CENSORED) & (yv > 0)
            if trunc.any():
                draw[trunc] = sample_truncated(e[trunc], yv[trunc], al, sh, k0, k1, rng)
            draw = np.where(cv == EXACT, yv, np.maximum(draw, yv))
            out[sl, j] = draw
    return out


# --------------------------------------------------------------------------- simulation


def simulate(
    params: dict[str, Any], dow: np.ndarray, x_disc: np.ndarray, x_act: np.ndarray, seed: int
) -> np.ndarray:
    """Demand (S, D, NH) from the model with given parameters (a, gamma, g, dow_eff, u, ...)."""
    eta = np.asarray(linear_predictor(params, dow, x_disc, x_act)) + params["u"][:, :, None]
    rng = np.random.default_rng(seed)
    return sample_hurdle(eta, params["alpha"], params["sigma_y"], params["k0"], params["k1"], rng)


def prior_draw(
    n_series: int, n_days: int, dynamic: bool, seed: int, x_disc: np.ndarray, x_act: np.ndarray,
    dow: np.ndarray,
) -> dict[str, Any]:  # fmt: skip
    """One draw of all parameters from the prior (for SBC and prior predictive checks)."""
    from numpyro.infer import Predictive

    pred = Predictive(demand_model, num_samples=1)
    d = pred(jax.random.PRNGKey(seed), y=None, code=None, dow=jnp.asarray(dow),
             x_disc=jnp.asarray(x_disc), x_act=jnp.asarray(x_act), dynamic=dynamic)  # fmt: skip
    return {k: np.asarray(v)[0] for k, v in d.items()}


## Files: Configs
Edit these cells, or use `config_with(...)` further down, to change run sizes.

In [ ]:
%%writefile configs/small.yaml
# Small, laptop/CI-friendly configuration.
seed: 20261112

dataset:
  repo_id: Dingdong-Inc/FreshRetailNet-50K
  # Pinned Hub commit SHA (printed by scripts/fetch_data.py on first download), then frozen.
  revision: c374e85975c82816c495acfa904cc55ab14a47d3
  data_dir: data/raw

splits:
  fit_days: 76          # days 1-76 of official train
  val_days: 14          # days 77-90 of official train
  # official `eval` split (7 days) is the locked final test
  holdout_store_frac: 0.10
  holdout_product_frac: 0.05

subset:
  n_series: 20          # first Bayesian experiments


In [ ]:
%%writefile configs/baselines.yaml
# Step 3: rolling-origin backtest of baseline forecasters on the validation window.
# All lead / horizon / step / window values are in the selected unit (CLAUDE.md Rule 1).
seed: 20261112
data_dir: data/raw
manifest: configs/manifests/splits_v1.json
out_dir: runs/step3
taus: [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]

units:
  day:
    lead: 1              # origin T = end of day T; targets days T+2 .. T+8
    horizon: 7           # one weekly review cycle
    step: 7              # folds at origins 74, 81 -> validation days 76..89
    n_folds: null        # null = as many folds as fit in the validation window
    window: sliding      # sliding | expanding
    window_length: 42    # feature times t in [T-L-H-41, T-L-H]
    min_history: 28
    train_stride: 1
    train_series: 12000  # seeded sample of in-sample series used for fitting
    eval_series: all     # all in-sample series are forecast
  hour:
    lead: 1              # origin T = end of hour T; targets hours T+2 .. T+25
    horizon: 24          # next 24 hours
    step: 24             # one fold per day, origin at 22:00 -> 14 folds over validation
    n_folds: null
    window: sliding
    window_length: 504   # 21 days of feature times
    min_history: 672     # 28 days of history before the first training example
    train_stride: 6      # every 6th hour as a training feature time
    train_series: 500
    eval_series: 500     # same seeded sample (hourly is expensive)

glm: {power: 1.5, alpha: 0.0001, n_samples: 400}
lgbm: {num_rounds: 300, num_threads: 12}
bootstrap: {n_boot: 2000, reference: clim, point_reference: snaive}


In [ ]:
%%writefile configs/censoring.yaml
# Step 4: censoring benchmark (evaluation track 2, latent-demand recovery on a proxy truth).
seed: 20261112
data_dir: data/raw
manifest: configs/manifests/splits_v1.json
out_dir: runs/step4
n_series: 5000          # seeded sample of in-sample series
window: 28              # days of history before a day used for its expected demand / hour means

calibration:            # fitted on the fit window only (days window .. fit_days - 1)
  coverages: {start: 0.5, stop: 2.0, step: 0.05}
  sds: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
  leak_first: 0.18      # residual sales in flagged hours (docs/data_card.md, Finding 3)
  leak_later: 0.08
  # p_restock and restock_mean_hours are measured on the fit window

positions: [-3, -2, -1, 0, 1]   # hours relative to the first flagged hour (realism check)

# Pre-declared sensitivity scenarios: overrides of the calibrated configuration.
sensitivity:
  no_leak: {leak_first: 0.0, leak_later: 0.0}
  high_leak: {leak_first: 0.5, leak_later: 0.25}
  no_restock: {p_restock: 0.0}
  scarce_stock: {coverage_mult: 0.8}
  ample_stock: {coverage_mult: 1.25}

bootstrap: {n_boot: 2000, reference: profile}


In [ ]:
%%writefile configs/bayes.yaml
# Step 5: Bayesian latent-demand model ladder.
seed: 20261112
data_dir: data/raw
manifest: configs/manifests/splits_v1.json
out_dir: runs/step5

models: [M0, M1, M2]     # M3 = M2 run with --n-series 100
n_series: 20             # seeded sample of in-sample series (100 is a superset of 20)
fit_days: fit            # the official fit window (days 0..75); every benchmark day is later

mcmc: {num_warmup: 1000, num_samples: 1000, num_chains: 4, target_accept: 0.9, max_tree_depth: 10}

# Step 4 benchmark, rebuilt for these series on the validation days with the calibrated
# injection configuration of runs/step4/8ba303c79bd4 (docs/results_step4.md).
benchmark:
  window: 28
  seed_offset: 1
  injection: {coverage: 1.15, coverage_sd: 0.6, leak_first: 0.16711784156890871,
              leak_later: 0.07448787932732963, p_restock: 0.187, restock_mean_hours: 5.75}

reconstruction: {n_draws: 400, grid: 121}
taus: [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]
checks: {ppc_draws: 200, loo_draws: 1000}
bootstrap: {n_boot: 2000, references: [hour_mean, profile]}

# Parameter recovery: simulate from the model with known parameters on the real covariates and
# calendar of the same series, inject stockouts with the Step 4 mechanism, fit the ladder.
recovery:
  truth: {alpha: -2.0, sigma_a: 0.8, tau_g: 0.3, sigma_u: 0.35, b_disc: 0.8, b_act: 0.3,
          sigma_y: 0.9, k0: 0.5, k1: 1.0, rho: 0.5}
  dynamic_truth: true


In [ ]:
%%writefile configs/manifests/splits_v1.json
{
  "manifest_version": 1,
  "seed": 20261112,
  "source": {
    "repo_id": "Dingdong-Inc/FreshRetailNet-50K",
    "revision": "c374e85975c82816c495acfa904cc55ab14a47d3",
    "train_sha256": "6706832db892bbae4969c19d87e07975d2543d2ba7d7d4756360654785de5a3d"
  },
  "temporal": {
    "fit": {
      "start": "2024-03-28",
      "end": "2024-06-11",
      "n_days": 76
    },
    "val": {
      "start": "2024-06-12",
      "end": "2024-06-25",
      "n_days": 14
    },
    "test": {
      "split": "official eval",
      "locked": true
    }
  },
  "group_holdout": {
    "rule": "series is held out if store_id in stores OR product_id in products",
    "store_frac": 0.1,
    "product_frac": 0.05,
    "stores": [
      0,
      14,
      36,
      56,
      59,
      76,
      77,
      80,
      83,
      112,
      119,
      120,
      123,
      135,
      143,
      146,
      164,
      165,
      172,
      180,
      188,
      231,
      278,
      285,
      293,
      301,
      318,
      338,
      343,
      372,
      382,
      394,
      396,
      413,
      418,
      419,
      423,
      430,
      440,
      442,
      444,
      445,
      449,
      490,
      505,
      522,
      529,
      533,
      537,
      541,
      549,
      558,
      567,
      584,
      593,
      594,
      599,
      604,
      610,
      623,
      626,
      629,
      634,
      638,
      647,
      653,
      656,
      672,
      673,
      674,
      681,
      691,
      692,
      703,
      717,
      721,
      744,
      748,
      751,
      757,
      770,
      774,
      780,
      782,
      798,
      801,
      840,
      847,
      854,
      875
    ],
    "products": [
      7,
      20,
      37,
      39,
      46,
      59,
      87,
      123,
      131,
      181,
      185,
      201,
      251,
      252,
      278,
      294,
      321,
      326,
      327,
      363,
      366,
      388,
      447,
      462,
      487,
      510,
      513,
      557,
      593,
      606,
      611,
      612,
      623,
      642,
      677,
      679,
      682,
      725,
      730,
      741,
      754,
      846,
      849
    ]
  },
  "series_counts": {
    "total": 50000,
    "in_sample": 43883,
    "holdout_store_only": 4758,
    "holdout_product_only": 1242,
    "holdout_both": 117
  },
  "digest": "f27ff354ca686ff8bc811a5f1227c8eabe11f9caf0d6907e65b6ba9cc32a47db"
}

## Files: Scripts
Command-line entry points; the run cells below call them.

In [ ]:
%%writefile scripts/fetch_data.py
"""Download FreshRetailNet-50K at a pinned revision and record checksums.

Usage: python scripts/fetch_data.py [--config configs/small.yaml] [--verify]
"""

from __future__ import annotations

import argparse
import json

from sctwin.data import fetch, verify_checksums
from sctwin.repro import load_config


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/small.yaml")
    ap.add_argument("--verify", action="store_true", help="only re-check local checksums")
    a = ap.parse_args()
    cfg = load_config(a.config)["dataset"]
    if a.verify:
        res = verify_checksums(cfg["data_dir"])
        print(json.dumps(res, indent=2))
        raise SystemExit(0 if all(res.values()) else 1)
    prov = fetch(cfg["data_dir"], revision=cfg.get("revision"), repo_id=cfg["repo_id"])
    print(json.dumps(prov, indent=2))
    if not cfg.get("revision"):
        print(f"\nPin this in the config:  dataset.revision: {prov['revision']}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/audit_data.py
"""Run the Step 1 audit and write split manifests + the 20-series development subset.

Usage: python scripts/audit_data.py [--config configs/small.yaml]
Outputs (all under data/, never committed except the manifests copied to configs/manifests/):
  data/audit/audit_report.json
  data/audit/run_manifest.json
  configs/manifests/splits_v1.json      (immutable)
  data/subsets/dev20.parquet
"""

from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import polars as pl

from sctwin.audit import run_audit
from sctwin.data import SERIES_KEY, load_split, verify_checksums
from sctwin.repro import load_config, seed_everything, write_run_manifest
from sctwin.splits import assign_roles, make_manifest, write_manifest


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/small.yaml")
    a = ap.parse_args()
    cfg = load_config(a.config)
    seed_everything(cfg["seed"])
    data_dir = Path(cfg["dataset"]["data_dir"])

    checks = verify_checksums(data_dir)
    if not all(checks.values()):
        raise SystemExit(f"checksum mismatch: {checks}")
    prov = json.loads((data_dir / "provenance.json").read_text(encoding="utf-8"))

    t0 = time.perf_counter()
    train, test = load_split(data_dir, "train"), load_split(data_dir, "eval")
    report = run_audit(train, test)
    report["provenance"] = prov
    report["runtime_s"] = round(time.perf_counter() - t0, 1)

    out = Path("data/audit")
    out.mkdir(parents=True, exist_ok=True)
    (out / "audit_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    write_run_manifest(out, cfg, stage="01_data_audit", all_gates_pass=report["all_gates_pass"])

    sp = cfg["splits"]
    m = make_manifest(
        train,
        fit_days=sp["fit_days"],
        val_days=sp["val_days"],
        holdout_store_frac=sp["holdout_store_frac"],
        holdout_product_frac=sp["holdout_product_frac"],
        seed=cfg["seed"],
        source={
            "repo_id": prov["repo_id"],
            "revision": prov["revision"],
            "train_sha256": prov["files"]["train"]["sha256"],
        },
    )
    write_manifest(m, "configs/manifests/splits_v1.json")

    # Development subset: in-sample series only, seeded sample (train + val days only).
    tr = assign_roles(train, m).filter(~pl.col("holdout"))
    keys = tr.select(SERIES_KEY).unique().sort(SERIES_KEY)
    keys = keys.sample(n=cfg["subset"]["n_series"], seed=cfg["seed"])
    sub = tr.join(keys, on=list(SERIES_KEY), how="semi")
    Path("data/subsets").mkdir(parents=True, exist_ok=True)
    sub.write_parquet("data/subsets/dev20.parquet")

    print(
        json.dumps(
            {
                "gates": report["gates"],
                "all_gates_pass": report["all_gates_pass"],
                "series_counts": m["series_counts"],
                "runtime_s": report["runtime_s"],
            },
            indent=2,
        )
    )


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/run_baselines.py
"""Step 3: rolling-origin backtest of baseline forecasters on the validation window.

Usage: python scripts/run_baselines.py --unit day|hour [--config ...] [--smoke]
Writes runs/step3/<unit>/<config_hash>/{predictions.parquet, metrics.json, run_manifest.json}.
Only the official train split is loaded; the locked test split is never read here.
"""

from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import numpy as np

from sctwin.backtest import BacktestSpec, make_origins
from sctwin.baseline_runner import add_groups, run_backtest
from sctwin.data import load_split
from sctwin.evaluation import compare_to_reference, evaluate
from sctwin.features import FeatureBuilder
from sctwin.panel import build_panel
from sctwin.repro import config_hash, load_config, seed_everything, write_run_manifest
from sctwin.splits import read_manifest
from sctwin.timeunit import TIMEZONE, resample


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--unit", choices=["day", "hour"], required=True)
    ap.add_argument("--config", default="configs/baselines.yaml")
    ap.add_argument("--smoke", action="store_true", help="tiny run to check the pipeline")
    args = ap.parse_args()
    cfg = load_config(args.config)
    uc = cfg["units"][args.unit]
    if args.smoke:
        uc.update(train_series=200, eval_series=200, n_folds=2)
        cfg["lgbm"]["num_rounds"] = 20
        cfg["glm"]["n_samples"] = 100
        cfg["bootstrap"]["n_boot"] = 200
        cfg["out_dir"] = "runs/step3_smoke"
    run_cfg = {k: v for k, v in cfg.items() if k != "units"} | {"unit": args.unit, "unit_cfg": uc}
    rng = seed_everything(cfg["seed"])
    out = Path(cfg["out_dir"]) / args.unit / config_hash(run_cfg)
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()

    manifest = read_manifest(cfg["manifest"])
    panel = build_panel(load_split(cfg["data_dir"], "train"))
    g = manifest["group_holdout"]
    holdout = panel.keys["store_id"].is_in(g["stores"]) | panel.keys["product_id"].is_in(
        g["products"]
    )
    in_sample = np.flatnonzero(~holdout.to_numpy())

    def sample(n: int | str) -> np.ndarray:
        if n == "all" or int(n) >= in_sample.size:
            return in_sample
        return np.sort(rng.choice(in_sample, int(n), replace=False))

    train_s = sample(uc["train_series"])
    eval_s = train_s if uc["eval_series"] == uc["train_series"] else sample(uc["eval_series"])
    used = np.union1d(train_s, eval_s)
    sub = panel.subset(used)  # resample only the series we need (hourly is large)
    pos = {int(s): i for i, s in enumerate(used)}
    tr_local = np.array([pos[int(s)] for s in train_s])
    ev_local = np.array([pos[int(s)] for s in eval_s])

    us = resample(sub, args.unit)
    upd = us.units_per_day
    n_fit_days = manifest["temporal"]["fit"]["n_days"]
    first_target, last_target = n_fit_days * upd, panel.shape[1] * upd - 1  # validation units
    origins = make_origins(first_target, last_target, uc["lead"], uc["horizon"], uc["step"],
                           uc["n_folds"])  # fmt: skip
    spec = BacktestSpec(args.unit, uc["lead"], uc["horizon"], origins, uc["window"],
                        uc["window_length"], uc["min_history"], uc["train_stride"])  # fmt: skip

    # low volume: bottom quartile of mean sales over units <= the first origin (no later data)
    hist = us.sales[:, : origins[0] + 1].mean(axis=1)
    low_vol = set(np.flatnonzero(hist <= np.quantile(hist[ev_local], 0.25)).tolist())

    fb = FeatureBuilder(us, sub.keys)
    pred, fitlog = run_backtest(fb, spec, tr_local, ev_local, cfg)
    pred = add_groups(pred, fb, low_vol)
    pred.write_parquet(out / "predictions.parquet")
    taus = tuple(cfg["taus"])
    bs = cfg["bootstrap"]
    report = {
        "protocol": {
            "unit": args.unit,
            "timezone_assumed": TIMEZONE,
            "lead": spec.lead,
            "horizon": spec.horizon,
            "step": uc["step"],
            "window": spec.window,
            "window_length": spec.window_length,
            "train_stride": spec.train_stride,
            "origins": list(origins),
            "n_folds": len(origins),
            "rule": "train on feature times t <= T - L - H; predict (T + L, T + L + H]",
            "targets": "validation window only; official test split not loaded",
            "features": list(fb.feature_names),
        },
        "n_train_series": int(train_s.size),
        "n_eval_series": int(eval_s.size),
        "fit_log": fitlog,
        "metrics": evaluate(pred, taus),
        "compare_wis": compare_to_reference(
            pred, bs["reference"], "wis", taus, n_boot=bs["n_boot"], seed=cfg["seed"]
        ),  # fmt: skip
        "compare_ae": compare_to_reference(
            pred, bs["point_reference"], "ae", taus, n_boot=bs["n_boot"], seed=cfg["seed"]
        ),  # fmt: skip
        "runtime_s": round(time.perf_counter() - t0, 1),
    }
    (out / "metrics.json").write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")
    write_run_manifest(out, run_cfg, stage="03_baselines", smoke=args.smoke)

    ov = report["metrics"]["overall"]["models"]
    print(f"\n== {args.unit}: {len(origins)} folds, {eval_s.size} series, validation window")
    for m, r in ov.items():
        cov = r.get("intervals", {}).get("0.9", {}).get("coverage", float("nan"))
        mae = r.get("mae", r.get("mae_median", float("nan")))
        print(f"  {m:8s} mae={mae:.4f} wis={r.get('wis', float('nan')):.4f} cov90={cov:.3f}")
    print(f"\nwritten to {out}  ({report['runtime_s']} s)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/run_censoring_benchmark.py
"""Step 4: semi-synthetic censoring benchmark on real fully in-stock days.

Usage: python scripts/run_censoring_benchmark.py [--config configs/censoring.yaml] [--smoke]

1. Measure real stockout statistics on the fit window (rate, timing, restocks, sales around
   the first flagged hour).
2. Calibrate the injection (stock coverage and spread) on fully in-stock fit-window days.
3. Build the benchmark on fully in-stock validation days, check its realism against step 1,
   and score the reconstruction baselines on the injected (known) demand proxy.
4. Repeat for the pre-declared sensitivity scenarios.

Writes runs/step4/<config_hash>/{metrics.json, run_manifest.json}. Only the official train
split is loaded; the locked test split is never read here.
"""

from __future__ import annotations

import argparse
import dataclasses
import json
import time
from pathlib import Path
from typing import Any

import numpy as np

from sctwin.censoring import (
    HI,
    IMPUTERS,
    LO,
    Benchmark,
    CensorConfig,
    build_benchmark,
    calibrate,
    calibrate_leak,
    calibrate_restock,
    candidate_days,
    expected_window_demand,
    position_ratios,
    recovery_frames,
    stockout_stats,
)
from sctwin.data import load_split
from sctwin.evaluation import compare_to_reference, evaluate
from sctwin.panel import build_panel
from sctwin.repro import config_hash, load_config, seed_everything, write_run_manifest
from sctwin.splits import read_manifest

TAUS = (0.5,)  # point methods only in Step 4


def score(bm: Benchmark, n_boot: int, reference: str, seed: int) -> dict[str, Any]:
    est = {m: f(bm.injection.observed, bm.injection.flag, bm.hour_means)
           for m, f in IMPUTERS.items()}  # fmt: skip
    hourly, daily = recovery_frames(bm, est)
    out: dict[str, Any] = {
        "n_days": int(bm.series.size),
        "n_stockout_days": int(daily.height),
        "n_censored_hours": int(hourly.height),
        "hourly": evaluate(hourly, TAUS),
        "daily": evaluate(daily, TAUS),
    }
    if n_boot:
        out["compare_hourly_ae"] = compare_to_reference(hourly, reference, "ae", TAUS, n_boot, seed)
        out["compare_daily_ae"] = compare_to_reference(daily, reference, "ae", TAUS, n_boot, seed)
    return out


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/censoring.yaml")
    ap.add_argument("--smoke", action="store_true")
    args = ap.parse_args()
    cfg = load_config(args.config)
    if args.smoke:
        cfg.update(n_series=300, out_dir="runs/step4_smoke")
        cfg["bootstrap"]["n_boot"] = 200
    rng = seed_everything(cfg["seed"])
    out = Path(cfg["out_dir"]) / config_hash(cfg)
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()

    manifest = read_manifest(cfg["manifest"])
    panel = build_panel(load_split(cfg["data_dir"], "train"))
    g = manifest["group_holdout"]
    hold = panel.keys["store_id"].is_in(g["stores"]) | panel.keys["product_id"].is_in(g["products"])
    in_sample = np.flatnonzero(~hold.to_numpy())
    series = np.sort(rng.choice(in_sample, min(cfg["n_series"], in_sample.size), replace=False))
    sub = panel.subset(series)
    sales, stock = sub.sales, sub.stock
    s_local = np.arange(series.size)
    n_fit, n_days, w = manifest["temporal"]["fit"]["n_days"], panel.shape[1], cfg["window"]
    fit_days, val_days = np.arange(w, n_fit), np.arange(n_fit, n_days)

    # 1. real stockout statistics, fit window only
    real_flags = stock[:, w:n_fit].reshape(-1, 24)
    real = stockout_stats(real_flags)
    full_fit = (stock[:, :n_fit, LO:HI] == 0).all(-1)  # (S, n_fit) fully in-stock days
    cnt = full_fit.sum(1)[:, None]
    in_stock_sum = (sales[:, :n_fit] * full_fit[..., None]).sum(1)
    baseline = np.where(cnt > 0, in_stock_sum / np.maximum(cnt, 1), np.nan)
    pos = tuple(cfg["positions"])
    real_pos = position_ratios(
        sales[:, w:n_fit].reshape(-1, 24), real_flags,
        np.repeat(baseline, n_fit - w, axis=0), pos,
    )  # fmt: skip

    # 2. calibration on fully in-stock fit-window days
    cal = cfg["calibration"]
    base = CensorConfig(leak_first=cal["leak_first"], leak_later=cal["leak_later"],
                        p_restock=real["restock_rate"],
                        restock_mean_hours=max(1.0, real["restock_mean_hours"]))  # fmt: skip
    fit_rows = candidate_days(sales, stock, s_local, fit_days, w)
    c = cal["coverages"]
    best, grid = calibrate(
        sales[fit_rows[0], fit_rows[1]].astype(np.float64),
        expected_window_demand(fit_rows[2]),
        real, base,
        np.round(np.arange(c["start"], c["stop"] + 1e-9, c["step"]), 4),
        np.asarray(cal["sds"], dtype=np.float64), seed=cfg["seed"],
    )  # fmt: skip
    best = calibrate_restock(
        sales[fit_rows[0], fit_rows[1]].astype(np.float64),
        expected_window_demand(fit_rows[2]),
        real, best,
        np.round(np.linspace(0.5, 2.0, 16) * real["restock_rate"], 4),
        np.round(np.arange(1.0, 8.01, 0.25), 2), seed=cfg["seed"],
    )  # fmt: skip
    best = calibrate_leak(
        sales[fit_rows[0], fit_rows[1]].astype(np.float64),
        expected_window_demand(fit_rows[2]),
        baseline[fit_rows[0]], real_pos, best, seed=cfg["seed"],
    )  # fmt: skip
    print(f"calibrated: {dataclasses.asdict(best)}")

    # 3. benchmark on fully in-stock validation days
    val_rows = candidate_days(sales, stock, s_local, val_days, w)
    bs = cfg["bootstrap"]
    bm = build_benchmark(sales, val_rows, best, seed=cfg["seed"] + 1)
    inj_pos = position_ratios(bm.injection.observed, bm.injection.flag,
                              baseline[bm.series], pos)  # fmt: skip
    main_res = score(bm, bs["n_boot"], bs["reference"], cfg["seed"])

    # 4. sensitivity scenarios (same days, same random numbers)
    sens: dict[str, Any] = {}
    for name, ov in cfg["sensitivity"].items():
        ov = dict(ov)
        mult = ov.pop("coverage_mult", 1.0)
        sc = dataclasses.replace(best, coverage=best.coverage * mult, **ov)
        b2 = build_benchmark(sales, val_rows, sc, seed=cfg["seed"] + 1)
        r = score(b2, 0, bs["reference"], cfg["seed"])
        sens[name] = {"config": dataclasses.asdict(sc),
                      "stats": stockout_stats(b2.injection.flag),
                      "hourly": r["hourly"]["overall"], "daily": r["daily"]["overall"]}  # fmt: skip

    report = {
        "truth": "demand proxy = real sales on fully in-stock days (hours 6-21); not demand",
        "series": int(series.size),
        "window_days": w,
        "calibration_days": [int(fit_days[0]), int(fit_days[-1])],
        "benchmark_days": [int(val_days[0]), int(val_days[-1])],
        "real_fit_window": {"stats": real, "position_ratios": real_pos},
        "injected": {"stats": stockout_stats(bm.injection.flag), "position_ratios": inj_pos},
        "calibrated_config": dataclasses.asdict(best),
        "calibration_grid_top": sorted(grid, key=lambda r: r["loss"])[:10],
        "benchmark": main_res,
        "sensitivity": sens,
        "runtime_s": round(time.perf_counter() - t0, 1),
    }
    (out / "metrics.json").write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")
    write_run_manifest(out, cfg, stage="04_censoring", smoke=args.smoke)

    print(f"\nreal (fit)  : {real}\ninjected    : {report['injected']['stats']}")
    print(f"positions real {real_pos}\n          inj  {inj_pos}")
    print(f"\n== {main_res['n_stockout_days']} stockout days / {main_res['n_days']} days, "
          f"{main_res['n_censored_hours']} censored hours")  # fmt: skip
    for level in ("hourly", "daily"):
        for m, r in main_res[level]["overall"]["models"].items():
            print(f"  {level:6s} {m:12s} mae={r['mae']:.4f} wape={r['wape']:.3f} "
                  f"bias={r['bias']:+.4f}")  # fmt: skip
    print(f"\nwritten to {out}  ({report['runtime_s']} s)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/run_bayes.py
"""Step 5: fit the Bayesian demand-model ladder and evaluate it.

Usage:
  python scripts/run_bayes.py [--config configs/bayes.yaml] [--models M0,M1,M2]
                              [--n-series 20] [--part real|recovery|all] [--smoke]

real      fit each model on the fit window (days 0..75) of n seeded series; diagnostics
          (R-hat, ESS, divergences), posterior predictive checks, PSIS-LOO on in-stock hours;
          reconstruct the Step 4 benchmark days of those series (validation days, all after the
          fitted window) and score them against the baselines on the same rows.
recovery  simulate from the model with known parameters on the same series' covariates and
          calendar, inject stockouts with the Step 4 mechanism, fit the ladder, check that the
          90% credible intervals cover the true values.

Writes runs/step5/<config_hash>/{metrics.json, posterior_<model>.npz, run_manifest.json}.
Only the official train split is loaded; the locked test split is never read here.
"""

from __future__ import annotations

import argparse
import json
import time
from pathlib import Path
from typing import Any

import numpy as np


def parse() -> argparse.Namespace:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/bayes.yaml")
    ap.add_argument("--models", default=None, help="comma-separated, e.g. M0,M1,M2")
    ap.add_argument("--n-series", type=int, default=None)
    ap.add_argument("--part", choices=["real", "recovery", "all"], default="all")
    ap.add_argument("--smoke", action="store_true")
    return ap.parse_args()


ARGS = parse()

import numpyro  # noqa: E402  (host devices must be set before JAX initialises)

from sctwin.repro import config_hash, load_config, seed_everything, write_run_manifest  # noqa: E402

CFG = load_config(ARGS.config)
if ARGS.smoke:
    CFG.update(n_series=4, out_dir="runs/step5_smoke")
    CFG["mcmc"].update(num_warmup=150, num_samples=100, num_chains=2)
    CFG["reconstruction"]["n_draws"] = 50
    CFG["checks"].update(ppc_draws=20, loo_draws=100)
    CFG["bootstrap"]["n_boot"] = 200
if ARGS.models:
    CFG["models"] = ARGS.models.split(",")
if ARGS.n_series:
    CFG["n_series"] = ARGS.n_series
numpyro.set_host_device_count(CFG["mcmc"]["num_chains"])

import arviz as az  # noqa: E402

from sctwin import demand_model as dm  # noqa: E402
from sctwin.censoring import (  # noqa: E402
    HI,
    IMPUTERS,
    LO,
    CensorConfig,
    build_benchmark,
    candidate_days,
    hour_profile,
    inject_stockouts,
    recovery_frames,
    stock_levels,
)
from sctwin.data import load_split  # noqa: E402
from sctwin.evaluation import compare_to_reference, evaluate, qcol  # noqa: E402
from sctwin.panel import Panel, availability_classes, build_panel  # noqa: E402
from sctwin.splits import read_manifest  # noqa: E402


def mcmc_kwargs(seed: int) -> dict[str, Any]:
    m = CFG["mcmc"]
    return {"num_warmup": m["num_warmup"], "num_samples": m["num_samples"],
            "num_chains": m["num_chains"], "target_accept": m["target_accept"],
            "max_tree_depth": m["max_tree_depth"], "seed": seed}  # fmt: skip


def loo(ll: np.ndarray) -> Any:
    return az.loo(az.from_dict({"posterior": {"x": np.zeros((1, ll.shape[0]))},
                                "log_likelihood": {"y": ll[None]}}))  # fmt: skip


def save_posterior(path: Path, res: dm.FitResult) -> None:
    keep = {k: v.astype(np.float32) for k, v in res.samples.items()
            if not k.endswith("_raw") and k != "u"}  # fmt: skip
    keep["u_last"] = res.samples["u"][:, :, -1].astype(np.float32)  # state at the window end
    np.savez_compressed(path, **keep)


# --------------------------------------------------------------------------- real data


def run_real(panel: Panel, series: np.ndarray, weekday: np.ndarray, n_fit: int,
             out: Path) -> dict[str, Any]:  # fmt: skip
    sub = panel.subset(series)
    n_days = panel.shape[1]
    s_local = np.arange(series.size)
    fit_days = np.arange(n_fit)
    data = dm.make_model_data(sub.sales, sub.stock, sub.discount, sub.activity, weekday,
                              s_local, fit_days)  # fmt: skip

    b = CFG["benchmark"]
    rows = candidate_days(sub.sales, sub.stock, s_local, np.arange(n_fit, n_days), b["window"])
    bm = build_benchmark(sub.sales, rows, CensorConfig(**b["injection"]),
                         seed=CFG["seed"] + b["seed_offset"])  # fmt: skip
    if bm.day.min() <= fit_days.max():
        raise AssertionError("benchmark days must be after the fitted window (Rule 1)")
    inj = bm.injection
    so = np.flatnonzero(inj.stocked_out)
    day_rows = dm.DayRows(
        s_pos=bm.series[so], day=bm.day[so], y=inj.observed[so, LO:HI],
        cls=availability_classes(inj.flag[so])[:, LO:HI], dow=weekday[bm.day[so]],
        x_disc=dm.discount_feature(sub.discount[bm.series[so], bm.day[so]]),
        x_act=sub.activity[bm.series[so], bm.day[so]].astype(np.float64),
    )  # fmt: skip
    print(f"fit: {series.size} series x {n_fit} days; benchmark: {bm.series.size} days, "
          f"{so.size} with a stockout", flush=True)  # fmt: skip

    est = {m: f(inj.observed, inj.flag, bm.hour_means) for m, f in IMPUTERS.items()}
    totals: dict[str, np.ndarray] = {}
    fits: dict[str, Any] = {}
    lls: dict[str, np.ndarray] = {}
    for k, name in enumerate(CFG["models"]):
        spec = dm.LADDER[name]
        print(f"[{name}] fitting ...", flush=True)
        res = dm.fit(data, spec, **mcmc_kwargs(CFG["seed"] + k))
        d = res.diagnostics
        print(f"[{name}] {d['runtime_s']} s, divergences={d['divergences']}, "
              f"max R-hat={d['max_rhat']:.3f}, min ESS={d['min_ess']:.0f}", flush=True)  # fmt: skip
        save_posterior(out / f"posterior_{name}.npz", res)
        lls[name] = dm.pointwise_loglik(res, data, max_draws=CFG["checks"]["loo_draws"])
        lo = loo(lls[name])
        rc = CFG["reconstruction"]
        draws = dm.reconstruct(res, data, day_rows, n_draws=rc["n_draws"], grid=rc["grid"],
                               seed=CFG["seed"])  # fmt: skip
        hourly_est = inj.observed.copy()
        hourly_est[so, LO:HI] = np.median(draws, axis=1)
        est[name] = hourly_est
        totals[name] = draws.sum(axis=-1)  # (R, n_draws) hours 6-21
        fits[name] = {
            "diagnostics": d,
            "ppc_in_stock_hours": dm.predictive_checks(
                res, data, CFG["checks"]["ppc_draws"], seed=CFG["seed"]
            ),  # fmt: skip
            "loo_in_stock_hours": {
                "elpd": float(lo.elpd),
                "se": float(lo.se),
                "p_loo": float(lo.p),
                "pareto_k_gt_0.7": int((np.asarray(lo.pareto_k) > 0.7).sum()),
                "n_obs": int(lls[name].shape[1]),
            },  # fmt: skip
        }

    hourly, daily = recovery_frames(bm, est)
    taus = tuple(CFG["taus"])
    for name, t in totals.items():
        q = np.quantile(t, taus, axis=1).T
        daily = daily.with_columns(
            **{f"{name}__point": np.quantile(t, 0.5, axis=1)},
            **{qcol(name, tau): q[:, i] for i, tau in enumerate(taus)},
        )
    bs = CFG["bootstrap"]
    comp = {}
    for ref in bs["references"]:
        comp[f"daily_ae_vs_{ref}"] = compare_to_reference(daily, ref, "ae", taus, bs["n_boot"],
                                                          CFG["seed"])  # fmt: skip
        comp[f"hourly_ae_vs_{ref}"] = compare_to_reference(hourly, ref, "ae", (0.5,),
                                                           bs["n_boot"], CFG["seed"])  # fmt: skip
    if len(totals) > 1 and "M0" in totals:
        comp["daily_wis_vs_M0"] = compare_to_reference(daily, "M0", "wis", taus, bs["n_boot"],
                                                       CFG["seed"])  # fmt: skip
    names = list(lls)
    best = max(names, key=lambda n: fits[n]["loo_in_stock_hours"]["elpd"])
    loo_cmp = {}
    for n in names:  # paired pointwise differences (same in-stock observations)
        da = np.asarray(loo(lls[n]).elpd_i) - np.asarray(loo(lls[best]).elpd_i)
        loo_cmp[n] = {"elpd_diff": float(da.sum()), "se_diff": float(np.sqrt(da.size) * da.std())}
    return {
        "truth": "demand proxy = real sales on fully in-stock validation days (Step 4)",
        "n_series": int(series.size), "fit_days": [0, int(n_fit - 1)],
        "benchmark_days": [int(bm.day.min()), int(bm.day.max())],
        "n_benchmark_days": int(bm.series.size), "n_stockout_days": int(so.size),
        "models": fits, "loo_compare": {"best": best, "models": loo_cmp},
        "daily": evaluate(daily, taus), "hourly": evaluate(hourly, (0.5,)),
        "comparisons": comp,
    }  # fmt: skip


# --------------------------------------------------------------------------- recovery


def run_recovery(panel: Panel, series: np.ndarray, weekday: np.ndarray, n_fit: int) -> dict:
    sub = panel.subset(series)
    n_s, days = series.size, np.arange(n_fit)
    rng = np.random.default_rng(CFG["seed"] + 7)
    t = dict(CFG["recovery"]["truth"])
    x_disc = dm.discount_feature(sub.discount[:, :n_fit])
    x_act = sub.activity[:, :n_fit].astype(np.float64)
    raw_g = rng.standard_normal((n_s, dm.NH))
    raw_u = rng.standard_normal((n_s, n_fit))
    rho = t["rho"] if CFG["recovery"]["dynamic_truth"] else 0.0
    truth = {
        **t, "a": t["sigma_a"] * rng.standard_normal(n_s),
        "gamma": dm._centred(np.log(hour_profile()[LO:HI])),
        "g": t["tau_g"] * np.asarray(dm._centred(dm._centred(raw_g, -1), 0)),
        "dow_eff": dm._centred(np.array([0.0, -0.05, -0.05, 0.0, 0.1, 0.2, 0.15])),
        "u": np.asarray(dm.ar1(raw_u, rho, t["sigma_u"])),
    }  # fmt: skip
    dow = weekday[days]
    demand = dm.simulate(truth, dow, x_disc, x_act, seed=CFG["seed"] + 8)
    eta = np.asarray(dm.linear_predictor(truth, dow, x_disc, x_act)) + truth["u"][:, :, None]
    full = np.zeros((n_s * n_fit, 24))
    full[:, LO:HI] = demand.reshape(-1, dm.NH)
    cfg = CensorConfig(**CFG["benchmark"]["injection"])
    r2 = np.random.default_rng(CFG["seed"] + 9)
    k = stock_levels(np.exp(eta).sum(-1).ravel(), r2, cfg)
    inj = inject_stockouts(full, k, r2, cfg)
    data = dm.ModelData(
        y=inj.observed[:, LO:HI].reshape(n_s, n_fit, dm.NH),
        cls=availability_classes(inj.flag)[:, LO:HI].reshape(n_s, n_fit, dm.NH),
        dow=dow, x_disc=x_disc, x_act=x_act, series=series, days=days,
    )  # fmt: skip
    true_level = float(np.exp(eta).sum(-1).mean())
    out: dict[str, Any] = {"truth": t, "stockout_day_rate": float(inj.stocked_out.mean()),
                           "true_mean_window_demand": true_level, "models": {}}  # fmt: skip
    for j, name in enumerate(CFG["models"]):
        res = dm.fit(data, dm.LADDER[name], **mcmc_kwargs(CFG["seed"] + 100 + j))
        s = res.samples
        rows = {}
        for p in (*dm.SCALARS, "rho"):
            if p not in s or p not in t:
                continue
            lo, hi = np.quantile(s[p], [0.05, 0.95])
            rows[p] = {"true": t[p], "mean": float(s[p].mean()), "q05": float(lo),
                       "q95": float(hi), "covered": bool(lo <= t[p] <= hi)}  # fmt: skip
        # posterior mean of the expected hours 6-21 demand per day, averaged (level recovery)
        idx = np.linspace(0, len(s["alpha"]) - 1, 200).astype(int)
        lv = []
        for i in idx:
            p = {k: s[k][i] for k in ("alpha", "a", "gamma", "g", "dow_eff", "b_disc", "b_act")}
            e = np.asarray(dm.linear_predictor(p, dow, x_disc, x_act)) + s["u"][i][:, :, None]
            lv.append(np.exp(e).sum(-1).mean())
        lo, hi = np.quantile(lv, [0.05, 0.95])
        out["models"][name] = {
            "diagnostics": {k: res.diagnostics[k]
                            for k in ("runtime_s", "divergences", "max_rhat", "min_ess")},
            "scalars": rows,
            "mean_window_demand": {"mean": float(np.mean(lv)), "q05": float(lo), "q95": float(hi),
                                   "covered": bool(lo <= true_level <= hi)},
        }  # fmt: skip
        print(
            f"[recovery {name}] level {np.mean(lv):.3f} vs true {true_level:.3f}; covered "
            f"{sum(r['covered'] for r in rows.values())}/{len(rows)} scalars",
            flush=True,
        )
    return out


def main() -> None:
    rng = seed_everything(CFG["seed"])
    out = Path(CFG["out_dir"]) / config_hash(CFG)
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    manifest = read_manifest(CFG["manifest"])
    panel = build_panel(load_split(CFG["data_dir"], "train"))
    g = manifest["group_holdout"]
    hold = panel.keys["store_id"].is_in(g["stores"]) | panel.keys["product_id"].is_in(g["products"])
    in_sample = np.flatnonzero(~hold.to_numpy())
    series = np.sort(rng.permutation(in_sample)[: CFG["n_series"]])  # prefix: 20 within 100
    weekday = np.array([d.weekday() for d in panel.dates])
    n_fit = manifest["temporal"]["fit"]["n_days"]

    report: dict[str, Any] = {"config": CFG, "series": series.tolist()}
    if ARGS.part in ("real", "all"):
        report["real"] = run_real(panel, series, weekday, n_fit, out)
    if ARGS.part in ("recovery", "all"):
        report["recovery"] = run_recovery(panel, series, weekday, n_fit)
    report["runtime_s"] = round(time.perf_counter() - t0, 1)
    (out / f"metrics_{ARGS.part}.json").write_text(json.dumps(report, indent=2, default=str),
                                                   encoding="utf-8")  # fmt: skip
    write_run_manifest(out, CFG, stage="05_bayes", part=ARGS.part, smoke=ARGS.smoke)

    if "real" in report:
        dly = report["real"]["daily"]["overall"]["models"]
        print("\n== daily totals on stockout days (proxy truth)")
        for m, r in dly.items():
            cov = r.get("intervals", {}).get("0.9", {}).get("coverage", float("nan"))
            print(f"  {m:12s} mae={r['mae']:.4f} bias={r['bias']:+.4f} cov90={cov:.3f}")
    print(f"\nwritten to {out}  ({report['runtime_s']} s)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/run_sbc.py
"""Step 5: simulation-based calibration (SBC) of the censored model M1.

Usage: python scripts/run_sbc.py [--reps 200] [--n-series 4] [--n-days 14] [--seed 1]

Each replication draws all parameters from the prior, simulates hourly demand, censors it with
the Step 4 mechanism and fits M1. If inference is correct, the rank of every true scalar among
the (thinned) posterior draws is uniform. For the censored likelihood to be exact here, the
opening stock is drawn independently of the parameters and flagged hours sell nothing
(leak = 0). Writes runs/sbc/<hash>/{ranks.json, run_manifest.json}.
"""

from __future__ import annotations

import argparse
import functools
import json
import time
from pathlib import Path

import numpy as np

ap = argparse.ArgumentParser()
ap.add_argument("--reps", type=int, default=200)
ap.add_argument("--n-series", type=int, default=4)
ap.add_argument("--n-days", type=int, default=14)
ap.add_argument("--warmup", type=int, default=400)
ap.add_argument("--samples", type=int, default=400)
ap.add_argument("--thin", type=int, default=4)
ap.add_argument("--seed", type=int, default=1)
ap.add_argument("--model", default="M1")
args = ap.parse_args()

import jax  # noqa: E402
from numpyro.infer import MCMC, NUTS  # noqa: E402
from scipy import stats  # noqa: E402

from sctwin import demand_model as dm  # noqa: E402
from sctwin.censoring import HI, LO, CensorConfig, inject_stockouts  # noqa: E402
from sctwin.panel import availability_classes  # noqa: E402
from sctwin.repro import config_hash, write_run_manifest  # noqa: E402

SCAL = [s for s in dm.SCALARS] + (["rho"] if dm.LADDER[args.model].dynamic else [])


def main() -> None:
    cfg = vars(args)
    out = Path("runs/sbc") / config_hash(cfg)
    out.mkdir(parents=True, exist_ok=True)
    spec = dm.LADDER[args.model]
    rng = np.random.default_rng(args.seed)
    n_s, n_d = args.n_series, args.n_days
    inj_cfg = CensorConfig(leak_first=0.0, leak_later=0.0, p_restock=0.187, restock_mean_hours=5.75)
    model = functools.partial(dm.demand_model, dynamic=spec.dynamic)
    mcmc = MCMC(NUTS(model, target_accept_prob=0.9), num_warmup=args.warmup,
                num_samples=args.samples, num_chains=1, progress_bar=False,
                jit_model_args=True)  # fmt: skip
    ranks: dict[str, list[int]] = {k: [] for k in SCAL}
    log = []
    t0 = time.perf_counter()
    for r in range(args.reps):
        x_disc = np.where(rng.random((n_s, n_d)) < 0.3, rng.uniform(0, 0.4, (n_s, n_d)), 0.0)
        x_act = (rng.random((n_s, n_d)) < 0.2).astype(np.float64)
        dow = np.arange(n_d) % 7
        theta = dm.prior_draw(n_s, n_d, spec.dynamic, int(rng.integers(2**31)), x_disc, x_act, dow)
        demand = dm.simulate(theta, dow, x_disc, x_act, seed=int(rng.integers(2**31)))
        full = np.zeros((n_s * n_d, 24))
        full[:, LO:HI] = demand.reshape(-1, dm.NH)
        stock = np.exp(rng.normal(np.log(2.0), 1.0, n_s * n_d))  # independent of theta
        inj = inject_stockouts(
            full, stock, np.random.default_rng(int(rng.integers(2**31))), inj_cfg
        )
        data = dm.ModelData(
            y=inj.observed[:, LO:HI].reshape(n_s, n_d, dm.NH),
            cls=availability_classes(inj.flag)[:, LO:HI].reshape(n_s, n_d, dm.NH),
            dow=dow, x_disc=x_disc, x_act=x_act, series=np.arange(n_s), days=np.arange(n_d),
        )  # fmt: skip
        a = dm.model_args(data, spec)
        a.pop("dynamic")
        mcmc.run(jax.random.PRNGKey(int(rng.integers(2**31))), **a, extra_fields=("diverging",))
        s = {k: np.asarray(v)[:: args.thin] for k, v in mcmc.get_samples().items()}
        div = int(np.asarray(mcmc.get_extra_fields()["diverging"]).sum())
        for k in SCAL:
            ranks[k].append(int((s[k] < theta[k]).sum()))
        log.append({"rep": r, "divergences": div, "stockout_rate": float(inj.stocked_out.mean())})
        if (r + 1) % 10 == 0:
            print(f"rep {r + 1}/{args.reps}  {time.perf_counter() - t0:.0f} s", flush=True)

    n_post = len(range(0, args.samples, args.thin))
    bins = 10
    res = {}
    for k, v in ranks.items():
        h = np.histogram(v, bins=bins, range=(0, n_post + 1))[0]
        res[k] = {"hist": h.tolist(), "chi2_p": float(stats.chisquare(h).pvalue),
                  "mean_rank_frac": float(np.mean(v) / n_post)}  # fmt: skip
    p_holm = np.minimum(
        1.0, np.sort([r["chi2_p"] for r in res.values()]) * np.arange(len(res), 0, -1)
    )
    report = {"config": cfg, "n_post_draws": n_post, "scalars": res, "ranks": ranks,
              "min_p_holm": float(np.min(np.maximum.accumulate(p_holm))),
              "divergent_reps": int(sum(x["divergences"] > 0 for x in log)),
              "mean_stockout_rate": float(np.mean([x["stockout_rate"] for x in log])),
              "runtime_s": round(time.perf_counter() - t0, 1)}  # fmt: skip
    (out / "ranks.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    write_run_manifest(out, cfg, stage="05_sbc")
    for k, r in res.items():
        print(f"  {k:8s} chi2 p={r['chi2_p']:.3f} mean rank={r['mean_rank_frac']:.2f} {r['hist']}")
    print(f"min Holm-adjusted p = {report['min_p_holm']:.3f}; written to {out}")


if __name__ == "__main__":
    main()


## Files: Tests
Synthetic tests (no data needed) and real-data tests (`-m data`).

In [ ]:
%%writefile tests/conftest.py
"""Synthetic FreshRetailNet-shaped fixtures. No network or real data needed."""

from __future__ import annotations

from datetime import date, timedelta

import numpy as np
import polars as pl
import pytest

QUANTUM = 0.0123  # synthetic "global normalization coefficient"


def make_frame(
    n_stores: int = 4,
    n_products: int = 5,
    n_days: int = 10,
    start: date = date(2024, 3, 1),
    seed: int = 0,
) -> pl.DataFrame:
    """Daily rows with 24-hour arrays. Stockouts start at a random hour and persist to 23,
    so every series has known, internally consistent stockout counts."""
    rng = np.random.default_rng(seed)
    rows = []
    profile = np.r_[np.zeros(6), np.ones(16), np.zeros(2)] + 0.05
    for s in range(n_stores):
        for p in range(n_products):
            for d in range(n_days):
                units = rng.poisson(3 * profile)
                status = np.zeros(24, dtype=np.int64)
                if rng.random() < 0.3:
                    status[rng.integers(8, 24) :] = 1
                    units = np.where(status == 1, 0, units)
                hs = units * QUANTUM
                rows.append(
                    {
                        "city_id": s % 2,
                        "store_id": 100 + s,
                        "management_group_id": p % 2,
                        "first_category_id": p,
                        "second_category_id": p,
                        "third_category_id": p,
                        "product_id": 500 + p,
                        "dt": (start + timedelta(days=d)).isoformat(),
                        "sale_amount": float(hs.sum()),
                        "hours_sale": hs.tolist(),
                        "stock_hour6_22_cnt": int(status[6:22].sum()),
                        "hours_stock_status": status.tolist(),
                        "discount": float(rng.choice([1.0, 0.9])),
                        "holiday_flag": int(d % 7 == 0),
                        "activity_flag": int(rng.random() < 0.2),
                        "precpt": float(d % 3),
                        "avg_temperature": 20.0 + d % 5,
                        "avg_humidity": 60.0,
                        "avg_wind_level": 2.0,
                    }
                )
    return pl.DataFrame(rows).with_columns(pl.col("stock_hour6_22_cnt").cast(pl.Int32))


@pytest.fixture
def frame() -> pl.DataFrame:
    return make_frame()


@pytest.fixture
def train_test() -> tuple[pl.DataFrame, pl.DataFrame]:
    tr = make_frame(n_days=20, seed=1)
    te = make_frame(n_days=7, start=date(2024, 3, 21), seed=2)
    return tr, te


In [ ]:
%%writefile tests/test_audit.py
"""The audit must find defects we plant, and must not raise false alarms on clean data."""

import json

import numpy as np
import polars as pl
from conftest import QUANTUM

from sctwin import audit


def test_clean_fixture_passes_all_gates(train_test):
    rep = audit.run_audit(*train_test)
    assert rep["all_gates_pass"], {k: v for k, v in rep["gates"].items() if not v}
    json.dumps(rep)  # serializable


def test_detects_duplicate_keys(frame):
    dup = pl.concat([frame, frame.head(3)])
    assert audit.check_structure(dup)["duplicate_keys"] == 3


def test_detects_date_gap(frame):
    gapped = frame.filter(pl.col("dt") != "2024-03-05")
    s = audit.check_structure(gapped)
    assert s["series_with_date_gaps"] == s["n_series"]


def test_detects_daily_hourly_mismatch(frame):
    bad = frame.with_columns(
        pl.when(pl.int_range(pl.len()) < 4)
        .then(pl.col("sale_amount") + 1)
        .otherwise(pl.col("sale_amount"))
    )
    v = audit.check_values(bad)
    assert v["daily_vs_hourly_sum"]["frac_rows_abs_diff_gt_1e-6"] == round(4 / frame.height, 8)


def test_detects_negative_sales(frame):
    bad = frame.with_columns(
        pl.when(pl.int_range(pl.len()) == 0)
        .then(pl.col("hours_sale").list.eval(pl.element() - 1))
        .otherwise(pl.col("hours_sale"))
    )
    assert audit.check_values(bad)["hourly_negative"] > 0


def test_quantum_recovered():
    rng = np.random.default_rng(0)
    vals = rng.integers(1, 50, 10_000) * QUANTUM
    q = audit.detect_quantum(vals)["quantum"]
    assert q is not None and q["divisor"] == 1
    np.testing.assert_allclose(q["q"], QUANTUM)


def test_quantum_when_min_is_not_one_unit():
    rng = np.random.default_rng(0)
    vals = rng.integers(2, 50, 10_000) * 2 * QUANTUM + rng.integers(0, 2, 10_000) * QUANTUM
    vals = vals[vals > 0]
    q = audit.detect_quantum(vals)["quantum"]
    assert q is not None
    np.testing.assert_allclose(q["q"], QUANTUM, rtol=1e-9)


def test_no_quantum_for_continuous_values():
    vals = np.random.default_rng(0).gamma(2.0, 1.0, 10_000)
    assert audit.detect_quantum(vals)["quantum"] is None


def test_stockout_count_window_identified(frame):
    s = audit.check_stockouts(frame)
    assert s["cnt_field_match_by_window"]["h6_21"] == 1.0
    assert set(s["status_values"]) <= {"0", "1"}


def test_stockout_runs_persist_in_fixture(frame):
    r = audit.check_stockouts(frame)["runs"]
    assert r["frac_runs_ending_at_hour_23"] == 1.0
    assert r["frac_positive_later_flagged_hours"] == 0.0


def test_non_binary_status_fails_gate(train_test):
    tr, te = train_test
    bad = tr.with_columns(pl.col("hours_stock_status").list.eval(pl.element() * 2))
    rep = audit.run_audit(bad, te)
    assert not rep["gates"]["train.status_binary"]


def test_cross_split_detects_overlap(train_test):
    tr, te = train_test
    leaky = pl.concat([te, tr.tail(5)])
    c = audit.check_cross_split(tr, leaky)
    assert c["overlapping_keys"] == 5
    assert c["series_test_not_strictly_after_train"] > 0


def test_covariate_granularity(frame):
    c = audit.check_covariates(frame)
    assert c["max_distinct_holiday_per_date"] == 1
    assert c["max_distinct_city_per_store"] == 1


def test_lattice_detects_tenth_grid(frame):
    lat = audit.check_lattice(
        frame.with_columns(pl.col("hours_sale").list.eval(pl.element() * 0) + pl.lit(0.0)),
        steps=(0.1,),
    )
    assert lat["0.1"]["frac_series_fully_on_lattice"] == 1.0  # all zeros: trivially on lattice
    tenth = frame.with_columns(
        pl.col("hours_sale").list.eval((pl.element() / QUANTUM).round(0) * 0.1)
    )
    assert audit.check_lattice(tenth, steps=(0.1,))["0.1"]["frac_positive_on_lattice"] == 1.0
    assert audit.check_lattice(frame, steps=(0.1,))["0.1"]["frac_positive_on_lattice"] < 0.5


def _plant_surge(df: pl.DataFrame, factor: float) -> pl.DataFrame:
    hs = np.array(df["hours_sale"].to_list())
    st = np.array(df["hours_stock_status"].to_list())
    for i in range(len(hs)):
        on = np.flatnonzero(st[i])
        if on.size and on[0] >= 7:
            hs[i, on[0] - 1] *= factor
    return df.with_columns(hours_sale=pl.Series(hs.tolist()), sale_amount=pl.Series(hs.sum(axis=1)))


def test_transitions_exogenous_stockouts_ratio_near_one():
    from conftest import make_frame

    df = make_frame(n_stores=6, n_products=6, n_days=60, seed=3)
    t = audit.check_availability_transitions(df, min_full_days=5)
    assert 0.7 < t["before_run_start"]["-1h"]["ratio"] < 1.3
    assert t["first_flagged_hour"]["ratio"] == 0.0


def test_transitions_detect_demand_driven_depletion():
    from conftest import make_frame

    df = _plant_surge(make_frame(n_stores=6, n_products=6, n_days=60, seed=3), factor=3.0)
    t = audit.check_availability_transitions(df, min_full_days=5)
    assert t["before_run_start"]["-1h"]["ratio"] > 2.0
    assert 0.7 < t["before_run_start"]["-2h"]["ratio"] < 1.3


def test_hierarchy_strict_tree(frame):
    assert audit.check_hierarchy(frame)["is_strict_tree"]
    bad = frame.with_columns(
        city_id=pl.when(pl.int_range(pl.len()) == 0).then(99).otherwise(pl.col("city_id"))
    )
    h = audit.check_hierarchy(bad)
    assert not h["is_strict_tree"] and h["store_id->city_id"]["children_with_multiple_parents"] == 1


In [ ]:
%%writefile tests/test_data.py
import numpy as np
import polars as pl
import pytest

from sctwin.data import hourly_matrix, to_hourly_long, validate_schema


def test_schema_ok_on_fixture(frame):
    rep = validate_schema(frame)
    assert rep.ok, rep


def test_schema_detects_missing_and_wrong_type(frame):
    bad = frame.drop("discount").with_columns(pl.col("sale_amount").cast(pl.Utf8))
    rep = validate_schema(bad)
    assert "discount" in rep.missing
    assert rep.wrong_type["sale_amount"] == ("float", "str")
    assert not rep.ok


def test_hourly_matrix_shape_and_values(frame):
    m = hourly_matrix(frame, "hours_sale")
    assert m.shape == (frame.height, 24)
    np.testing.assert_allclose(m.sum(axis=1), frame["sale_amount"].to_numpy())


def test_hourly_matrix_rejects_wrong_length(frame):
    bad = frame.with_columns(
        pl.when(pl.int_range(pl.len()) == 0)
        .then(pl.col("hours_sale").list.head(23))
        .otherwise(pl.col("hours_sale"))
        .alias("hours_sale")
    )
    with pytest.raises(ValueError, match="expected 24"):
        hourly_matrix(bad, "hours_sale")


def test_hourly_long_preserves_totals(frame):
    long = to_hourly_long(frame).collect()
    assert long.height == frame.height * 24
    assert set(long["hour"].unique().to_list()) == set(range(24))
    np.testing.assert_allclose(long["sale"].sum(), frame["sale_amount"].sum())
    assert long["stockout"].sum() == frame["hours_stock_status"].list.sum().sum()
    # hour index aligns with array position
    first = long.filter(
        (pl.col("store_id") == 100) & (pl.col("product_id") == 500) & (pl.col("dt") == "2024-03-01")
    ).sort("hour")
    np.testing.assert_allclose(first["sale"].to_numpy(), frame["hours_sale"][0].to_numpy())


In [ ]:
%%writefile tests/test_features.py
import numpy as np
import polars as pl
import pytest
from test_no_leakage import SMALL, builder, random_panel

from sctwin.features import FeatureBuilder, trailing_mean, units_since
from sctwin.panel import (
    CLS_A,
    CLS_NIGHT,
    CLS_P_FIRST,
    CLS_P_PRE,
    CLS_U,
    availability_classes,
    build_panel,
)
from sctwin.timeunit import resample

# --------------------------------------------------------------------------- primitives


def test_trailing_mean_hand():
    x = np.array([[1.0, 2, 3, 4]])
    np.testing.assert_array_equal(trailing_mean(x, 2), [[np.nan, 1, 1.5, 2.5]])
    m = np.array([[True, False, True, True]])
    np.testing.assert_array_equal(trailing_mean(x, 2, mask=m), [[np.nan, 1, 1, 3]])


def test_units_since():
    e = np.array([[0, 1, 0, 0, 1, 0]], dtype=bool)
    np.testing.assert_array_equal(units_since(e), [[np.nan, np.nan, 1, 2, 3, 1]])


def test_availability_classes_hand():
    st = np.zeros(24, dtype=np.int8)
    st[[10, 11, 12, 15, 3]] = 1
    c = availability_classes(st)
    expect = {9: CLS_P_PRE, 10: CLS_P_FIRST, 11: CLS_U, 12: CLS_U, 13: CLS_A, 14: CLS_P_PRE,
              15: CLS_P_FIRST, 16: CLS_A, 3: CLS_NIGHT, 22: CLS_NIGHT, 6: CLS_A}  # fmt: skip
    for h, v in expect.items():
        assert c[h] == v, h


# --------------------------------------------------------------------------- panel / resample


def test_build_panel_roundtrip(frame):
    p = build_panel(frame.sample(fraction=1.0, shuffle=True, seed=1))
    assert p.shape == (20, 10)
    i = p.keys.with_row_index().filter((pl.col("store_id") == 102) & (pl.col("product_id") == 503))[
        "index"
    ][0]
    row = frame.filter((pl.col("store_id") == 102) & (pl.col("product_id") == 503)).sort("dt")
    np.testing.assert_allclose(p.sales[i], np.array(row["hours_sale"].to_list()))


def test_build_panel_rejects_holes_and_duplicates(frame):
    with pytest.raises(ValueError, match="rectangle"):
        build_panel(frame.slice(1))
    with pytest.raises(ValueError, match="duplicate"):
        build_panel(pl.concat([frame, frame.head(1)]))


def test_resample_day_and_hour():
    p = random_panel(2, 5, 0)
    d, h = resample(p, "day"), resample(p, "hour")
    np.testing.assert_allclose(d.sales, p.sales.sum(-1))
    assert h.sales.shape == (2, 120)
    np.testing.assert_array_equal(h.sales[1, 2 * 24 + 7], p.sales[1, 2, 7])
    np.testing.assert_array_equal(d.stockout[0], p.stock[0, :, 6:22].sum(-1))
    assert h.last_complete_day(np.array([22, 23, 47, 48])).tolist() == [-1, 0, 1, 1]
    assert d.last_complete_day(np.array([0, 3])).tolist() == [0, 3]
    with pytest.raises(ValueError):
        resample(p, "week")


# --------------------------------------------------------------------------- features


@pytest.mark.parametrize("unit", ["day", "hour"])
def test_lags_are_relative_to_origin(unit):
    p = random_panel(1, 12, 2)
    fb = builder(p, unit)
    x = fb.us.sales[0]
    origin = 200 if unit == "hour" else 6
    r = fb.rows([origin], lead=2, horizon=3)
    for k in fb.cfg.lags:
        np.testing.assert_array_equal(r[f"sale_lag{k}"].to_numpy(), x[origin - k])
    w = fb.cfg.windows[0]
    np.testing.assert_allclose(r[f"sale_mean{w}"][0], x[origin - w + 1 : origin + 1].mean())
    assert (r["steps_ahead"].to_numpy() == [3, 4, 5]).all()
    assert (r["target_idx"].to_numpy() == origin + np.array([3, 4, 5])).all()


def test_seasonal_last_and_instock_mean_bruteforce():
    p = random_panel(1, 20, 3)
    fb = builder(p, "hour")
    origin, lead, horizon = 10 * 24 + 13, 1, 24
    r = fb.rows([origin], lead, horizon)
    x, ok = fb.us.sales[0], fb.us.instock_next_known[0]
    for row in r.iter_rows(named=True):
        u = row["target_idx"]
        src = [u - 24 * k for k in range(1, 40) if u - 24 * k <= origin][: fb.cfg.n_seasons]
        assert row["sale_seasonal_last"] == x[src[0]]
        good = [x[i] for i in src if ok[i] and i + 1 <= origin]
        exp = np.mean(good) if good else np.nan
        np.testing.assert_allclose(row["sale_seasonal_instock_mean"], exp)


def test_no_target_time_promotions_in_features():
    fb = builder(random_panel(1, 10, 0), "day")
    assert not any(n in fb.feature_names for n in ("discount", "activity", "target_discount"))
    assert all(not n.startswith("y") for n in fb.feature_names)


def test_discount_anomalies_flagged():
    p = random_panel(2, 10, 0)
    fb = builder(p, "day")
    bad = (p.discount <= 0) | (p.discount > 1)
    assert fb.discount_bad.sum() == bad.sum() and np.isnan(fb.discount[bad]).all()


def test_default_configs_build():
    p = random_panel(2, 40, 1)
    for unit in ("day", "hour"):
        fb = FeatureBuilder(resample(p, unit), p.keys)
        r = fb.rows([fb.us.n_units - 30], 1, 7)
        assert set(fb.feature_names) <= set(r.columns) and r.height == 14
    assert SMALL["day"].season == 7


In [ ]:
%%writefile tests/test_metrics.py
"""Metrics against hand calculations, closed forms and known identities."""

import math

import numpy as np
import pytest

from sctwin import metrics as M

TAUS = [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]


def _normal_ppf(p):
    # inverse CDF via bisection on erf (avoids a scipy dependency in core tests)
    lo, hi = -10.0, 10.0
    for _ in range(80):
        mid = (lo + hi) / 2
        if 0.5 * (1 + math.erf(mid / math.sqrt(2))) < p:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2


def test_mae_wape():
    y, f = np.array([1.0, 2, 3]), np.array([2.0, 2, 1])
    assert M.mae(y, f) == pytest.approx(1.0)
    assert M.wape(y, f) == pytest.approx(3 / 6)
    assert M.wape(np.zeros(3), f) is None


def test_pinball_hand():
    out = M.pinball(np.array([1.0]), np.array([[0.0, 2.0]]), [0.1, 0.9])
    np.testing.assert_allclose(out, [[0.1, 0.1]])


def test_wis_hand():
    # y=0, quantiles (0.25, 0.5, 0.75) = (1, 2, 3): (0.5*2 + 0.25*(2 + 4*1)) / 1.5
    w = M.wis(np.array([0.0]), np.array([[1.0, 2.0, 3.0]]), [0.25, 0.5, 0.75])
    np.testing.assert_allclose(w, [2.5 / 1.5])


def test_wis_equals_scaled_pinball_sum():
    rng = np.random.default_rng(0)
    y = rng.gamma(2, 1, 500)
    q, _ = M.sort_quantiles(rng.gamma(2, 1, (500, len(TAUS))))
    k = (len(TAUS) - 1) / 2
    np.testing.assert_allclose(M.wis(y, q, TAUS), M.pinball(y, q, TAUS).sum(1) / (k + 0.5))


def test_wis_rejects_bad_taus():
    with pytest.raises(ValueError):
        M.wis(np.zeros(1), np.zeros((1, 2)), [0.1, 0.9])
    with pytest.raises(ValueError):
        M.wis(np.zeros(1), np.zeros((1, 3)), [0.1, 0.5, 0.8])


def test_crps_normal_closed_form_value():
    # y = mu, sigma = 1: 2 phi(0) - 1/sqrt(pi)
    np.testing.assert_allclose(M.crps_normal(np.array([0.0]), 0.0, 1.0), [0.2336949], atol=1e-6)


def test_crps_samples_matches_normal():
    rng = np.random.default_rng(1)
    y = np.array([-1.0, 0.0, 0.7, 2.5])
    samples = rng.normal(0.0, 1.0, (4, 200_000))
    np.testing.assert_allclose(M.crps_samples(y, samples), M.crps_normal(y, 0.0, 1.0), rtol=1e-2)


def test_crps_single_sample_is_absolute_error():
    np.testing.assert_allclose(M.crps_samples(np.array([1.0]), np.array([[3.5]])), [2.5])


def test_crps_quantiles_dense_grid_matches_normal():
    taus = np.linspace(0.005, 0.995, 199)
    q = np.array([[_normal_ppf(t) for t in taus]] * 3)
    y = np.array([-0.5, 0.0, 1.2])
    np.testing.assert_allclose(M.crps_quantiles(y, q, taus), M.crps_normal(y, 0.0, 1.0), rtol=2e-2)


def test_interval_coverage_calibrated():
    rng = np.random.default_rng(2)
    y = rng.normal(size=20_000)
    q = np.tile([_normal_ppf(t) for t in TAUS], (y.size, 1))
    cov = M.interval_coverage(y, q, TAUS)
    for level in ("0.9", "0.8", "0.5"):
        assert abs(cov[level]["coverage"] - cov[level]["nominal"]) < 0.015


def test_randomized_pit_uniform_with_point_mass():
    rng = np.random.default_rng(3)
    y = rng.poisson(0.7, 20_000).astype(float)  # ~50% zeros
    s = rng.poisson(0.7, (20_000, 400)).astype(float)
    hist, _ = np.histogram(M.randomized_pit(y, s, rng), bins=10, range=(0, 1))
    assert np.all(np.abs(hist / y.size - 0.1) < 0.015)


def test_sort_quantiles():
    q, frac = M.sort_quantiles(np.array([[1.0, 0.5, 2.0], [0.0, 1.0, 2.0]]))
    np.testing.assert_array_equal(q, [[0.5, 1, 2], [0, 1, 2]])
    assert frac == 0.5


def test_bootstrap_identical_losses():
    loss = np.random.default_rng(0).random(100)
    r = M.paired_bootstrap(loss, loss, np.arange(100) // 10)
    assert r.mean_diff == 0 and r.ci_low == 0 and r.ci_high == 0 and r.p_value == 1.0


def test_bootstrap_detects_shift():
    rng = np.random.default_rng(1)
    b = rng.random(2000)
    a = b - 0.1 + rng.normal(0, 0.05, 2000)
    r = M.paired_bootstrap(a, b, np.arange(2000) // 20)
    assert r.ci_high < 0 and r.p_value < 0.01


def test_cluster_bootstrap_wider_under_within_cluster_correlation():
    rng = np.random.default_rng(2)
    n_cl, per = 40, 50
    effect = np.repeat(rng.normal(0, 1, n_cl), per)  # shared per-cluster component
    d = effect + rng.normal(0, 0.1, n_cl * per)
    clustered = M.paired_bootstrap(d, np.zeros_like(d), np.repeat(np.arange(n_cl), per))
    iid = M.paired_bootstrap(d, np.zeros_like(d), np.arange(d.size))
    assert (clustered.ci_high - clustered.ci_low) > 3 * (iid.ci_high - iid.ci_low)


def test_holm_hand():
    np.testing.assert_allclose(M.holm([0.01, 0.04, 0.03]), [0.03, 0.06, 0.06])
    np.testing.assert_allclose(M.holm([0.5, 0.6]), [1.0, 1.0])


In [ ]:
%%writefile tests/test_no_leakage.py
"""Leakage tests (CLAUDE.md Rules 1 and 2), run for both units "day" and "hour".

``perturb_after`` replaces everything not available at the end of unit T: hourly sales/flags
after T (including later hours of the same day) and the daily fields (weather, discount,
activity) of every day not complete at T. The calendar is untouched (known in advance).
Features at origins <= T, training sets of the fold at T, and (in test_baseline.py) model
predictions must be unchanged.
"""

from __future__ import annotations

from datetime import date, timedelta

import numpy as np
import polars as pl
import pytest
from hypothesis import given, settings
from hypothesis import strategies as st

from sctwin.backtest import BacktestSpec, check_split, make_origins
from sctwin.features import FeatureBuilder, FeatureConfig
from sctwin.panel import HIERARCHY, Panel
from sctwin.timeunit import resample

UNITS = ("day", "hour")
SMALL = {
    "day": FeatureConfig(lags=(0, 1, 6), windows=(3, 7), season=7, n_seasons=3,
                         extra_seasons=(), stockout_windows=(3,), cov_days=3),
    "hour": FeatureConfig(lags=(0, 1, 23), windows=(6, 24, 48), season=24, n_seasons=3,
                          extra_seasons=(168,), stockout_windows=(6, 24), cov_days=3),
}  # fmt: skip


def random_panel(n_series: int, n_days: int, seed: int) -> Panel:
    rng = np.random.default_rng(seed)
    keys = pl.DataFrame({c: np.arange(n_series) + 10 * j for j, c in enumerate(HIERARCHY)})
    shape = (n_series, n_days, 24)
    return Panel(
        keys=keys,
        dates=[date(2024, 1, 1) + timedelta(days=d) for d in range(n_days)],
        sales=np.round(rng.gamma(1.0, 0.3, shape) * (rng.random(shape) < 0.5), 1),
        stock=(rng.random(shape) < 0.3).astype(np.int8),
        discount=rng.choice([0.0, 0.8, 1.0, 1.05], (n_series, n_days)),
        activity=(rng.random((n_series, n_days)) < 0.3).astype(np.int8),
        holiday=(rng.random(n_days) < 0.3).astype(np.int8),
        weather=rng.normal(size=(n_series, n_days, 4)),
    )


def perturb_after(p: Panel, unit: str, origin: int, seed: int = 0) -> Panel:
    rng = np.random.default_rng(seed + 1)
    n_series, n_days = p.shape
    sales, stock = p.sales.copy(), p.stock.copy()
    first_hour = (origin + 1) * 24 if unit == "day" else origin + 1
    last_day = origin if unit == "day" else (origin + 1) // 24 - 1  # last complete day
    fs, fk = sales.reshape(n_series, -1), stock.reshape(n_series, -1)  # views
    fs[:, first_hour:] = rng.gamma(2.0, 5.0, fs[:, first_hour:].shape)
    fk[:, first_hour:] = 1 - fk[:, first_hour:]
    weather, discount, activity = p.weather.copy(), p.discount.copy(), p.activity.copy()
    weather[:, last_day + 1 :] += 100.0
    discount[:, last_day + 1 :] = 0.5
    activity[:, last_day + 1 :] = 1 - activity[:, last_day + 1 :]
    return Panel(p.keys, p.dates, sales, stock, discount, activity, p.holiday.copy(), weather)


def builder(p: Panel, unit: str, cfg: FeatureConfig | None = None) -> FeatureBuilder:
    return FeatureBuilder(resample(p, unit), p.keys, cfg or SMALL[unit])


def _features(df: pl.DataFrame, fb: FeatureBuilder) -> np.ndarray:
    return df.sort(["series_idx", "origin_idx", "step"]).select(fb.feature_names).to_numpy()


def _case(unit: str, n_days: int, frac: float, seed: int) -> tuple[Panel, int]:
    p = random_panel(2, n_days, seed)
    n_units = n_days * (24 if unit == "hour" else 1)
    return p, int(frac * (n_units - 2))


# --------------------------------------------------------------------------- features


@pytest.mark.parametrize("unit", UNITS)
@settings(max_examples=30, deadline=None)
@given(
    n_days=st.integers(2, 20),
    frac=st.floats(0, 1),
    lead=st.integers(0, 3),
    horizon=st.integers(1, 10),
    seed=st.integers(0, 10_000),
)
def test_features_ignore_data_after_origin(unit, n_days, frac, lead, horizon, seed):
    p, origin = _case(unit, n_days, frac, seed)
    q = perturb_after(p, unit, origin, seed)
    origins = np.arange(origin + 1)  # every origin <= T
    fa, fq = builder(p, unit), builder(q, unit)
    a = fa.rows(origins, lead, horizon)
    b = fq.rows(origins, lead, horizon)
    np.testing.assert_array_equal(_features(a, fa), _features(b, fq))


@pytest.mark.parametrize("unit", UNITS)
@settings(max_examples=30, deadline=None)
@given(
    n_days=st.integers(4, 20),
    frac=st.floats(0.3, 1),
    lead=st.integers(0, 3),
    horizon=st.integers(1, 10),
    window=st.sampled_from(["sliding", "expanding"]),
    seed=st.integers(0, 10_000),
)
def test_training_set_ignores_data_after_origin(unit, n_days, frac, lead, horizon, window, seed):
    """Every training feature AND training target of the fold at T is unchanged when all data
    after T is replaced, and every training target lies at or before T."""
    p, origin = _case(unit, n_days, frac, seed)
    spec = BacktestSpec(unit, lead, horizon, (origin,), window=window, window_length=30)
    q = perturb_after(p, unit, origin, seed)
    fa, fq = builder(p, unit), builder(q, unit)
    ta = fa.rows(spec.train_times(origin), lead, horizon)
    tb = fq.rows(spec.train_times(origin), lead, horizon)
    check_split(ta, fa.rows([origin], lead, horizon), origin, spec)
    if ta.height:
        assert int(ta["target_idx"].max()) <= origin
    cols = [*fa.feature_names, "y", "y_stockout"]
    key = ["series_idx", "origin_idx", "step"]
    np.testing.assert_array_equal(ta.sort(key).select(cols).to_numpy(),
                                  tb.sort(key).select(cols).to_numpy())  # fmt: skip


def test_hour_instock_indicator_does_not_peek_at_next_hour():
    """Regression: the hourly in-stock indicator of hour u uses flag u+1. Changing ONLY the
    flag of hour T+1 must not change any feature at origin T."""
    p = random_panel(2, 10, 5)
    for origin in (30, 100, 191):
        q = Panel(p.keys, p.dates, p.sales, p.stock.copy(), p.discount, p.activity, p.holiday,
                  p.weather)  # fmt: skip
        flat = q.stock.reshape(2, -1)
        flat[:, origin + 1] = 1 - flat[:, origin + 1]
        fa, fq = builder(p, "hour"), builder(q, "hour")
        np.testing.assert_array_equal(
            _features(fa.rows([origin], 1, 24), fa), _features(fq.rows([origin], 1, 24), fq)
        )


@pytest.mark.parametrize("unit", UNITS)
def test_perturbation_is_not_vacuous(unit):
    """Changing unit T itself (available at T) must change the features at origin T."""
    p = random_panel(2, 20, 0)
    origin = 15 if unit == "day" else 15 * 24 + 23  # end of day 15 in both cases
    q = perturb_after(p, unit, origin - 1)
    fa, fq = builder(p, unit), builder(q, unit)
    a = fa.rows([origin], 1, 3).select(fa.feature_names).to_numpy()
    b = fq.rows([origin], 1, 3).select(fq.feature_names).to_numpy()
    changed = {
        c for k, c in enumerate(fa.feature_names)
        if not np.array_equal(a[:, k], b[:, k], equal_nan=True)
    }  # fmt: skip
    w = SMALL[unit].stockout_windows[0]
    assert {"sale_lag0", f"stockout_mean{w}", "precpt_lastday"} <= changed


def test_hour_origin_mid_day_uses_previous_complete_day():
    p = random_panel(1, 5, 1)
    fb = builder(p, "hour")
    for origin, expect_day in ((2 * 24 + 10, 1), (2 * 24 + 23, 2), (5, -1)):
        got = fb.rows([origin], 0, 1)["precpt_lastday"][0]
        if expect_day < 0:
            assert np.isnan(got)
        else:
            assert got == p.weather[0, expect_day, 0]


# --------------------------------------------------------------------------- splits


def test_train_times_full_window_rule():
    spec = BacktestSpec("day", lead=1, horizon=7, origins=(74,), window="sliding",
                        window_length=42, min_history=28)  # fmt: skip
    t = spec.train_times(74)
    assert t.max() == 74 - 1 - 7 and t.min() == max(28, 74 - 8 - 42 + 1)
    assert BacktestSpec("day", 1, 7, (90,), window_length=10).train_times(90).tolist() == list(
        range(73, 83)
    )
    exp = BacktestSpec("hour", 1, 24, (500,), window="expanding", min_history=48,
                       train_stride=6)  # fmt: skip
    t = exp.train_times(500)
    assert t.max() == 475 and t.min() >= 48 and (np.diff(t) == 6).all()


def test_check_split_catches_violations():
    p = random_panel(1, 60, 0)
    fb = builder(p, "day")
    spec = BacktestSpec("day", 1, 7, (40,), window="sliding", window_length=20)
    test = fb.rows([40], 1, 7)
    with pytest.raises(AssertionError, match="t > T - L - H"):
        check_split(fb.rows(np.arange(20, 35), 1, 7), test, 40, spec)
    with pytest.raises(AssertionError, match="partial window"):
        check_split(fb.rows(np.arange(20, 33), 1, 7, max_target=38), test, 40, spec)
    with pytest.raises(AssertionError, match="another origin"):
        check_split(fb.rows(np.arange(20, 33), 1, 7), fb.rows([40, 41], 1, 7), 40, spec)
    check_split(fb.rows(spec.train_times(40), 1, 7), test, 40, spec)


def test_make_origins():
    assert make_origins(76, 89, lead=1, horizon=7, step=7) == (74, 81)
    hour = make_origins(76 * 24, 90 * 24 - 1, lead=1, horizon=24, step=24)
    assert hour[0] == 76 * 24 - 2 and len(hour) == 14
    assert make_origins(76 * 24, 90 * 24 - 1, 1, 24, 24, n_folds=3) == hour[:3]


In [ ]:
%%writefile tests/test_repro.py
import json

from sctwin.repro import config_hash, seed_everything, write_run_manifest


def test_config_hash_key_order_independent():
    assert config_hash({"a": 1, "b": {"c": 2, "d": 3}}) == config_hash(
        {"b": {"d": 3, "c": 2}, "a": 1}
    )
    assert config_hash({"a": 1}) != config_hash({"a": 2})


def test_seed_determinism():
    a = seed_everything(5).normal(size=5)
    b = seed_everything(5).normal(size=5)
    assert (a == b).all()


def test_run_manifest(tmp_path):
    p = write_run_manifest(tmp_path, {"seed": 1}, stage="test")
    m = json.loads(p.read_text())
    assert m["stage"] == "test" and m["environment"]["python"]


In [ ]:
%%writefile tests/test_temporal_splits.py
import polars as pl
import pytest

from sctwin.splits import assign_roles, make_manifest, read_manifest, write_manifest

KW = dict(fit_days=14, val_days=6, holdout_store_frac=0.25, holdout_product_frac=0.2, seed=7)


@pytest.fixture
def manifest(train_test):
    return make_manifest(train_test[0], **KW)


def test_fit_val_partition_dates(train_test, manifest):
    tr = assign_roles(train_test[0], manifest)
    fit = tr.filter(pl.col("role") == "fit")["dt"].str.to_date()
    val = tr.filter(pl.col("role") == "val")["dt"].str.to_date()
    assert fit.max() < val.min()
    assert fit.n_unique() == 14 and val.n_unique() == 6
    assert tr.height == train_test[0].height


def test_eval_strictly_after_val(train_test, manifest):
    te = train_test[1]
    assert (
        te["dt"].str.to_date().min()
        > pl.Series([manifest["temporal"]["val"]["end"]]).str.to_date()[0]
    )


def test_wrong_day_count_rejected(train_test):
    with pytest.raises(ValueError, match="distinct dates"):
        make_manifest(train_test[0], **{**KW, "fit_days": 10})


def test_group_holdout_rule(train_test, manifest):
    tr = assign_roles(train_test[0], manifest)
    g = manifest["group_holdout"]
    assert len(g["stores"]) == 1 and len(g["products"]) == 1
    ho = tr.filter(pl.col("holdout"))
    assert (ho["store_id"].is_in(g["stores"]) | ho["product_id"].is_in(g["products"])).all()
    ins = tr.filter(~pl.col("holdout"))
    assert not ins["store_id"].is_in(g["stores"]).any()
    assert not ins["product_id"].is_in(g["products"]).any()
    c = manifest["series_counts"]
    assert (
        c["in_sample"] + c["holdout_store_only"] + c["holdout_product_only"] + c["holdout_both"]
        == c["total"]
    )


def test_manifest_deterministic_and_order_independent(train_test):
    a = make_manifest(train_test[0], **KW)
    b = make_manifest(train_test[0].sample(fraction=1.0, shuffle=True, seed=3), **KW)
    assert a == b


def test_manifest_write_once(tmp_path, manifest):
    p = tmp_path / "m.json"
    write_manifest(manifest, p)
    write_manifest(manifest, p)  # identical: no-op
    changed = {**manifest, "seed": 999}
    with pytest.raises(FileExistsError):
        write_manifest(changed, p)
    assert read_manifest(p) == manifest


def test_manifest_tamper_detected(tmp_path, manifest):
    p = tmp_path / "m.json"
    write_manifest(manifest, p)
    p.write_text(p.read_text().replace('"seed": 7', '"seed": 8'))
    with pytest.raises(ValueError, match="digest"):
        read_manifest(p)


In [ ]:
%%writefile tests/test_baseline.py
"""Baselines: known-answer checks, determinism, and end-to-end leakage safety (both units)."""

import numpy as np
import pytest
from test_no_leakage import builder, perturb_after, random_panel

from sctwin.backtest import BacktestSpec
from sctwin.baseline import TAUS, GLMDesign, tweedie_quantiles, tweedie_sample
from sctwin.baseline_runner import add_groups, run_backtest
from sctwin.evaluation import evaluate, qcol

CFG = {
    "taus": list(TAUS),
    "seed": 1,
    "glm": {"power": 1.5, "alpha": 1e-3, "n_samples": 100},
    "lgbm": {"num_rounds": 10, "num_threads": 2, "params": {"min_data_in_leaf": 5}},
}
SPECS = {
    "day": dict(lead=1, horizon=7, origins=(44, 51), window="sliding", window_length=20,
                min_history=7),
    "hour": dict(lead=1, horizon=24, origins=(20 * 24 + 22, 21 * 24 + 22), window="sliding",
                 window_length=72, min_history=48, train_stride=3),
}  # fmt: skip
QUIET = {"log": lambda *_: None}


def run(p, unit, origins=None):
    kw = dict(SPECS[unit])
    if origins is not None:
        kw["origins"] = origins
    spec = BacktestSpec(unit, **kw)
    fb = builder(p, unit)
    s = np.arange(p.shape[0])
    pred, log = run_backtest(fb, spec, s, s, CFG, **QUIET)
    return fb, spec, pred, log


def test_tweedie_sample_moments():
    rng = np.random.default_rng(0)
    mu, phi, power = np.array([0.3, 2.0]), 1.2, 1.5
    s = tweedie_sample(mu, phi, power, 400_000, rng)
    np.testing.assert_allclose(s.mean(1), mu, rtol=0.02)
    np.testing.assert_allclose(s.var(1), phi * mu**power, rtol=0.03)
    assert (s >= 0).all() and (s[0] == 0).mean() > 0.2  # point mass at zero


def test_tweedie_quantiles_ordered():
    q = tweedie_quantiles(np.array([0.1, 1.0, 5.0]), 1.0, 1.5, TAUS, n=500)
    assert (np.diff(q, axis=1) >= 0).all()


def test_glm_design_fitted_on_train_only():
    import polars as pl

    tr = pl.DataFrame({"a": [1.0, 2.0, 3.0], "c": [0, 1, 1]})
    te = pl.DataFrame({"a": [100.0, float("nan")], "c": [1, 0]})
    d = GLMDesign(numeric=["a"], categorical=["c"], log_cols=[]).fit(tr)
    x = d.transform(te)
    assert d.means["a"] == 2.0  # test values never enter the statistics
    np.testing.assert_allclose(x[1], [0.0, 1.0, 0.0])  # imputed with train mean -> 0, flag 1


@pytest.fixture(scope="module", params=["day", "hour"])
def result(request):
    unit = request.param
    n_days = 60 if unit == "day" else 25
    p = random_panel(4, n_days, 7)
    return (unit, p, *run(p, unit))


def test_runner_output_and_protocol(result):
    unit, p, fb, spec, pred, log = result
    assert pred.height == len(spec.origins) * 4 * spec.horizon
    assert set(pred["origin_idx"].unique().to_list()) == set(spec.origins)
    lag = pred["target_idx"] - pred["origin_idx"]
    assert lag.min() == spec.lead + 1 and lag.max() == spec.lead + spec.horizon
    for m in ("clim", "glm", "lgbm_q"):
        q = pred.select([qcol(m, t) for t in TAUS]).to_numpy()
        ok = ~np.isnan(q).any(axis=1)
        assert (np.diff(q[ok], axis=1) >= -1e-12).all(), m
    for r in log:
        assert r["train_rows"] > 0
        assert r["train_t_range"][1] <= r["origin"] - spec.lead - spec.horizon
    res = evaluate(add_groups(pred, fb, {0}), TAUS)
    assert {"snaive", "clim", "glm", "lgbm", "lgbm_q"} <= set(res["overall"]["models"])
    assert set(res["folds"]) == {str(o) for o in spec.origins}


def test_runner_is_deterministic(result):
    unit, p, _, _, pred, _ = result
    assert pred.equals(run(p, unit)[2])


def test_predictions_ignore_data_after_origin(result):
    """End-to-end: replacing all data after the fold origin T leaves every model's prediction
    unchanged (features, training set, fitted statistics and models all as of T)."""
    unit, p, _, spec, _, _ = result
    o = spec.origins[0]
    a = run(p, unit, origins=(o,))[2]
    b = run(perturb_after(p, unit, o, seed=3), unit, origins=(o,))[2]
    cols = [c for c in a.columns if "__" in c]
    np.testing.assert_array_equal(a.select(cols).to_numpy(), b.select(cols).to_numpy())
    assert not a["y"].equals(b["y"])  # the outcomes did change: the check is not vacuous


In [ ]:
%%writefile tests/test_censoring.py
"""Step 4 censoring benchmark: injection invariants, point-in-time history, reconstruction
baselines, calibration and the fully synthetic generator."""

from __future__ import annotations

import dataclasses

import numpy as np
import pytest
from hypothesis import given, settings
from hypothesis import strategies as st

from sctwin.censoring import (
    HI,
    IMPUTERS,
    LO,
    CensorConfig,
    SyntheticParams,
    build_benchmark,
    calibrate,
    candidate_days,
    censor_synthetic,
    expected_window_demand,
    hour_profile,
    inject_stockouts,
    position_ratios,
    recovery_frames,
    simulate_latent,
    stock_levels,
    stockout_stats,
    trailing_hour_means,
)
from sctwin.evaluation import evaluate

NO_NOISE = CensorConfig(leak_first=0.0, leak_later=0.0, p_restock=0.0)


def demand_rows(n: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    d = rng.gamma(1.0, 1.0, (n, 24)) * (rng.random((n, 24)) < 0.7)
    d[:, :LO] *= 0.1
    return d


# --------------------------------------------------------------------------- injection


@settings(max_examples=60, deadline=None)
@given(
    seed=st.integers(0, 10_000),
    leak_first=st.floats(0, 1),
    leak_later=st.floats(0, 1),
    p_restock=st.floats(0, 1),
    scale=st.floats(0.05, 3.0),
)
def test_injection_keeps_truth_and_bounds(seed, leak_first, leak_later, p_restock, scale):
    d = demand_rows(50, seed)
    d0 = d.copy()
    cfg = CensorConfig(leak_first=leak_first, leak_later=leak_later, p_restock=p_restock)
    k = scale * d[:, LO:HI].sum(1) + 1e-3
    inj = inject_stockouts(d, k, np.random.default_rng(seed), cfg)
    np.testing.assert_array_equal(d, d0)  # the truth is never modified
    assert (inj.observed <= d + 1e-12).all() and (inj.observed >= 0).all()
    np.testing.assert_array_equal(inj.observed[~inj.censored], d[~inj.censored])
    assert not inj.flag[:, :LO].any() and not inj.flag[:, HI:].any()
    assert not inj.censored[:, :LO].any() and not inj.censored[:, HI:].any()
    assert ((inj.flag == 1) <= inj.censored).all()  # flagged hours are censored


def test_depletion_follows_cumulative_demand():
    d = demand_rows(500, 1)
    rng = np.random.default_rng(1)
    k = rng.uniform(0.1, 1.2, 500) * d[:, LO:HI].sum(1) + 1e-6
    inj = inject_stockouts(d, k, np.random.default_rng(2), NO_NOISE)
    cum = np.cumsum(d[:, LO:HI], 1)
    for i in range(500):
        h = inj.depletion[i]
        if h < 0:
            assert cum[i, -1] < k[i] and not inj.flag[i].any()
            continue
        j = h - LO
        assert cum[i, j] >= k[i] and (j == 0 or cum[i, j - 1] < k[i])
        # everything sold up to and including the depletion hour is exactly the stock
        assert inj.observed[i, LO : h + 1].sum() == pytest.approx(k[i])
        # no restock: flagged from h + 1 to the end of the window, nothing sold
        assert inj.flag[i, h + 1 : HI].all() and not inj.flag[i, LO : h + 1].any()
        assert (inj.observed[i, h + 1 : HI] == 0).all()


def test_leak_and_restock():
    d = np.zeros((1, 24))
    d[0, LO:HI] = 1.0
    cfg = CensorConfig(leak_first=0.5, leak_later=0.25, p_restock=1.0, restock_mean_hours=1.0)
    inj = inject_stockouts(d, np.array([3.5]), np.random.default_rng(0), cfg)
    # cumulative demand reaches 3.5 in the 4th window hour; restock after exactly 1 hour
    h = LO + 3
    assert inj.depletion[0] == h and inj.restock[0] == h + 2
    np.testing.assert_allclose(inj.observed[0, h], 0.5)  # 3.5 - 3 units sold before
    assert inj.flag[0, h + 1] == 1 and inj.observed[0, h + 1] == 0.5  # first flagged hour leaks
    assert inj.flag[0, h + 2 :].sum() == 0 and (inj.observed[0, h + 2 : HI] == 1.0).all()


def test_common_random_numbers_and_determinism():
    d = demand_rows(200, 3)
    e = d[:, LO:HI].sum(1)
    a, b = CensorConfig(coverage=0.8), CensorConfig(coverage=1.2)
    ka = stock_levels(e, np.random.default_rng(5), a)
    kb = stock_levels(e, np.random.default_rng(5), b)
    np.testing.assert_allclose(ka / kb, 0.8 / 1.2)  # same draws, different coverage
    i1 = inject_stockouts(d, ka, np.random.default_rng(5), a)
    i2 = inject_stockouts(d, ka, np.random.default_rng(5), a)
    np.testing.assert_array_equal(i1.observed, i2.observed)
    # more stock never creates more stockouts
    assert (
        inject_stockouts(d, kb, np.random.default_rng(5), b).stocked_out <= i1.stocked_out
    ).all()


def test_config_validation():
    with pytest.raises(ValueError):
        CensorConfig(leak_first=1.5)
    with pytest.raises(ValueError):
        inject_stockouts(np.ones((2, 24)), np.array([1.0, 0.0]), np.random.default_rng(0),
                         CensorConfig())  # fmt: skip


# --------------------------------------------------------------------------- point-in-time history


def panel_arrays(n_series: int, n_days: int, seed: int) -> tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    sales = rng.gamma(1.0, 1.0, (n_series, n_days, 24))
    stock = (rng.random((n_series, n_days, 24)) < 0.2).astype(np.int8)
    return sales, stock


def test_trailing_hour_means_known_value():
    sales, stock = panel_arrays(2, 10, 0)
    s, d, w = np.array([1]), np.array([7]), 3
    got = trailing_hour_means(sales, stock, s, d, w)[0]
    x, ok = sales[1, 4:7], stock[1, 4:7] == 0
    with np.errstate(invalid="ignore"):
        want = np.where(ok.sum(0) > 0, (x * ok).sum(0) / ok.sum(0), np.nan)
    np.testing.assert_allclose(got, want, equal_nan=True)


@settings(max_examples=30, deadline=None)
@given(seed=st.integers(0, 10_000), day=st.integers(1, 19), window=st.integers(1, 10))
def test_history_uses_only_days_before_the_censored_day(seed, day, window):
    """Origin T = end of the censored day d. The methods' history must not change when day d
    itself or any later day is replaced (day d enters only through its censored observation)."""
    sales, stock = panel_arrays(3, 20, seed)
    s, d = np.arange(3), np.full(3, day)
    a = trailing_hour_means(sales, stock, s, d, window)
    rng = np.random.default_rng(seed + 1)
    sales2, stock2 = sales.copy(), stock.copy()
    sales2[:, day:] = rng.gamma(3.0, 5.0, sales2[:, day:].shape)
    stock2[:, day:] = 1 - stock2[:, day:]
    np.testing.assert_array_equal(a, trailing_hour_means(sales2, stock2, s, d, window))


def test_reconstruction_ignores_data_after_the_day():
    """End-to-end: replacing all days after d (and the demand of censored hours beyond what is
    observed) leaves every method's reconstruction of day d unchanged."""
    sales, stock = panel_arrays(6, 30, 2)
    stock[:, 20:] = 0
    rows = candidate_days(sales, stock, np.arange(6), np.array([20]), window=14)
    bm = build_benchmark(sales, rows, CensorConfig(), seed=4)
    sales2 = sales.copy()
    sales2[:, 21:] *= 10.0
    stock2 = stock.copy()
    stock2[:, 21:] = 1
    bm2 = build_benchmark(sales2, candidate_days(sales2, stock2, np.arange(6), np.array([20]),
                                                 window=14), CensorConfig(), seed=4)  # fmt: skip
    for name, f in IMPUTERS.items():
        a = f(bm.injection.observed, bm.injection.flag, bm.hour_means)
        b = f(bm2.injection.observed, bm2.injection.flag, bm2.hour_means)
        np.testing.assert_array_equal(a, b, err_msg=name)


# --------------------------------------------------------------------------- reconstruction


def test_imputers_leave_uncensored_days_alone():
    d = demand_rows(20, 5)
    flag = np.zeros((20, 24), dtype=np.int8)
    hm = np.full((20, 24), 0.7)
    for name, f in IMPUTERS.items():
        np.testing.assert_array_equal(f(d, flag, hm), d, err_msg=name)


def test_profile_recovers_exact_profile_demand():
    """Demand that is exactly level x profile is recovered by profile-scaling; ignoring the
    censoring under-estimates it."""
    prof = hour_profile()
    n = 50
    level = np.random.default_rng(0).uniform(0.5, 3.0, n)
    demand = level[:, None] * prof[None, :]
    k = 0.6 * demand[:, LO:HI].sum(1)
    inj = inject_stockouts(demand, k, np.random.default_rng(1), NO_NOISE)
    hm = np.tile(prof, (n, 1))  # history has the same shape (any scale)
    est = IMPUTERS["profile_pre"](inj.observed, inj.flag, hm)
    np.testing.assert_allclose(est, demand, rtol=1e-12)
    obs = IMPUTERS["observed"](inj.observed, inj.flag, hm)
    assert obs[:, LO:HI].sum() < 0.7 * demand[:, LO:HI].sum()
    # without treating the depletion hour as censored, its lower bound biases the level down
    plain = IMPUTERS["profile"](inj.observed, inj.flag, hm)
    assert plain[:, LO:HI].sum() < est[:, LO:HI].sum()


def test_hour_mean_imputer_never_below_observed():
    d = demand_rows(100, 7)
    inj = inject_stockouts(d, 0.5 * d[:, LO:HI].sum(1) + 1e-6, np.random.default_rng(0),
                           CensorConfig())  # fmt: skip
    hm = np.full((100, 24), 0.3)
    for f in IMPUTERS.values():
        assert (f(inj.observed, inj.flag, hm) >= inj.observed).all()


# --------------------------------------------------------------------------- diagnostics


def test_stockout_stats_known_example():
    f = np.zeros((4, 24), dtype=np.int8)
    f[0, 10:HI] = 1  # first flag 10, no restock
    f[1, 14:16] = 1  # first flag 14, restock after 2 hours
    f[2, LO] = 1  # out at hour 6: excluded (not in stock at the start)
    st_ = stockout_stats(f)
    assert st_["n_days"] == 3 and st_["stockout_rate"] == pytest.approx(2 / 3)
    assert st_["first_hour_q50"] == 12 and st_["restock_rate"] == 0.5
    assert st_["restock_mean_hours"] == 2


def test_position_ratios_known_example():
    obs = np.ones((1, 24))
    obs[0, 11] = 3.0
    flag = np.zeros((1, 24), dtype=np.int8)
    flag[0, 12:HI] = 1
    obs[0, 12:HI] = 0.5
    r = position_ratios(obs, flag, np.ones((1, 24)), (-2, -1, 0, 1))
    assert r == {"-2": 1.0, "-1": 3.0, "+0": 0.5, "+1": 0.5}


def test_calibration_recovers_known_coverage():
    d = demand_rows(3000, 11)
    e = d[:, LO:HI].sum(1)
    true = CensorConfig(coverage=0.9, coverage_sd=0.3)
    rng = np.random.default_rng(9)
    target = stockout_stats(inject_stockouts(d, stock_levels(e, rng, true), rng, true).flag)
    cfg, grid = calibrate(d, e, target, CensorConfig(), np.round(np.arange(0.5, 1.31, 0.1), 2),
                          np.array([0.1, 0.3, 0.5]), seed=9)  # fmt: skip
    assert len(grid) == 27
    assert cfg.coverage == pytest.approx(0.9) and cfg.coverage_sd == pytest.approx(0.3)


def test_recovery_frames_and_evaluation():
    sales, stock = panel_arrays(8, 40, 3)
    stock[:, 30:] = 0
    rows = candidate_days(sales, stock, np.arange(8), np.arange(30, 40), window=14)
    bm = build_benchmark(sales, rows, CensorConfig(coverage=0.7), seed=0)
    est = {m: f(bm.injection.observed, bm.injection.flag, bm.hour_means)
           for m, f in IMPUTERS.items()}  # fmt: skip
    hourly, daily = recovery_frames(bm, est)
    assert hourly.height == int(bm.injection.censored.sum())
    assert daily.height == int(bm.injection.stocked_out.sum()) > 0
    np.testing.assert_allclose(hourly["observed__point"], hourly["observed"])
    res = evaluate(daily, (0.5,))
    assert res["overall"]["models"]["observed"]["bias"] < 0  # ignoring censoring under-states
    assert expected_window_demand(bm.hour_means).shape == (bm.series.size,)


# --------------------------------------------------------------------------- synthetic generator


def test_synthetic_generator_moments():
    p = SyntheticParams(level_mean=0.0, level_sd=0.3, rho=0.5, sigma_z=0.3)
    sp = simulate_latent(400, 120, p, seed=0)
    assert sp.demand.shape == sp.mu.shape == (400, 120, 24)
    assert sp.demand.mean() == pytest.approx(sp.mu.mean(), rel=0.03)
    assert sp.z.var() == pytest.approx(p.sigma_z**2 / (1 - p.rho**2), rel=0.1)
    lag1 = np.corrcoef(sp.z[:, 1:].ravel(), sp.z[:, :-1].ravel())[0, 1]
    assert lag1 == pytest.approx(p.rho, abs=0.05)
    assert hour_profile()[LO:HI].mean() == pytest.approx(1.0)
    np.testing.assert_array_equal(simulate_latent(5, 10, p, 3).demand,
                                  simulate_latent(5, 10, p, 3).demand)  # fmt: skip


def test_censor_synthetic_bounds():
    sp = simulate_latent(50, 30, SyntheticParams(), seed=1)
    inj = censor_synthetic(sp, CensorConfig(coverage=0.9), seed=2)
    assert inj.observed.shape == sp.demand.shape
    assert (inj.observed <= sp.demand + 1e-12).all()
    assert 0.1 < inj.stocked_out.mean() < 0.9


def test_leak_calibration_matches_target_ratios():
    from sctwin.censoring import calibrate_leak

    d = demand_rows(3000, 13)
    e = d[:, LO:HI].sum(1)
    base = np.tile(d.mean(0), (3000, 1))
    true = CensorConfig(coverage=0.8, leak_first=0.1, leak_later=0.03)
    rng = np.random.default_rng(4)
    inj = inject_stockouts(d, stock_levels(e, rng, true), rng, true)
    target = position_ratios(inj.observed, inj.flag, base, (0, 1))
    start = dataclasses.replace(true, leak_first=0.4, leak_later=0.2)
    got = calibrate_leak(d, e, base, target, start, seed=4)
    assert got.leak_first == pytest.approx(0.1) and got.leak_later == pytest.approx(0.03)


def test_restock_calibration_recovers_truncated_parameters():
    from sctwin.censoring import calibrate_restock

    d = demand_rows(4000, 17)
    e = d[:, LO:HI].sum(1)
    true = CensorConfig(coverage=0.6, p_restock=0.3, restock_mean_hours=4.0)
    rng = np.random.default_rng(8)
    target = stockout_stats(inject_stockouts(d, stock_levels(e, rng, true), rng, true).flag)
    assert target["restock_mean_hours"] < 4.0  # truncation by the end of the window
    got = calibrate_restock(d, e, target, dataclasses.replace(true, p_restock=0.1),
                            np.array([0.1, 0.2, 0.3, 0.4]), np.array([2.0, 3.0, 4.0, 5.0]),
                            seed=8)  # fmt: skip
    assert got.p_restock == pytest.approx(0.3) and got.restock_mean_hours == pytest.approx(4.0)


In [ ]:
%%writefile tests/test_demand_model.py
"""Step 5 demand model: likelihood, samplers, AR(1), data, reconstruction (point-in-time)."""

from __future__ import annotations

import numpy as np
import pytest
from scipy import stats

from sctwin import demand_model as dm
from sctwin.panel import CLS_A, CLS_P_FIRST, CLS_P_PRE, CLS_U

PAR = {"alpha": -1.0, "sigma_y": 0.7, "k0": 0.3, "k1": 0.9}


def _p_pos(eta: np.ndarray) -> np.ndarray:
    return 1 / (1 + np.exp(-(PAR["k0"] + PAR["k1"] * (eta - PAR["alpha"]))))


def _ll(y, code, eta):
    return np.asarray(dm.loglik(np.asarray(y), np.asarray(code), np.asarray(eta), PAR["alpha"],
                                PAR["sigma_y"], PAR["k0"], PAR["k1"]))  # fmt: skip


def test_loglik_matches_scipy():
    eta = np.array([-2.0, -0.5, 0.3, 0.3, 1.0])
    y = np.array([0.0, 0.4, 1.7, 0.0, 3.0])
    p = _p_pos(eta)
    m = np.asarray(dm.hurdle_terms(eta, PAR["alpha"], PAR["sigma_y"], PAR["k0"], PAR["k1"],
                                   xp=np)[2])  # fmt: skip
    ln = stats.lognorm(s=PAR["sigma_y"], scale=np.exp(m))
    exact = np.where(y > 0, np.log(p) + ln.logpdf(np.where(y > 0, y, 1.0)), np.log1p(-p))
    np.testing.assert_allclose(_ll(y, np.zeros(5, int), eta), exact, rtol=1e-10)
    cens = np.where(y > 0, np.log(p) + ln.logsf(np.where(y > 0, y, 1.0)), 0.0)
    np.testing.assert_allclose(_ll(y, np.ones(5, int), eta), cens, rtol=1e-10)
    np.testing.assert_array_equal(_ll(y, np.full(5, 2), eta), 0.0)


def test_mean_is_exp_eta():
    rng = np.random.default_rng(0)
    eta = np.full(400_000, 0.2)
    d = dm.sample_hurdle(eta, PAR["alpha"], PAR["sigma_y"], PAR["k0"], PAR["k1"], rng)
    assert d.mean() == pytest.approx(np.exp(0.2), rel=0.01)
    assert (d == 0).mean() == pytest.approx(1 - _p_pos(np.array([0.2]))[0], abs=0.005)


def test_truncated_sampler_matches_conditional_distribution():
    rng = np.random.default_rng(1)
    eta, lower = np.full(50_000, -0.5), np.full(50_000, 1.5)  # far in the right tail
    x = dm.sample_truncated(eta, lower, PAR["alpha"], PAR["sigma_y"], PAR["k0"], PAR["k1"], rng)
    assert (x >= 1.5).all()
    m = float(dm.hurdle_terms(-0.5, PAR["alpha"], PAR["sigma_y"], PAR["k0"], PAR["k1"], xp=np)[2])
    ln = stats.lognorm(s=PAR["sigma_y"], scale=np.exp(m))
    cdf = lambda v: (ln.cdf(v) - ln.cdf(1.5)) / ln.sf(1.5)  # noqa: E731
    assert stats.kstest(x, cdf).pvalue > 0.01


def test_ladder_treatments():
    m0, m1, m2 = (dm.LADDER[k].codes() for k in ("M0", "M1", "M2"))
    assert m0[CLS_A] == m0[CLS_P_PRE] == dm.EXACT and m0[CLS_P_FIRST] == m0[CLS_U] == dm.DROP
    assert m1[CLS_P_PRE] == m1[CLS_P_FIRST] == dm.CENSORED and m1[CLS_U] == dm.DROP
    assert (m1 == m2).all() and dm.LADDER["M2"].dynamic and not dm.LADDER["M1"].dynamic
    m1d, m2d = dm.LADDER["M1d"].codes(), dm.LADDER["M2d"].codes()
    assert m1d[CLS_P_PRE] == dm.CENSORED and m1d[CLS_P_FIRST] == m1d[CLS_U] == dm.DROP
    assert (m1d == m2d).all() and dm.LADDER["M2d"].dynamic


def test_ar1_matches_loop():
    rng = np.random.default_rng(2)
    raw = rng.standard_normal((3, 9))
    got = np.asarray(dm.ar1(raw, 0.6, 0.4))
    want = np.empty_like(raw)
    want[:, 0] = raw[:, 0] * 0.4 / np.sqrt(1 - 0.36)
    for t in range(1, 9):
        want[:, t] = 0.6 * want[:, t - 1] + 0.4 * raw[:, t]
    np.testing.assert_allclose(got, want, rtol=1e-12)


def test_make_model_data():
    rng = np.random.default_rng(3)
    sales = rng.gamma(1, 1, (4, 10, 24))
    stock = np.zeros((4, 10, 24), np.int8)
    stock[1, 2, 15:] = 1
    disc = np.full((4, 10), 0.9)
    disc[0, 0] = -1.0  # anomaly -> 0
    data = dm.make_model_data(sales, stock, disc, np.zeros((4, 10)), np.arange(10) % 7,
                              np.array([0, 1]), np.arange(0, 5))  # fmt: skip
    assert data.y.shape == (2, 5, dm.NH) and data.x_disc[0, 0] == 0.0
    assert data.x_disc[1, 1] == pytest.approx(0.1)
    assert data.cls[1, 2, 15 - dm.LO - 1] == CLS_P_PRE and data.cls[1, 2, 15 - dm.LO] == CLS_P_FIRST
    with pytest.raises(ValueError):
        dm.make_model_data(sales, stock, disc, disc, np.arange(10), np.array([0]), np.array([1, 3]))


# --------------------------------------------------------------------------- reconstruction


def fake_fit(spec: dm.ModelSpec, n_s: int, n_d: int, n_draws: int = 8) -> dm.FitResult:
    """A FitResult whose draws are all equal to known parameters (no MCMC needed)."""
    one = {
        "alpha": -1.0, "a": np.linspace(-0.3, 0.3, n_s), "gamma": np.zeros(dm.NH),
        "g": np.zeros((n_s, dm.NH)), "dow_eff": np.zeros(7), "b_disc": 0.0, "b_act": 0.0,
        "sigma_u": 0.3, "u": np.zeros((n_s, n_d)), "rho": 0.5, **PAR,
    }  # fmt: skip
    samples = {
        k: np.repeat(np.asarray(v, dtype=float)[None], n_draws, axis=0) for k, v in one.items()
    }
    return dm.FitResult(spec, samples, {})


def fitted_data(n_s: int, n_d: int) -> dm.ModelData:
    z = np.zeros((n_s, n_d))
    return dm.ModelData(np.zeros((n_s, n_d, dm.NH)), np.zeros((n_s, n_d, dm.NH), np.int8),
                        np.arange(n_d) % 7, z, z, np.arange(n_s), np.arange(n_d))  # fmt: skip


def rows_for(y: np.ndarray, cls: np.ndarray, day: int) -> dm.DayRows:
    r = y.shape[0]
    return dm.DayRows(np.arange(r) % 2, np.full(r, day), y, cls, np.zeros(r, int),
                      np.zeros(r), np.zeros(r))  # fmt: skip


@pytest.mark.parametrize("model", ["M0", "M1", "M2"])
def test_reconstruction_respects_observations(model):
    spec = dm.LADDER[model]
    res, data = fake_fit(spec, 2, 10), fitted_data(2, 10)
    rng = np.random.default_rng(4)
    y = np.round(rng.gamma(1, 0.5, (6, dm.NH)), 1)
    cls = np.full((6, dm.NH), CLS_A, np.int8)
    cls[:, 9] = CLS_P_PRE
    cls[:, 10] = CLS_P_FIRST
    cls[:, 11:] = CLS_U
    out = dm.reconstruct(res, data, rows_for(y, cls, 12), n_draws=8, seed=1)
    assert out.shape == (6, 8, dm.NH)
    np.testing.assert_array_equal(out[:, :, :9], np.repeat(y[:, None, :9], 8, axis=1))
    assert (out >= y[:, None, :] - 1e-12).all()  # never below what was sold
    if model == "M0":  # depletion hour taken at face value
        np.testing.assert_array_equal(out[:, :, 9], np.repeat(y[:, None, 9], 8, axis=1))


def test_reconstruction_is_point_in_time():
    res, data = fake_fit(dm.LADDER["M1"], 2, 10), fitted_data(2, 10)
    y = np.ones((2, dm.NH))
    cls = np.zeros((2, dm.NH), np.int8)
    with pytest.raises(ValueError, match="after every day"):
        dm.reconstruct(res, data, rows_for(y, cls, 9))
    # same observations -> same reconstruction, whatever the (unseen) demand was
    a = dm.reconstruct(res, data, rows_for(y, cls, 10), n_draws=4, seed=2)
    b = dm.reconstruct(res, data, rows_for(y.copy(), cls.copy(), 10), n_draws=4, seed=2)
    np.testing.assert_array_equal(a, b)


def test_reconstruction_without_information_follows_the_prior():
    """All hours dropped: the day effect is drawn from its prior, so the mean demand per hour
    is E[exp(eta + u)] = exp(eta + sigma_u^2 / 2)."""
    res, data = fake_fit(dm.LADDER["M1"], 2, 10, n_draws=100), fitted_data(2, 10)
    y = np.zeros((400, dm.NH))
    cls = np.full((400, dm.NH), CLS_U, np.int8)
    out = dm.reconstruct(res, data, rows_for(y, cls, 10), n_draws=100, seed=3, chunk=400)
    eta = -1.0 + np.where(np.arange(400) % 2 == 0, -0.3, 0.3)
    want = np.mean(np.exp(eta + 0.3**2 / 2))
    assert out.mean() == pytest.approx(want, rel=0.04)


def test_censoring_raises_the_reconstruction():
    """A high sale in the depletion hour signals a busy day: M1 (censored) reconstructs more
    demand than M0 (which takes that hour at face value and ignores the lower bound)."""
    y = np.full((200, dm.NH), 0.3)
    y[:, 9] = 1.2
    cls = np.full((200, dm.NH), CLS_A, np.int8)
    cls[:, 9], cls[:, 10], cls[:, 11:] = CLS_P_PRE, CLS_P_FIRST, CLS_U
    y[:, 10:] = 0.0
    data = fitted_data(2, 10)
    tot = {
        m: dm.reconstruct(fake_fit(dm.LADDER[m], 2, 10, 50), data, rows_for(y, cls, 10),
                          n_draws=50, seed=5).sum(-1).mean()
        for m in ("M0", "M1")
    }  # fmt: skip
    assert tot["M1"] > tot["M0"]


# --------------------------------------------------------------------------- tiny fit


def test_tiny_fit_runs_and_reports_diagnostics():
    rng = np.random.default_rng(6)
    n_s, n_d = 3, 10
    shp = (n_s, n_d, dm.NH)
    data = dm.ModelData(
        y=np.round(rng.gamma(1.0, 0.3, shp) * (rng.random(shp) < 0.6), 1),
        cls=np.zeros((n_s, n_d, dm.NH), np.int8), dow=np.arange(n_d) % 7,
        x_disc=np.zeros((n_s, n_d)), x_act=np.zeros((n_s, n_d)),
        series=np.arange(n_s), days=np.arange(n_d),
    )  # fmt: skip
    res = dm.fit(data, dm.LADDER["M2"], num_warmup=40, num_samples=20, num_chains=1, seed=0)
    d = res.diagnostics
    assert {"divergences", "max_rhat", "min_ess", "runtime_s", "scalars"} <= set(d)
    assert res.samples["u"].shape == (20, n_s, n_d) and np.isfinite(res.samples["alpha"]).all()
    ll = dm.pointwise_loglik(res, data, max_draws=5)
    assert ll.shape[0] == 5 and np.isfinite(ll).all()
    ppc = dm.predictive_checks(res, data, n_draws=5)
    assert 0 <= ppc["zero_frac"]["ppc_p"] <= 1


In [ ]:
%%writefile tests/test_real_data.py
"""Checks against the downloaded dataset. Run with `pytest -m data`; skipped when absent."""

import json
from pathlib import Path

import numpy as np
import polars as pl
import pytest

from sctwin.data import verify_checksums
from sctwin.splits import assign_roles, read_manifest

RAW = Path("data/raw")
REPORT = Path("data/audit/audit_report.json")
MANIFEST = Path("configs/manifests/splits_v1.json")

pytestmark = [
    pytest.mark.data,
    pytest.mark.skipif(not (RAW / "provenance.json").exists(), reason="dataset not downloaded"),
]


def test_checksums_unchanged():
    assert all(verify_checksums(RAW).values())


def test_audit_gates_pass():
    rep = json.loads(REPORT.read_text())
    assert rep["all_gates_pass"], {k: v for k, v in rep["gates"].items() if not v}


def test_manifest_matches_pinned_data():
    m = read_manifest(MANIFEST)
    prov = json.loads((RAW / "provenance.json").read_text())
    assert m["source"]["revision"] == prov["revision"]
    assert m["source"]["train_sha256"] == prov["files"]["train"]["sha256"]


def test_dev_subset_is_in_sample_and_pre_test():
    m = read_manifest(MANIFEST)
    sub = pl.read_parquet("data/subsets/dev20.parquet")
    assert not assign_roles(sub, m)["holdout"].any()
    assert sub["dt"].max() <= m["temporal"]["val"]["end"]


@pytest.mark.parametrize("unit", ["day", "hour"])
def test_real_features_and_training_sets_ignore_data_after_origin(unit):
    """Rules 1 and 2 on the real 20-series development subset: features at every origin <= T
    and the training set (features and targets) of the fold at T are unchanged when all data
    after T is replaced."""
    from test_no_leakage import perturb_after

    from sctwin.backtest import BacktestSpec, check_split
    from sctwin.features import FeatureBuilder
    from sctwin.panel import build_panel
    from sctwin.timeunit import resample

    sub = pl.read_parquet("data/subsets/dev20.parquet").drop(["role", "holdout"])
    p = build_panel(sub)
    assert p.shape == (20, 90)
    upd = 24 if unit == "hour" else 1
    lead, horizon = (1, 7) if unit == "day" else (1, 24)
    for day in (30, 60, 80):
        origin = day * upd + (upd - 3 if unit == "hour" else 0)  # hour: 21:00, mid-evening
        q = perturb_after(p, unit, origin, seed=day)
        fa = FeatureBuilder(resample(p, unit), p.keys)
        fq = FeatureBuilder(resample(q, unit), q.keys)
        cols = list(fa.feature_names)
        origins = np.arange(max(0, origin - 3 * upd), origin + 1)
        a = fa.rows(origins, lead, horizon).select(cols).to_numpy()
        b = fq.rows(origins, lead, horizon).select(cols).to_numpy()
        np.testing.assert_array_equal(a, b)
        spec = BacktestSpec(unit, lead, horizon, (origin,), window="expanding")
        ta = fa.rows(spec.train_times(origin), lead, horizon)
        tb = fq.rows(spec.train_times(origin), lead, horizon)
        check_split(ta, fa.rows([origin], lead, horizon), origin, spec)
        np.testing.assert_array_equal(
            ta.select([*cols, "y"]).to_numpy(), tb.select([*cols, "y"]).to_numpy()
        )


## Install and helpers

In [ ]:
# Installs the package in editable mode with all extras (Bayesian stack included).
!pip install -q -e ".[dev,models,bayes]"
sys.path.insert(0, str(ROOT / "src"))  # make the package importable in this kernel too

In [ ]:
import copy, glob, json
import pandas as pd
import yaml

FAST = True          # small, quick runs; set False for the full runs
CORES = os.cpu_count() or 2

def config_with(path, updates, name):
    """Copy a YAML config with nested updates applied; returns the new path."""
    cfg = yaml.safe_load(open(path, encoding="utf-8"))
    def merge(a, b):
        for k, v in b.items():
            if isinstance(v, dict) and isinstance(a.get(k), dict):
                merge(a[k], v)
            else:
                a[k] = v
    merge(cfg, copy.deepcopy(updates))
    out = f"configs/nb_{name}.yaml"
    yaml.safe_dump(cfg, open(out, "w", encoding="utf-8"), sort_keys=False)
    return out

def latest(pattern):
    files = sorted(glob.glob(pattern, recursive=True), key=os.path.getmtime)
    if not files:
        raise FileNotFoundError(pattern)
    print("reading", files[-1])
    return json.load(open(files[-1], encoding="utf-8"))

def model_table(models):
    rows = {}
    for m, r in models.items():
        iv = r.get("intervals", {})
        rows[m] = {"mae": r.get("mae", r.get("mae_median")), "bias": r.get("bias"),
                   "wis": r.get("wis"), "cov50": iv.get("0.5", {}).get("coverage"),
                   "cov90": iv.get("0.9", {}).get("coverage")}
    return pd.DataFrame(rows).T.round(4)

## Run: tests (synthetic, no data needed)
The leakage tests run for both units, "day" and "hour".

In [ ]:
!python -m pytest -W error -m "not data" -q -p no:cacheprovider

## Run: Step 1, data download and audit
Downloads the pinned dataset revision, verifies checksums, re-runs the audit (it re-writes the
identical split manifest) and the real-data tests, including the real-data leakage checks.

In [ ]:
!python scripts/fetch_data.py
!python scripts/audit_data.py
!python -m pytest -m data -q -p no:cacheprovider

## Run: Step 3, baseline backtest (observed sales)
Rolling origin; train on feature times t <= T - L - H; predict (T + L, T + L + H].
Full day run: about 15 min on 12 cores and about 6 GB RAM. FAST uses 200 series and 2 folds.

In [ ]:
cfg3 = config_with("configs/baselines.yaml", {"lgbm": {"num_threads": CORES}}, "baselines")
smoke = "--smoke" if FAST else ""
!python scripts/run_baselines.py --unit day --config {cfg3} {smoke}
!python scripts/run_baselines.py --unit hour --config {cfg3} {smoke}

In [ ]:
for unit in ("day", "hour"):
    m = latest(f"runs/step3*/{unit}/*/metrics.json")
    print(unit, "origins", m["protocol"]["origins"][:5], "...")
    display(model_table(m["metrics"]["overall"]["models"]))
    display(pd.DataFrame({o: {k: v.get("wis", v.get("mae")) for k, v in f["models"].items()}
                          for o, f in m["metrics"]["folds"].items()}).T.round(4))

## Run: Step 4, censoring benchmark (proxy truth)
Demand-driven stockouts injected into fully in-stock days; calibration on the fit window only.
Full run: about 1.5 min for 5,000 series. FAST uses 300 series.

In [ ]:
smoke = "--smoke" if FAST else ""
!python scripts/run_censoring_benchmark.py {smoke}

In [ ]:
m = latest("runs/step4*/*/metrics.json")
print("calibrated injection:", m["calibrated_config"])
display(pd.DataFrame({"real": m["real_fit_window"]["stats"],
                      "injected": m["injected"]["stats"]}).round(3))
display(pd.DataFrame({"real": m["real_fit_window"]["position_ratios"],
                      "injected": m["injected"]["position_ratios"]}).round(3))
print("daily totals on stockout days")
display(model_table(m["benchmark"]["daily"]["overall"]["models"]))

## Run: Step 5, Bayesian model ladder (M0, M1, M2)
NumPyro NUTS. FAST: 4 series and short chains (a few minutes). Full: 20 series, 4 chains x
(1000 + 1000) draws (about 1-2 h on CPU). M3 = M2 with `--n-series 100 --models M2`.
On Colab, a GPU runtime can speed up the larger fits.

In [ ]:
chains = min(4, CORES)
cfg5 = config_with("configs/bayes.yaml", {"mcmc": {"num_chains": chains}}, "bayes")
smoke = "--smoke" if FAST else ""
!python scripts/run_bayes.py --config {cfg5} --part all {smoke}

In [ ]:
m = latest("runs/step5*/*/metrics_all.json")
real = m["real"]
keys = ("runtime_s", "divergences", "max_rhat", "min_ess")
display(pd.DataFrame({k: {x: v["diagnostics"][x] for x in keys}
                      for k, v in real["models"].items()}).T)
print("daily totals on stockout days (proxy truth); baselines are point estimates")
display(model_table(real["daily"]["overall"]["models"]))
print("PSIS-LOO on in-stock hours:", real["loo_compare"])
for name, r in m["recovery"]["models"].items():
    print("recovery", name, r["mean_window_demand"])
    display(pd.DataFrame(r["scalars"]).T)

## Run: Step 5, simulation-based calibration of M1
Ranks of the true parameters among posterior draws should be uniform (chi-square p-values not
small after Holm). Full: 200 replications (about 40 min). FAST: 20.

In [ ]:
reps = 20 if FAST else 200
!python scripts/run_sbc.py --reps {reps}
s = latest("runs/sbc/*/ranks.json")
display(pd.DataFrame(s["scalars"]).T)
print("min Holm-adjusted p:", s["min_p_holm"])
print("divergent replications:", s["divergent_reps"])